In [2]:
!pip install -q \
utilsforecast \
statsforecast \
holidays \
hyperopt \
category_encoders \
sktime \
shap \
lightgbm \
catboost \
xgboost \
plotly \
tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 49.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 62.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 48.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that 

# Загрузка библиотек

In [3]:
# Базовые библиотеки и утилиты
import numpy as np
import pandas as pd
import datetime
import os
import time
import time as time_module
import warnings
from copy import deepcopy
from bisect import bisect_left
from itertools import product
from typing import Union, Optional

# Настройки pandas
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.max_columns', None)

# Отображение в Jupyter
from IPython.display import display
%matplotlib inline

# Визуализация
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
from utilsforecast.plotting import plot_series

# Работа с календарём
import holidays

# Прогресс-бары
from tqdm import tqdm

# SciPy
import scipy as sp
from scipy import stats
from scipy.stats import ttest_1samp, shapiro, ks_2samp, zscore
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# Statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.datasets import longley
from statsmodels.formula.api import ols
from statsmodels.graphics import tsaplots
from statsmodels.graphics.gofplots import qqplot
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.tsa.stattools import acf, adfuller, kpss
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.ar_model import ar_select_order
from statsmodels.tsa.arima.model import ARIMA

# StatsForecast
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, _TS

# Scikit-learn: модели, препроцессинг, метрики
from sklearn.cluster import DBSCAN
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, mean_absolute_error, mean_squared_error, precision_score,
    r2_score, recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import (
    GridSearchCV, ParameterGrid, RandomizedSearchCV,
    TimeSeriesSplit, train_test_split
)
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Boosting модели
from catboost import CatBoostRegressor
import catboost as cb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier, XGBRegressor
import xgboost as xgb

# Гиперпараметрический поиск
from hyperopt import fmin, tpe, Trials, STATUS_OK, hp

# Кодировщики категориальных признаков
import category_encoders as ce

# Дополнительные утилиты
from sktime.utils.plotting import plot_correlations

# SHAP
import shap

Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7ac73cd03920>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
                     ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: /usr/local/lib/python3.12/dist-packages/numpy.libs/libscipy_openblas64_-32a4b2

In [4]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/kaggle/input/datasets/johndberrios/train-2/train_2.csv


In [5]:
df = pd.read_csv('/kaggle/input/datasets/johndberrios/train-2/train_2.csv')

In [6]:
df.head()

,new_id,Год,Месяц,Среднее количество промо товаров в чеке,Среднее количество товаров в чеке,Среднее количество отмен,Рабочие часы в день,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,Регион,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м),Количество касс,Флаг алкогольной лицензии,РТО
0,0,2024,1,1.080,6.030,147.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,75147744.850
1,0,2023,1,1.320,6.040,162.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,74914754.220
2,0,2025,1,0.820,6.000,145.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,87125506.920
3,0,2025,2,0.900,6.000,118.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,82659801.630
4,0,2024,2,1.250,6.060,154.000,16.000,Новый,Большой,Ярославль г,Ярославская обл,603883,3775,138,73,1,0,0,0,3,1,10,1,74209339.110


# Функции создания признаков
# Подготовка

In [7]:
rename_cols = {
    "new_id": "new_id",
    "Год": "year",
    "Месяц": "month",
    "Среднее количество промо товаров в чеке": "avg_promo_items_per_receipt",
    "Среднее количество товаров в чеке": "avg_items_per_receipt",
    "Среднее количество отмен": "avg_cancellations",
    "Рабочие часы в день": "working_hours_per_day",
    "Дата открытия, категориальный": "opening_date_category",
    "Торговая площадь, категориальный": "trading_area_category",
    "Населенный пункт": "settlement",
    "Регион": "region",
    "Численность населения": "population_size",
    "Количество домохозяйств": "number_of_households",
    "Трафик пеший, в час": "pedestrian_traffic_per_hour",
    "Трафик авто, в час": "car_traffic_per_hour",
    "Маркетплейсы, доставки, постаматы (100 м)": "marketplaces_deliveries_parcel_lockers_100m",
    "Медицинские уч. и аптеки (300 м)": "medical_institutions_and_pharmacies_300m",
    "Школы (300 м)": "schools_300m",
    "Остановки (300 м)": "public_transport_stops_300m",
    "Продуктовые магазины (500 м)": "grocery_stores_500m",
    "Пятерочки (500 м)": "pyaterochka_stores_500m",
    "Количество касс": "number_of_cash_registers",
    "Флаг алкогольной лицензии": "alcohol_license_flag",
    "РТО": "pto",
}

df = df.rename(columns=rename_cols)

# Функция для создания признаков и сплита (без кросс-валидации)

In [8]:
from typing import Literal, Optional, Sequence

import gc
import warnings
import numpy as np
import pandas as pd


def make_panel_time_split(
    df: pd.DataFrame,
    *,
    group_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    split_mode: Literal["train_val", "train_val_test", "train_test"] = "train_val",
    val_time_idx: Optional[int] = None,
    val_year: Optional[int] = None,
    val_month: Optional[int] = None,
    test_time_idx: Optional[int] = None,
    test_year: Optional[int] = None,
    test_month: Optional[int] = None,
    min_year: Optional[int] = None,
    create_time_col: bool = True,
    create_test_if_missing: bool = True,
    test_target_value=np.nan,
    future_unknown_cols: Optional[Sequence[str]] = None,
    drop_target_na_from_train: bool = True,
    require_known_val_target: bool = True,
    check_duplicates: bool = True,
    validate_time_col: bool = True,
    expected_test_size: Optional[int] = None,
    copy: bool = True,
    verbose: bool = True,
):
    """
    Leakage-safe time split для panel monthly forecasting.

    split_mode:
    - "train_val": train = months before val, val = selected validation month;
    - "train_val_test": train + val + artificial/existing test month;
    - "train_test": train = all known history before test, test = artificial/existing test month.

    Если test-месяца нет в df, он создаётся из последней известной строки каждого магазина.
    Важно: если дальше строятся лаги/rolling, test-строки должны быть созданы ДО feature engineering.
    """
    if split_mode not in {"train_val", "train_val_test", "train_test"}:
        raise ValueError("split_mode должен быть 'train_val', 'train_val_test' или 'train_test'.")

    if copy:
        df = df.copy()

    def _check_time_args(prefix, time_idx, year, month):
        if time_idx is not None and (year is not None or month is not None):
            raise ValueError(
                f"Для {prefix} передай либо {prefix}_time_idx, либо пару "
                f"{prefix}_year + {prefix}_month, но не оба варианта."
            )
        if (year is None) ^ (month is None):
            raise ValueError(
                f"Для {prefix} нужно передавать year и month вместе. "
                f"Получено: {prefix}_year={year}, {prefix}_month={month}."
            )

    _check_time_args("val", val_time_idx, val_year, val_month)
    _check_time_args("test", test_time_idx, test_year, test_month)

    if split_mode == "train_test" and (
        val_time_idx is not None or val_year is not None or val_month is not None
    ):
        raise ValueError("В split_mode='train_test' validation-месяц не используется.")

    required_cols = {group_col, target_col}
    has_time_col = time_col in df.columns
    has_year_month = year_col in df.columns and month_col in df.columns

    if not has_time_col:
        if create_time_col:
            required_cols.update({year_col, month_col})
        else:
            required_cols.add(time_col)

    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"В df отсутствуют обязательные колонки: {sorted(missing)}")

    if has_year_month:
        if df[year_col].isna().any() or df[month_col].isna().any():
            raise ValueError(f"В {year_col}/{month_col} есть пропуски.")
        df[year_col] = pd.to_numeric(df[year_col], errors="raise").astype(int)
        df[month_col] = pd.to_numeric(df[month_col], errors="raise").astype(int)
        bad_months = sorted(set(df[month_col].unique()) - set(range(1, 13)))
        if bad_months:
            raise ValueError(f"В колонке '{month_col}' некорректные месяцы: {bad_months}.")

    if has_time_col:
        if df[time_col].isna().any():
            raise ValueError(f"В колонке '{time_col}' есть пропуски.")
        df[time_col] = pd.to_numeric(df[time_col], errors="raise").astype(int)

        if min_year is None and has_year_month:
            inferred = df[year_col] - ((df[time_col] - (df[month_col] - 1)) // 12)
            unique = sorted(inferred.unique())
            if len(unique) == 1:
                min_year = int(unique[0])
            else:
                raise ValueError(
                    f"Не удалось однозначно восстановить min_year из {time_col}. "
                    f"Передай min_year явно. Найдено: {unique[:10]}"
                )

        if validate_time_col and has_year_month and min_year is not None:
            expected = ((df[year_col] - int(min_year)) * 12 + (df[month_col] - 1)).astype(int)
            bad = df[time_col].astype(int) != expected
            if bad.any():
                examples = df.loc[bad, [group_col, year_col, month_col, time_col]].head(10)
                raise ValueError(
                    f"'{time_col}' не совпадает с расчётом по min_year={min_year}. "
                    f"Примеры:\n{examples}"
                )
    else:
        if not has_year_month:
            raise ValueError(f"Нельзя создать '{time_col}', потому что нет year/month.")
        if min_year is None:
            min_year = int(df[year_col].min())
        df[time_col] = ((df[year_col] - int(min_year)) * 12 + (df[month_col] - 1)).astype(int)

    known_target_times = sorted(df.loc[df[target_col].notna(), time_col].unique())
    if not known_target_times:
        raise ValueError(f"В '{target_col}' нет известных значений.")
    max_known_time_idx = int(known_target_times[-1])

    def _resolve(label, time_idx, year, month, default=None):
        if time_idx is not None:
            return int(time_idx)
        if year is not None and month is not None:
            if min_year is None:
                raise ValueError(f"Для расчёта {label}_time_idx нужен min_year.")
            month = int(month)
            if not 1 <= month <= 12:
                raise ValueError(f"{label}_month должен быть от 1 до 12.")
            return (int(year) - int(min_year)) * 12 + (month - 1)
        return None if default is None else int(default)

    if split_mode in {"train_val", "train_val_test"}:
        val_time_idx = _resolve("val", val_time_idx, val_year, val_month, max_known_time_idx)
    else:
        val_time_idx = None

    if split_mode in {"train_val_test", "train_test"}:
        test_time_idx = _resolve("test", test_time_idx, test_year, test_month, max_known_time_idx + 1)
    else:
        test_time_idx = None

    if split_mode == "train_val_test" and not (val_time_idx < test_time_idx):
        raise ValueError(
            f"Ожидается val_time_idx < test_time_idx, получено "
            f"{val_time_idx=} и {test_time_idx=}."
        )

    df = df.sort_values([group_col, time_col]).reset_index(drop=True)

    if check_duplicates:
        dup_mask = df.duplicated([group_col, time_col], keep=False)
        if dup_mask.any():
            examples = df.loc[dup_mask, [group_col, time_col]].sort_values([group_col, time_col]).head(10)
            raise ValueError(
                f"Есть дубли по [{group_col}, {time_col}]. "
                f"Строк-дублей: {int(dup_mask.sum())}. Примеры:\n{examples}"
            )

    test = None
    if split_mode in {"train_val_test", "train_test"}:
        existing = df[df[time_col] == test_time_idx].copy()
        if not existing.empty:
            test = existing.copy()
        else:
            if not create_test_if_missing:
                raise ValueError(f"В df нет test_time_idx={test_time_idx}.")
            hist = df[(df[time_col] < test_time_idx) & (df[target_col].notna())].copy()
            if hist.empty:
                raise ValueError(f"Нет исторических строк до test_time_idx={test_time_idx}.")
            test = (
                hist.sort_values([group_col, time_col])
                .groupby(group_col, sort=False)
                .tail(1)
                .copy()
            )
            test[time_col] = int(test_time_idx)
            if has_year_month and min_year is not None:
                test[year_col] = int(min_year) + int(test_time_idx) // 12
                test[month_col] = int(test_time_idx) % 12 + 1
            test[target_col] = test_target_value

            if future_unknown_cols is not None:
                for col in future_unknown_cols:
                    if col in test.columns:
                        test[col] = np.nan
                    else:
                        warnings.warn(f"Колонка '{col}' из future_unknown_cols отсутствует в df.")

        test = test.sort_values([group_col, time_col]).reset_index(drop=True)
        if expected_test_size is not None and len(test) != int(expected_test_size):
            raise ValueError(
                f"Размер test не совпадает с expected_test_size: "
                f"получено {len(test)}, ожидалось {expected_test_size}."
            )

    if split_mode == "train_test":
        train_mask = df[time_col] < test_time_idx
    else:
        train_mask = df[time_col] < val_time_idx

    if drop_target_na_from_train:
        train_mask = train_mask & df[target_col].notna()

    train = df.loc[train_mask].copy()
    if train.empty:
        raise ValueError("train получился пустым.")

    val = None
    if split_mode in {"train_val", "train_val_test"}:
        val = df[df[time_col] == val_time_idx].copy()
        if val.empty:
            available = sorted(df[time_col].unique())
            raise ValueError(
                f"val пустой для val_time_idx={val_time_idx}. "
                f"Доступные time_idx: {available[:5]} ... {available[-5:]}"
            )
        if require_known_val_target and val[target_col].isna().any():
            n_nan = int(val[target_col].isna().sum())
            raise ValueError(
                f"В validation есть NaN в '{target_col}': {n_nan}. "
                "Скорее всего, validation попал на искусственный test-месяц."
            )
        val = val.sort_values([group_col, time_col]).reset_index(drop=True)

    train = train.sort_values([group_col, time_col]).reset_index(drop=True)

    if verbose:
        print("split_mode:", split_mode)
        print("min_year:", min_year)
        print("max_known_time_idx:", max_known_time_idx)
        if val_time_idx is not None:
            print("val_time_idx:", val_time_idx)
        if test_time_idx is not None:
            print("test_time_idx:", test_time_idx)
        print("train shape:", train.shape)
        if val is not None:
            print("val shape:", val.shape)
        if test is not None:
            print("test shape:", test.shape)

    if split_mode == "train_val":
        return train, val
    if split_mode == "train_test":
        return train, test
    return train, val, test




# ============================================================
# Base feature generator from uploaded file
# ============================================================

import gc
import numpy as np
import pandas as pd


def generate_retail_forecast_features_leakfree(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame | None = None,
    test_df: pd.DataFrame | None = None,
    *,
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    min_year: int = 2023,
    make_val_from_train: bool = False,
    val_time_idx: int | None = None,
    use_val_as_test_history: bool = True,
    add_calendar_features: bool = True,
    add_static_features: bool = True,
    add_target_history: bool = True,
    add_category_target_history: bool = True,
    add_non_target_monthly_history: bool = True,
    include_current_receipt_features: bool = False,
    current_monthly_cols: tuple[str, ...] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    current_monthly_derived_cols: tuple[str, ...] = (
        "promo_items_share",
        "promo_x_items",
        "items_x_working_hours",
        "items_x_working_days",
        "alcohol_x_items",
    ),
    working_hours_future_known: bool = False,
    target_lags: tuple[int, ...] = (1, 2, 3, 6, 12, 13),
    target_rolling_windows: tuple[int, ...] = (3, 12),
    target_rolling_min_periods: int = 2,
    target_history_scale: str = "raw",   # raw | log | both
    keep_raw_pto_lag_1_anchor_for_log: bool = True,
    add_log_target_dynamics: bool = True,
    group_specs: list | None = None,
    group_agg_stat: str = "mean",
    group_lag: int = 1,
    add_group_lag_feature: bool = True,
    add_group_relative_ratio: bool = True,
    add_group_relative_diff: bool = False,
    monthly_feature_config: dict | None = None,
    monthly_history_scale: str = "raw",  # raw | log | both
    monthly_rolling_min_periods: int = 2,
    add_monthly_cross_lags: bool = True,
    monthly_cross_lags: tuple[int, ...] = (1, 12),
    monthly_cross_feature_types: tuple[str, ...] = ("promo_share",),
    drop_current_monthly_cols: bool = True,
    drop_current_monthly_derived_cols: bool = True,
    drop_year_col: bool = True,
    drop_obvious_duplicates: bool = True,
    extra_drop_cols: list | tuple | None = None,
    fill_history_na: bool = False,
    fill_value: float = 0.0,
    downcast_float_features: bool = True,
    copy: bool = True,
    verbose: bool = True,
) -> dict:
    """
    Единый leak-free feature-pipeline для panel monthly forecasting.

    Основной контракт:
    - val/test target не используется для лагов и group target history;
    - текущие monthly-cols из val/test не используются для лагов;
    - лаги считаются через time-aware merge по time_col - lag, а не через shift строк;
    - rolling-признаки считаются только по прошлым наблюдениям внутри history;
    - для test можно использовать train+val как историю, если val уже является известным прошлым.
    """

    if target_history_scale not in {"raw", "log", "both"}:
        raise ValueError("target_history_scale должен быть 'raw', 'log' или 'both'.")
    if monthly_history_scale not in {"raw", "log", "both"}:
        raise ValueError("monthly_history_scale должен быть 'raw', 'log' или 'both'.")
    if group_agg_stat not in {"mean", "median"}:
        raise ValueError("group_agg_stat должен быть 'mean' или 'median'.")

    if copy:
        train_df = train_df.copy()
        val_df = val_df.copy() if val_df is not None else None
        test_df = test_df.copy() if test_df is not None else None

    # ============================================================
    # Helpers
    # ============================================================

    def _safe_divide(numerator, denominator, fill=np.nan):
        numerator = numerator if isinstance(numerator, pd.Series) else pd.Series(numerator)
        denominator = denominator if isinstance(denominator, pd.Series) else pd.Series(denominator)
        result = numerator / denominator.replace(0, np.nan)
        result = result.replace([np.inf, -np.inf], np.nan)
        if fill is not None and not (isinstance(fill, float) and np.isnan(fill)):
            result = result.fillna(fill)
        return result

    def _col(df, name, default=0.0):
        if name in df.columns:
            return df[name].fillna(default)
        return pd.Series(default, index=df.index)

    def _ensure_time_year_month(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
        df = df.copy()

        has_year_month = year_col in df.columns and month_col in df.columns
        has_time = time_col in df.columns

        if not has_year_month and not has_time:
            raise ValueError(
                f"В {name} нет ни пары '{year_col}'+'{month_col}', ни '{time_col}'."
            )

        if has_year_month:
            if df[year_col].isna().any() or df[month_col].isna().any():
                raise ValueError(f"В {name} есть NA в {year_col}/{month_col}.")
            df[year_col] = df[year_col].astype(int)
            df[month_col] = df[month_col].astype(int)
            bad_months = sorted(set(df[month_col].unique()) - set(range(1, 13)))
            if bad_months:
                raise ValueError(f"В {name} некорректные месяцы: {bad_months}.")
            # Важно: всегда пересчитываем глобальную шкалу, чтобы не было local-min-year.
            df[time_col] = ((df[year_col] - min_year) * 12 + (df[month_col] - 1)).astype(int)
        else:
            if df[time_col].isna().any():
                raise ValueError(f"В {name} есть NA в {time_col}.")
            df[time_col] = df[time_col].astype(int)
            df[year_col] = (min_year + df[time_col] // 12).astype(int)
            df[month_col] = (df[time_col] % 12 + 1).astype(int)

        return df

    def _validate_required(df: pd.DataFrame, required: set[str], name: str):
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"В {name} нет обязательных колонок: {missing}")

    def _validate_unique_panel(df: pd.DataFrame, name: str):
        if {id_col, time_col}.issubset(df.columns):
            duplicated = df.duplicated([id_col, time_col]).sum()
            if duplicated > 0:
                raise ValueError(
                    f"В {name} найдено {duplicated} дублей по [{id_col}, {time_col}]. "
                    "Для лагов ожидается одна строка на магазин-месяц."
                )

    def _part_concat(parts: dict[str, pd.DataFrame]) -> pd.DataFrame:
        out = []
        for part_name, df_part in parts.items():
            p = df_part.copy()
            p["__part__"] = part_name
            p["__row_order__"] = np.arange(len(p))
            out.append(p)
        return pd.concat(out, axis=0, ignore_index=True, sort=False)

    def _split_parts(combined: pd.DataFrame, part_names: list[str]) -> dict[str, pd.DataFrame]:
        result = {}
        for part_name in part_names:
            part = (
                combined[combined["__part__"] == part_name]
                .sort_values("__row_order__")
                .drop(columns=["__part__", "__row_order__"], errors="ignore")
                .reset_index(drop=True)
            )
            result[part_name] = part
        return result

    def _drop_preexisting_generated_cols(df: pd.DataFrame) -> pd.DataFrame:
        prefixes = [
            f"{target_col}_lag_", f"log_{target_col}_lag_", f"{target_col}_roll_",
            f"{target_col}_diff_", f"{target_col}_ratio_", f"{target_col}_seasonal_",
            f"log_{target_col}_diff_", f"{target_col}_lag_1_to_", f"{target_col}_lag_12_to_",
            "promo_items_share_lag_", "promo_x_items_lag_", "items_x_working_hours_lag_",
            "cancellations_per_item_lag_", "cancellations_per_working_hour_lag_",
            "region_", "settlement_",
        ]
        monthly_prefixes = []
        for c in current_monthly_cols:
            monthly_prefixes.extend([
                f"{c}_lag_", f"log_{c}_lag_", f"{c}_roll_", f"{c}_diff_", f"{c}_ratio_",
            ])

        explicit = {
            "season", "is_winter", "is_spring", "is_autumn", "is_quarter_start_month",
            "is_quarter_end_month", "is_halfyear_start_month", "is_halfyear_end_month",
            "is_year_start_month", "is_year_end_month", "month_in_halfyear",
            "month_name", "month_sin", "month_cos", "month_sin_2", "month_cos_2",
            "prev_month", "n_days_in_month", "n_weekend_days", "n_may_holiday_days",
            "n_single_public_holiday_days", "n_november_holiday_days",
            "n_december_additional_holiday_days", "n_holiday_days", "n_additional_holiday_days",
            "n_working_days", "n_additional_working_days", "is_leap_year", "holiday_month_type",
            "is_middle_of_quarter", "is_pre_new_year_month", "is_pre_may_holidays_month",
            "is_back_to_school_month", "is_business_activity_low_month", "is_business_activity_high_month",
            "total_traffic_per_hour", "pedestrian_traffic_share", "car_to_pedestrian_traffic_ratio",
            "log_population_size", "log_number_of_households", "log_pedestrian_traffic_per_hour",
            "log_car_traffic_per_hour", "is_moscow", "is_million_plus_city", "population_city_type",
            "has_school_nearby", "has_transport_nearby", "has_marketplace_nearby",
            "has_medical_nearby", "has_grocery_competition", "has_pyaterochka_nearby",
            "is_high_grocery_competition", "is_high_pyaterochka_density",
            "non_pyaterochka_grocery_stores_500m", "pyaterochka_share_among_grocery",
            "non_pyaterochka_competition_share", "grocery_per_1000_people",
            "pyaterochka_per_1000_people", "competition_per_1000_households",
            "pyaterochka_per_1000_households", "people_per_household",
            "households_per_1000_people", "traffic_per_1000_people", "traffic_per_1000_households",
            "pedestrian_traffic_per_1000_people", "social_infra_score", "commercial_infra_score",
            "poi_score", "poi_per_1000_households", "public_transport_share_in_social_infra",
            "medical_share_in_social_infra", "cash_register_hours_capacity", "traffic_per_cash_register",
            "traffic_per_working_hour", "households_per_cash_register", "population_per_cash_register",
            "cash_registers_per_total_traffic", "promo_items_share", "promo_x_items",
            "items_x_working_hours", "items_x_working_days", "alcohol_x_capacity",
            "alcohol_x_total_traffic", "alcohol_x_items", "traffic_x_total_poi",
            "traffic_x_grocery_competition", "traffic_x_pyaterochka_competition",
            "pedestrian_traffic_x_transport", "schools_x_back_to_school",
            "settlement_store_count", "region_store_count", "region_trading_area_store_count",
            "region_opening_age_store_count",
        }

        drop_cols = []
        for c in df.columns:
            if c in explicit:
                drop_cols.append(c)
            elif any(c.startswith(p) for p in prefixes + monthly_prefixes):
                # Не удаляем сырые region/settlement, только generated mean/count; это грубый фильтр ниже уточняем.
                if c in {"region", "settlement"}:
                    continue
                # region_* может быть исходной колонкой только маловероятно; для безопасности удаляем только *_mean/count.
                if c.startswith(("region_", "settlement_")) and not (c.endswith("_mean") or c.endswith("_count")):
                    continue
                drop_cols.append(c)
            elif f"_{target_col}_{group_agg_stat}_lag_" in c:
                drop_cols.append(c)

        return df.drop(columns=sorted(set(drop_cols)), errors="ignore")

    # ============================================================
    # Calendar and static feature blocks
    # ============================================================

    def _add_calendar(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        years = sorted(df[year_col].dropna().astype(int).unique())
        if not years:
            return df

        def _holiday_calendar():
            holidays = {
                # 2023
                "2023-01-01": "New_Year_Holidays", "2023-01-02": "New_Year_Holidays",
                "2023-01-03": "New_Year_Holidays", "2023-01-04": "New_Year_Holidays",
                "2023-01-05": "New_Year_Holidays", "2023-01-06": "New_Year_Holidays",
                "2023-01-07": "Christmas_Day", "2023-01-08": "New_Year_Holidays",
                "2023-02-23": "Defender_of_the_Fatherland_Day",
                "2023-03-08": "International_Womens_Day",
                "2023-05-01": "Spring_and_Labour_Holiday", "2023-05-09": "Victory_Day",
                "2023-06-12": "Day_of_Russia", "2023-11-04": "National_Unity_Day",
                "2023-02-24": "Additional_Holiday", "2023-05-08": "Additional_Holiday",
                "2023-11-06": "Additional_Holiday",
                # 2024
                "2024-01-01": "New_Year_Holidays", "2024-01-02": "New_Year_Holidays",
                "2024-01-03": "New_Year_Holidays", "2024-01-04": "New_Year_Holidays",
                "2024-01-05": "New_Year_Holidays", "2024-01-06": "New_Year_Holidays",
                "2024-01-07": "Christmas_Day", "2024-01-08": "New_Year_Holidays",
                "2024-02-23": "Defender_of_the_Fatherland_Day",
                "2024-03-08": "International_Womens_Day",
                "2024-05-01": "Spring_and_Labour_Holiday", "2024-05-09": "Victory_Day",
                "2024-06-12": "Day_of_Russia", "2024-11-04": "National_Unity_Day",
                "2024-04-29": "Additional_Holiday", "2024-04-30": "Additional_Holiday",
                "2024-05-10": "Additional_Holiday", "2024-12-30": "Additional_Holiday",
                "2024-12-31": "Additional_Holiday",
                # 2025
                "2025-01-01": "New_Year_Holidays", "2025-01-02": "New_Year_Holidays",
                "2025-01-03": "New_Year_Holidays", "2025-01-04": "New_Year_Holidays",
                "2025-01-05": "New_Year_Holidays", "2025-01-06": "New_Year_Holidays",
                "2025-01-07": "Christmas_Day", "2025-01-08": "New_Year_Holidays",
                "2025-02-23": "Defender_of_the_Fatherland_Day",
                "2025-03-08": "International_Womens_Day",
                "2025-05-01": "Spring_and_Labour_Holiday", "2025-05-09": "Victory_Day",
                "2025-06-12": "Day_of_Russia", "2025-11-04": "National_Unity_Day",
                "2025-05-02": "Additional_Holiday", "2025-05-08": "Additional_Holiday",
                "2025-06-13": "Additional_Holiday", "2025-11-03": "Additional_Holiday",
                "2025-12-31": "Additional_Holiday",
            }
            additional_working_days = {"2024-04-27", "2024-11-02", "2024-12-28", "2025-11-01"}
            return (
                {pd.to_datetime(k): v for k, v in holidays.items()},
                {pd.to_datetime(x) for x in additional_working_days},
            )

        holidays, additional_working_days = _holiday_calendar()

        df["season"] = df[month_col].map({
            12: "Winter", 1: "Winter", 2: "Winter",
            3: "Spring", 4: "Spring", 5: "Spring",
            6: "Summer", 7: "Summer", 8: "Summer",
            9: "Autumn", 10: "Autumn", 11: "Autumn",
        })
        df["is_winter"] = (df["season"] == "Winter").astype(int)
        df["is_spring"] = (df["season"] == "Spring").astype(int)
        df["is_autumn"] = (df["season"] == "Autumn").astype(int)
        df["is_quarter_start_month"] = df[month_col].isin([1, 4, 7, 10]).astype(int)
        df["is_quarter_end_month"] = df[month_col].isin([3, 6, 9, 12]).astype(int)
        df["is_halfyear_start_month"] = df[month_col].isin([1, 7]).astype(int)
        df["is_halfyear_end_month"] = df[month_col].isin([6, 12]).astype(int)
        df["is_year_start_month"] = (df[month_col] == 1).astype(int)
        df["is_year_end_month"] = (df[month_col] == 12).astype(int)
        df["month_in_halfyear"] = ((df[month_col] - 1) % 6 + 1).astype(int)
        df["month_name"] = df[month_col].map({
            1: "January", 2: "February", 3: "March", 4: "April", 5: "May", 6: "June",
            7: "July", 8: "August", 9: "September", 10: "October", 11: "November", 12: "December",
        })
        df["month_sin"] = np.sin(2 * np.pi * df[month_col] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df[month_col] / 12)
        df["month_sin_2"] = np.sin(2 * np.pi * 2 * df[month_col] / 12)
        df["month_cos_2"] = np.cos(2 * np.pi * 2 * df[month_col] / 12)
        df["prev_month"] = df[month_col].where(df[month_col] > 1, 13) - 1

        calendar = pd.DataFrame({
            "date": pd.date_range(f"{min(years)}-01-01", f"{max(years)}-12-31", freq="D")
        })
        calendar["year"] = calendar["date"].dt.year
        calendar["month"] = calendar["date"].dt.month
        calendar["weekday"] = calendar["date"].dt.dayofweek
        calendar["is_weekend"] = calendar["weekday"].isin([5, 6]).astype(int)
        calendar["holiday_name"] = calendar["date"].map(holidays)
        calendar["is_holiday"] = calendar["holiday_name"].notna().astype(int)
        calendar["is_additional_working_day"] = calendar["date"].isin(additional_working_days).astype(int)
        calendar["is_nonworking_day"] = (((calendar["is_weekend"] == 1) | (calendar["is_holiday"] == 1)) & (calendar["is_additional_working_day"] == 0)).astype(int)
        calendar["is_working_day"] = 1 - calendar["is_nonworking_day"]
        calendar["is_may_holiday"] = calendar["holiday_name"].isin(["Spring_and_Labour_Holiday", "Victory_Day"]).astype(int)
        calendar["is_single_public_holiday"] = calendar["holiday_name"].isin([
            "Defender_of_the_Fatherland_Day", "International_Womens_Day", "Day_of_Russia", "National_Unity_Day"
        ]).astype(int)
        calendar["is_additional_holiday"] = (calendar["holiday_name"] == "Additional_Holiday").astype(int)
        calendar["is_november_holiday"] = (calendar["holiday_name"] == "National_Unity_Day").astype(int)
        calendar["is_december_additional_holiday"] = ((calendar["month"] == 12) & (calendar["holiday_name"] == "Additional_Holiday")).astype(int)

        month_calendar = (
            calendar.groupby(["year", "month"], as_index=False)
            .agg(
                n_days_in_month=("date", "count"),
                n_weekend_days=("is_weekend", "sum"),
                n_may_holiday_days=("is_may_holiday", "sum"),
                n_single_public_holiday_days=("is_single_public_holiday", "sum"),
                n_november_holiday_days=("is_november_holiday", "sum"),
                n_december_additional_holiday_days=("is_december_additional_holiday", "sum"),
                n_holiday_days=("is_holiday", "sum"),
                n_additional_holiday_days=("is_additional_holiday", "sum"),
                n_working_days=("is_working_day", "sum"),
                n_additional_working_days=("is_additional_working_day", "sum"),
            )
        )
        month_calendar["is_leap_year"] = (
            ((month_calendar["year"] % 4 == 0) & ((month_calendar["year"] % 100 != 0) | (month_calendar["year"] % 400 == 0)))
        ).astype(int)
        month_calendar["holiday_month_type"] = month_calendar["month"].map(
            lambda m: "new_year_holidays" if m == 1 else
            "may_holidays" if m == 5 else
            "single_public_holiday" if m in [2, 3, 6, 11] else
            "pre_new_year" if m == 12 else
            "no_major_holiday"
        )

        df = df.merge(
            month_calendar,
            left_on=[year_col, month_col],
            right_on=["year", "month"],
            how="left",
            suffixes=("", "_calendar"),
        ).drop(columns=["year_calendar", "month_calendar"], errors="ignore")

        df["is_middle_of_quarter"] = (((df[month_col] - 1) % 3 + 1) == 2).astype(int)
        df["is_pre_new_year_month"] = df[month_col].isin([11, 12]).astype(int)
        df["is_pre_may_holidays_month"] = (df[month_col] == 4).astype(int)
        df["is_back_to_school_month"] = df[month_col].isin([8, 9]).astype(int)
        df["is_business_activity_low_month"] = df[month_col].isin([1, 5, 7, 8]).astype(int)
        df["is_business_activity_high_month"] = df[month_col].isin([3, 4, 9, 10, 11]).astype(int)

        return df

    def _add_static(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # Traffic
        df["total_traffic_per_hour"] = _col(df, "pedestrian_traffic_per_hour") + _col(df, "car_traffic_per_hour")
        df["pedestrian_traffic_share"] = _safe_divide(_col(df, "pedestrian_traffic_per_hour"), df["total_traffic_per_hour"], fill=0.0)
        df["car_to_pedestrian_traffic_ratio"] = _safe_divide(_col(df, "car_traffic_per_hour"), _col(df, "pedestrian_traffic_per_hour"), fill=0.0)

        for c in ["population_size", "number_of_households", "pedestrian_traffic_per_hour", "car_traffic_per_hour"]:
            if c in df.columns:
                df[f"log_{c}"] = np.log1p(df[c].clip(lower=0).fillna(0))

        if "settlement" in df.columns:
            df["is_moscow"] = df["settlement"].astype(str).str.strip().str.lower().eq("москва г").astype(int)
        else:
            df["is_moscow"] = 0

        df["is_million_plus_city"] = (_col(df, "population_size") >= 1_000_000).astype(int)

        def _city_type(pop, is_moscow):
            if is_moscow == 1:
                return "moscow"
            if pd.isna(pop) or pop <= 0:
                return "unknown_or_zero"
            if pop < 10_000:
                return "small_settlement"
            if pop < 50_000:
                return "small_town"
            if pop < 250_000:
                return "medium_city"
            if pop < 1_000_000:
                return "large_city"
            return "million_plus_city"

        df["population_city_type"] = [_city_type(p, m) for p, m in zip(_col(df, "population_size"), df["is_moscow"])]

        for src, dst in [
            ("schools_300m", "has_school_nearby"),
            ("public_transport_stops_300m", "has_transport_nearby"),
            ("marketplaces_deliveries_parcel_lockers_100m", "has_marketplace_nearby"),
            ("medical_institutions_and_pharmacies_300m", "has_medical_nearby"),
            ("grocery_stores_500m", "has_grocery_competition"),
            ("pyaterochka_stores_500m", "has_pyaterochka_nearby"),
        ]:
            df[dst] = (_col(df, src) > 0).astype(int)

        df["is_high_grocery_competition"] = (_col(df, "grocery_stores_500m") >= 5).astype(int)
        df["is_high_pyaterochka_density"] = (_col(df, "pyaterochka_stores_500m") >= 3).astype(int)

        df["non_pyaterochka_grocery_stores_500m"] = (_col(df, "grocery_stores_500m") - _col(df, "pyaterochka_stores_500m")).clip(lower=0)
        df["pyaterochka_share_among_grocery"] = _safe_divide(_col(df, "pyaterochka_stores_500m"), _col(df, "grocery_stores_500m"), fill=0.0)
        df["non_pyaterochka_competition_share"] = _safe_divide(df["non_pyaterochka_grocery_stores_500m"], _col(df, "grocery_stores_500m"), fill=0.0)
        df["grocery_per_1000_people"] = _safe_divide(_col(df, "grocery_stores_500m") * 1000, _col(df, "population_size"), fill=0.0)
        df["pyaterochka_per_1000_people"] = _safe_divide(_col(df, "pyaterochka_stores_500m") * 1000, _col(df, "population_size"), fill=0.0)
        df["competition_per_1000_households"] = _safe_divide(_col(df, "grocery_stores_500m") * 1000, _col(df, "number_of_households"), fill=0.0)
        df["pyaterochka_per_1000_households"] = _safe_divide(_col(df, "pyaterochka_stores_500m") * 1000, _col(df, "number_of_households"), fill=0.0)

        df["people_per_household"] = _safe_divide(_col(df, "population_size"), _col(df, "number_of_households"), fill=0.0)
        df["households_per_1000_people"] = _safe_divide(_col(df, "number_of_households") * 1000, _col(df, "population_size"), fill=0.0)

        df["traffic_per_1000_people"] = _safe_divide(df["total_traffic_per_hour"] * 1000, _col(df, "population_size"), fill=0.0)
        df["traffic_per_1000_households"] = _safe_divide(df["total_traffic_per_hour"] * 1000, _col(df, "number_of_households"), fill=0.0)
        df["pedestrian_traffic_per_1000_people"] = _safe_divide(_col(df, "pedestrian_traffic_per_hour") * 1000, _col(df, "population_size"), fill=0.0)

        df["social_infra_score"] = _col(df, "medical_institutions_and_pharmacies_300m") + _col(df, "schools_300m") + _col(df, "public_transport_stops_300m")
        df["commercial_infra_score"] = _col(df, "marketplaces_deliveries_parcel_lockers_100m") + _col(df, "grocery_stores_500m")
        df["poi_score"] = (
            _col(df, "marketplaces_deliveries_parcel_lockers_100m")
            + _col(df, "medical_institutions_and_pharmacies_300m")
            + _col(df, "schools_300m")
            + _col(df, "public_transport_stops_300m")
            + _col(df, "grocery_stores_500m")
        )
        df["poi_per_1000_households"] = _safe_divide(df["poi_score"] * 1000, _col(df, "number_of_households"), fill=0.0)
        df["public_transport_share_in_social_infra"] = _safe_divide(_col(df, "public_transport_stops_300m"), df["social_infra_score"], fill=0.0)
        df["medical_share_in_social_infra"] = _safe_divide(_col(df, "medical_institutions_and_pharmacies_300m"), df["social_infra_score"], fill=0.0)

        df["cash_register_hours_capacity"] = _col(df, "number_of_cash_registers") * _col(df, "working_hours_per_day")
        df["traffic_per_cash_register"] = _safe_divide(df["total_traffic_per_hour"], _col(df, "number_of_cash_registers"), fill=0.0)
        df["traffic_per_working_hour"] = _safe_divide(df["total_traffic_per_hour"], _col(df, "working_hours_per_day"), fill=0.0)
        df["households_per_cash_register"] = _safe_divide(_col(df, "number_of_households"), _col(df, "number_of_cash_registers"), fill=0.0)
        df["population_per_cash_register"] = _safe_divide(_col(df, "population_size"), _col(df, "number_of_cash_registers"), fill=0.0)
        df["cash_registers_per_total_traffic"] = _safe_divide(_col(df, "number_of_cash_registers"), df["total_traffic_per_hour"], fill=0.0)

        if include_current_receipt_features:
            if {"avg_promo_items_per_receipt", "avg_items_per_receipt"}.issubset(df.columns):
                df["promo_items_share"] = _safe_divide(_col(df, "avg_promo_items_per_receipt"), _col(df, "avg_items_per_receipt"), fill=0.0)
            if {"avg_items_per_receipt", "working_hours_per_day"}.issubset(df.columns):
                df["items_x_working_hours"] = _col(df, "avg_items_per_receipt") * _col(df, "working_hours_per_day")

        df["alcohol_x_capacity"] = _col(df, "alcohol_license_flag") * df["cash_register_hours_capacity"]
        df["alcohol_x_total_traffic"] = _col(df, "alcohol_license_flag") * df["total_traffic_per_hour"]
        if include_current_receipt_features and "avg_items_per_receipt" in df.columns:
            df["alcohol_x_items"] = _col(df, "alcohol_license_flag") * _col(df, "avg_items_per_receipt")

        df["traffic_x_total_poi"] = df["total_traffic_per_hour"] * df["poi_score"]
        df["traffic_x_grocery_competition"] = df["total_traffic_per_hour"] * _col(df, "grocery_stores_500m")
        df["traffic_x_pyaterochka_competition"] = df["total_traffic_per_hour"] * _col(df, "pyaterochka_stores_500m")
        df["pedestrian_traffic_x_transport"] = _col(df, "pedestrian_traffic_per_hour") * _col(df, "public_transport_stops_300m")
        if "is_back_to_school_month" in df.columns:
            df["schools_x_back_to_school"] = _col(df, "schools_300m") * _col(df, "is_back_to_school_month")

        # Store-level counts / exogenous group means. They use only exogenous known columns, not target.
        if id_col in df.columns:
            static_cols = [
                id_col, "settlement", "region", "opening_date_category", "trading_area_category",
                "total_traffic_per_hour", "grocery_stores_500m", "pyaterochka_stores_500m",
                "number_of_cash_registers", "poi_score",
            ]
            static_cols = [c for c in static_cols if c in df.columns]
            store_static = df[static_cols].drop_duplicates(subset=[id_col]).copy()

            for group_cols, new_name in [
                (["settlement"], "settlement_store_count"),
                (["region"], "region_store_count"),
                (["region", "trading_area_category"], "region_trading_area_store_count"),
                (["region", "opening_date_category"], "region_opening_age_store_count"),
            ]:
                if all(c in store_static.columns for c in group_cols):
                    cnt = store_static.groupby(group_cols, dropna=False)[id_col].nunique().rename(new_name).reset_index()
                    df = df.merge(cnt, on=group_cols, how="left")

            exog_cols = [c for c in [
                "total_traffic_per_hour", "grocery_stores_500m", "pyaterochka_stores_500m", "number_of_cash_registers", "poi_score"
            ] if c in store_static.columns]

            if exog_cols and "region" in store_static.columns:
                agg = store_static.groupby("region", dropna=False)[exog_cols].mean().add_prefix("region_").add_suffix("_mean").reset_index()
                df = df.merge(agg, on="region", how="left")
            if exog_cols and "settlement" in store_static.columns:
                agg = store_static.groupby("settlement", dropna=False)[exog_cols].mean().add_prefix("settlement_").add_suffix("_mean").reset_index()
                df = df.merge(agg, on="settlement", how="left")

        return df

    def _prepare_base_parts(parts: dict[str, pd.DataFrame]) -> tuple[dict[str, pd.DataFrame], dict]:
        checked = {}
        for name, p in parts.items():
            p = _ensure_time_year_month(p, name)
            _validate_required(p, {id_col, time_col}, name)
            checked[name] = p

        combined = _part_concat(checked)
        combined = _drop_preexisting_generated_cols(combined)
        combined = _ensure_time_year_month(combined, "combined")

        if add_calendar_features:
            combined = _add_calendar(combined)
            combined = _ensure_time_year_month(combined, "combined_after_calendar")

        if add_static_features:
            combined = _add_static(combined)

        parts_out = _split_parts(combined, list(parts.keys()))
        return parts_out, {"base_shape": combined.shape}

    # ============================================================
    # History feature blocks
    # ============================================================

    def _combine_history_future(history_df: pd.DataFrame, future_df: pd.DataFrame, future_name: str):
        history_df = history_df.copy()
        future_df = future_df.copy()
        history_df["__part__"] = "history"
        future_df["__part__"] = future_name
        history_df["__row_order__"] = np.arange(len(history_df))
        future_df["__row_order__"] = np.arange(len(future_df))
        combined = pd.concat([history_df, future_df], axis=0, ignore_index=True, sort=False)
        combined = _ensure_time_year_month(combined, "history_future_combined")
        return combined

    def _restore_pair(combined: pd.DataFrame, future_name: str):
        history_features = (
            combined[combined["__part__"] == "history"]
            .sort_values("__row_order__")
            .drop(columns=["__part__", "__row_order__"], errors="ignore")
            .reset_index(drop=True)
        )
        future_features = (
            combined[combined["__part__"] == future_name]
            .sort_values("__row_order__")
            .drop(columns=["__part__", "__row_order__"], errors="ignore")
            .reset_index(drop=True)
        )
        return history_features, future_features

    def _merge_lag_feature(combined, history, source_col, lag, feature_name, by_cols):
        if source_col not in history.columns:
            return combined, False
        lag_df = history[by_cols + [time_col, source_col]].copy()
        lag_df = lag_df.dropna(subset=[source_col])
        lag_df[time_col] = lag_df[time_col].astype(int) + int(lag)
        lag_df = lag_df.rename(columns={source_col: feature_name})
        lag_df = lag_df.drop_duplicates(subset=by_cols + [time_col], keep="last")
        combined = combined.merge(lag_df[by_cols + [time_col, feature_name]], on=by_cols + [time_col], how="left")
        return combined, True

    def _add_target_history_pair(history_df, future_df, future_name):
        combined = _combine_history_future(history_df, future_df, future_name)
        history = combined[combined["__part__"] == "history"].copy()
        _validate_required(history, {id_col, time_col, target_col}, "history_for_target_lags")
        _validate_unique_panel(history, "history_for_target_lags")

        created = []
        for lag in target_lags:
            fname = f"{target_col}_lag_{lag}"
            combined, ok = _merge_lag_feature(combined, history, target_col, lag, fname, [id_col])
            if ok:
                created.append(fname)
                if target_history_scale in {"log", "both"}:
                    log_name = f"log_{fname}"
                    combined[log_name] = np.log1p(combined[fname].clip(lower=0))
                    created.append(log_name)

        # Rolling features are based only on past history values; future target is ignored even if present.
        if target_rolling_windows:
            tmp = combined[[id_col, time_col, "__part__", target_col]].copy()
            tmp["__target_for_roll__"] = np.where(tmp["__part__"].eq("history"), tmp[target_col], np.nan)
            tmp = tmp.sort_values([id_col, time_col], kind="mergesort")
            shifted = tmp.groupby(id_col, sort=False)["__target_for_roll__"].shift(1)
            for window in target_rolling_windows:
                rg = shifted.groupby(tmp[id_col], sort=False)
                mean_col = f"{target_col}_roll_mean_{window}"
                std_col = f"{target_col}_roll_std_{window}"
                cv_col = f"{target_col}_roll_cv_{window}"
                tmp[mean_col] = rg.rolling(window, min_periods=target_rolling_min_periods).mean().reset_index(level=0, drop=True)
                tmp[std_col] = rg.rolling(window, min_periods=target_rolling_min_periods).std().reset_index(level=0, drop=True)
                tmp[cv_col] = _safe_divide(tmp[std_col], tmp[mean_col], fill=np.nan)
                created.extend([mean_col, std_col, cv_col])
            tmp = tmp.sort_index()
            for c in created:
                if c in tmp.columns and c not in combined.columns:
                    combined[c] = tmp[c]

        def lag_col(lag): return f"{target_col}_lag_{lag}"

        for a, b in [(1, 2), (1, 3), (1, 6), (1, 12), (3, 6)]:
            if {lag_col(a), lag_col(b)}.issubset(combined.columns):
                c = f"{target_col}_diff_lag_{a}_{b}"
                combined[c] = combined[lag_col(a)] - combined[lag_col(b)]
                created.append(c)

        for a, b in [(1, 2), (1, 3), (1, 6), (1, 12), (3, 6), (6, 12)]:
            if {lag_col(a), lag_col(b)}.issubset(combined.columns):
                c = f"{target_col}_ratio_lag_{a}_{b}"
                combined[c] = _safe_divide(combined[lag_col(a)], combined[lag_col(b)], fill=np.nan)
                created.append(c)

        for window in target_rolling_windows:
            roll_mean = f"{target_col}_roll_mean_{window}"
            if {lag_col(1), roll_mean}.issubset(combined.columns):
                c = f"{target_col}_lag_1_to_roll_mean_{window}"
                combined[c] = _safe_divide(combined[lag_col(1)], combined[roll_mean], fill=np.nan)
                created.append(c)

        if {lag_col(12), f"{target_col}_roll_mean_12"}.issubset(combined.columns):
            c = f"{target_col}_lag_12_to_roll_mean_12"
            combined[c] = _safe_divide(combined[lag_col(12)], combined[f"{target_col}_roll_mean_12"], fill=np.nan)
            created.append(c)

        if {lag_col(12), lag_col(13)}.issubset(combined.columns):
            c1 = f"{target_col}_seasonal_diff_12_13"
            c2 = f"{target_col}_seasonal_ratio_12_13"
            combined[c1] = combined[lag_col(12)] - combined[lag_col(13)]
            combined[c2] = _safe_divide(combined[lag_col(12)], combined[lag_col(13)], fill=np.nan)
            created.extend([c1, c2])

        if target_history_scale in {"log", "both"} and add_log_target_dynamics:
            for a, b in [(1, 2), (1, 3), (1, 6), (1, 12), (3, 6), (6, 12)]:
                la, lb = f"log_{lag_col(a)}", f"log_{lag_col(b)}"
                if {la, lb}.issubset(combined.columns):
                    c = f"log_{target_col}_diff_lag_{a}_{b}"
                    combined[c] = combined[la] - combined[lb]
                    created.append(c)

        created = sorted(set(c for c in created if c in combined.columns))
        combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)
        if fill_history_na:
            combined[created] = combined[created].fillna(fill_value)
        return (*_restore_pair(combined, future_name), created)

    def _add_category_history_pair(history_df, future_df, future_name):
        combined = _combine_history_future(history_df, future_df, future_name)
        history = combined[combined["__part__"] == "history"].copy()
        _validate_required(history, {id_col, time_col, target_col}, "history_for_category_lags")

        specs = group_specs
        if specs is None:
            specs = [
                "region",
                "settlement",
                ["region", "trading_area_category"],
                ["region", "opening_date_category"],
                ["settlement", "trading_area_category"],
            ]
        norm_specs = []
        for spec in specs:
            spec = [spec] if isinstance(spec, str) else list(spec)
            if all(c in combined.columns for c in spec):
                norm_specs.append(spec)
        if not norm_specs:
            return (*_restore_pair(combined, future_name), [])

        created = []
        internal_store_lag = None
        store_lag_1 = f"{target_col}_lag_1"
        if store_lag_1 not in combined.columns:
            internal_store_lag = f"__{target_col}_store_lag_1__"
            combined, _ = _merge_lag_feature(combined, history, target_col, 1, internal_store_lag, [id_col])
            store_lag_1 = internal_store_lag

        for group_cols in norm_specs:
            prefix = "__".join(group_cols)
            agg_col = f"__{prefix}_{target_col}_{group_agg_stat}__"
            monthly_agg = (
                history.groupby(group_cols + [time_col], dropna=False)[target_col]
                .agg(group_agg_stat)
                .reset_index()
                .rename(columns={target_col: agg_col})
            )
            monthly_agg[time_col] = monthly_agg[time_col].astype(int) + int(group_lag)
            group_lag_col = f"{prefix}_{target_col}_{group_agg_stat}_lag_{group_lag}"
            monthly_agg = monthly_agg.rename(columns={agg_col: group_lag_col})
            combined = combined.merge(monthly_agg[group_cols + [time_col, group_lag_col]], on=group_cols + [time_col], how="left")

            if add_group_lag_feature:
                created.append(group_lag_col)

            if add_group_relative_ratio:
                c = f"{target_col}_lag_1_to_{prefix}_{target_col}_{group_agg_stat}_lag_{group_lag}"
                combined[c] = _safe_divide(combined[store_lag_1], combined[group_lag_col], fill=np.nan)
                created.append(c)

            if add_group_relative_diff:
                c = f"{target_col}_lag_1_minus_{prefix}_{target_col}_{group_agg_stat}_lag_{group_lag}"
                combined[c] = combined[store_lag_1] - combined[group_lag_col]
                created.append(c)

        if internal_store_lag is not None:
            combined = combined.drop(columns=[internal_store_lag], errors="ignore")

        created = sorted(set(c for c in created if c in combined.columns))
        combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)
        if fill_history_na:
            combined[created] = combined[created].fillna(fill_value)
        return (*_restore_pair(combined, future_name), created)

    def _add_monthly_history_pair(history_df, future_df, future_name):
        combined = _combine_history_future(history_df, future_df, future_name)
        history = combined[combined["__part__"] == "history"].copy()
        _validate_required(history, {id_col, time_col}, "history_for_monthly_lags")
        _validate_unique_panel(history, "history_for_monthly_lags")

        existing_monthly = [c for c in current_monthly_cols if c in history.columns]
        if not existing_monthly:
            return (*_restore_pair(combined, future_name), [])

        if monthly_feature_config is None:
            cfg_all = {
                "avg_promo_items_per_receipt": {
                    "lags": (1, 3, 12), "rolling_windows": (3, 12),
                    "rolling_stats": ("std", "cv"), "diff_pairs": ((1, 12),), "ratio_pairs": ((1, 12),),
                },
                "avg_items_per_receipt": {
                    "lags": (1, 3, 12), "rolling_windows": (3, 12),
                    "rolling_stats": ("std", "cv"), "diff_pairs": ((1, 12),), "ratio_pairs": ((1, 12),),
                },
                "avg_cancellations": {
                    "lags": (1, 3, 12), "rolling_windows": (3, 12),
                    "rolling_stats": ("mean", "std", "cv"), "diff_pairs": ((1, 12),), "ratio_pairs": ((1, 12),),
                },
                "working_hours_per_day": {
                    "lags": (1,), "rolling_windows": (), "rolling_stats": (), "diff_pairs": (), "ratio_pairs": (),
                },
            }
        else:
            cfg_all = monthly_feature_config

        created = []

        for colname in existing_monthly:
            cfg = cfg_all.get(colname, {
                "lags": (1, 3, 12), "rolling_windows": (3, 12),
                "rolling_stats": ("std", "cv"), "diff_pairs": ((1, 12),), "ratio_pairs": ((1, 12),),
            })

            # Time-aware lags.
            for lag in tuple(cfg.get("lags", ())):
                c = f"{colname}_lag_{lag}"
                combined, ok = _merge_lag_feature(combined, history, colname, lag, c, [id_col])
                if ok:
                    created.append(c)
                    if monthly_history_scale in {"log", "both"}:
                        lc = f"log_{colname}_lag_{lag}"
                        combined[lc] = np.log1p(combined[c].clip(lower=0))
                        created.append(lc)

            # Rolling over past observed history.
            windows = tuple(cfg.get("rolling_windows", ()))
            stats = tuple(cfg.get("rolling_stats", ()))
            if windows and stats:
                tmp = combined[[id_col, time_col, "__part__", colname]].copy()
                tmp["__x__"] = np.where(tmp["__part__"].eq("history"), tmp[colname], np.nan)
                tmp = tmp.sort_values([id_col, time_col], kind="mergesort")
                shifted = tmp.groupby(id_col, sort=False)["__x__"].shift(1)
                for window in windows:
                    rg = shifted.groupby(tmp[id_col], sort=False)
                    mean_s = std_s = min_s = max_s = None
                    if "mean" in stats or "cv" in stats:
                        mean_s = rg.rolling(window, min_periods=monthly_rolling_min_periods).mean().reset_index(level=0, drop=True)
                        if "mean" in stats:
                            c = f"{colname}_roll_mean_{window}"
                            tmp[c] = mean_s
                            created.append(c)
                    if "std" in stats or "cv" in stats:
                        std_s = rg.rolling(window, min_periods=monthly_rolling_min_periods).std().reset_index(level=0, drop=True)
                        if "std" in stats:
                            c = f"{colname}_roll_std_{window}"
                            tmp[c] = std_s
                            created.append(c)
                    if "min" in stats or "range" in stats:
                        min_s = rg.rolling(window, min_periods=monthly_rolling_min_periods).min().reset_index(level=0, drop=True)
                        if "min" in stats:
                            c = f"{colname}_roll_min_{window}"
                            tmp[c] = min_s
                            created.append(c)
                    if "max" in stats or "range" in stats:
                        max_s = rg.rolling(window, min_periods=monthly_rolling_min_periods).max().reset_index(level=0, drop=True)
                        if "max" in stats:
                            c = f"{colname}_roll_max_{window}"
                            tmp[c] = max_s
                            created.append(c)
                    if "range" in stats:
                        c = f"{colname}_roll_range_{window}"
                        tmp[c] = max_s - min_s
                        created.append(c)
                    if "cv" in stats:
                        c = f"{colname}_roll_cv_{window}"
                        tmp[c] = _safe_divide(std_s, mean_s, fill=np.nan)
                        created.append(c)
                tmp = tmp.sort_index()
                for c in created:
                    if c in tmp.columns and c not in combined.columns:
                        combined[c] = tmp[c]

            for a, b in tuple(cfg.get("diff_pairs", ())):
                ca, cb = f"{colname}_lag_{a}", f"{colname}_lag_{b}"
                if {ca, cb}.issubset(combined.columns):
                    c = f"{colname}_diff_lag_{a}_{b}"
                    combined[c] = combined[ca] - combined[cb]
                    created.append(c)

            for a, b in tuple(cfg.get("ratio_pairs", ())):
                ca, cb = f"{colname}_lag_{a}", f"{colname}_lag_{b}"
                if {ca, cb}.issubset(combined.columns):
                    c = f"{colname}_ratio_lag_{a}_{b}"
                    combined[c] = _safe_divide(combined[ca], combined[cb], fill=np.nan)
                    created.append(c)

        if add_monthly_cross_lags:
            for lag in monthly_cross_lags:
                promo = f"avg_promo_items_per_receipt_lag_{lag}"
                items = f"avg_items_per_receipt_lag_{lag}"
                canc = f"avg_cancellations_lag_{lag}"
                hours = f"working_hours_per_day_lag_{lag}"

                if "promo_share" in monthly_cross_feature_types and {promo, items}.issubset(combined.columns):
                    c = f"promo_items_share_lag_{lag}"
                    combined[c] = _safe_divide(combined[promo], combined[items], fill=np.nan)
                    created.append(c)
                if "promo_x_items" in monthly_cross_feature_types and {promo, items}.issubset(combined.columns):
                    c = f"promo_x_items_lag_{lag}"
                    combined[c] = combined[promo] * combined[items]
                    created.append(c)
                if "items_x_hours" in monthly_cross_feature_types and {items, hours}.issubset(combined.columns):
                    c = f"items_x_working_hours_lag_{lag}"
                    combined[c] = combined[items] * combined[hours]
                    created.append(c)
                if "cancellations_per_item" in monthly_cross_feature_types and {canc, items}.issubset(combined.columns):
                    c = f"cancellations_per_item_lag_{lag}"
                    combined[c] = _safe_divide(combined[canc], combined[items], fill=np.nan)
                    created.append(c)
                if "cancellations_per_hour" in monthly_cross_feature_types and {canc, hours}.issubset(combined.columns):
                    c = f"cancellations_per_working_hour_lag_{lag}"
                    combined[c] = _safe_divide(combined[canc], combined[hours], fill=np.nan)
                    created.append(c)

        created = sorted(set(c for c in created if c in combined.columns))
        combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)
        if fill_history_na:
            combined[created] = combined[created].fillna(fill_value)

        if drop_current_monthly_cols:
            combined = combined.drop(columns=list(existing_monthly), errors="ignore")
        if drop_current_monthly_derived_cols:
            combined = combined.drop(columns=list(current_monthly_derived_cols), errors="ignore")

        return (*_restore_pair(combined, future_name), created)

    # ============================================================
    # Final cleanup
    # ============================================================

    def _future_unknown_cols():
        cols = []
        if drop_current_monthly_cols:
            cols.extend(list(current_monthly_cols))
        if drop_current_monthly_derived_cols:
            cols.extend(list(current_monthly_derived_cols))
        if not working_hours_future_known:
            cols.extend([
                "working_hours_per_day",
                "cash_register_hours_capacity",
                "traffic_per_working_hour",
                "alcohol_x_capacity",
                "items_x_working_hours",
            ])
        if drop_year_col:
            cols.append(year_col)
        return sorted(set(cols))

    def _obvious_duplicate_cols():
        cols = [
            f"{target_col}_roll_mean_3", f"{target_col}_roll_mean_12",
            f"{target_col}_lag_13", f"{target_col}_diff_lag_3_6", f"{target_col}_ratio_lag_3_6",
            "avg_items_per_receipt_diff_lag_1_12",
            "avg_cancellations_roll_mean_3", "avg_cancellations_roll_mean_12",
            "cancellations_per_item_lag_1", "cancellations_per_item_lag_3", "cancellations_per_item_lag_12",
            "settlement_store_count", "total_traffic_per_hour", "non_pyaterochka_grocery_stores_500m",
        ]

        if target_history_scale == "raw":
            cols.extend([f"log_{target_col}_lag_{l}" for l in target_lags])
            cols.extend([c for c in []])
        elif target_history_scale == "log":
            raw_lags = [f"{target_col}_lag_{l}" for l in target_lags if l != 1]
            if not keep_raw_pto_lag_1_anchor_for_log:
                raw_lags.append(f"{target_col}_lag_1")
            cols.extend(raw_lags)
            cols.extend([
                f"{target_col}_diff_lag_1_2", f"{target_col}_diff_lag_1_3",
                f"{target_col}_diff_lag_1_6", f"{target_col}_diff_lag_1_12",
                f"{target_col}_diff_lag_3_6", f"{target_col}_ratio_lag_1_2",
                f"{target_col}_ratio_lag_1_3", f"{target_col}_ratio_lag_1_6",
                f"{target_col}_ratio_lag_1_12", f"{target_col}_ratio_lag_3_6",
                f"{target_col}_ratio_lag_6_12", f"{target_col}_seasonal_diff_12_13",
                f"{target_col}_seasonal_ratio_12_13",
            ])

        if monthly_history_scale == "raw":
            for c in current_monthly_cols:
                cols.extend([f"log_{c}_lag_1", f"log_{c}_lag_3", f"log_{c}_lag_12"])
        elif monthly_history_scale == "log":
            for c in current_monthly_cols:
                cols.extend([f"{c}_lag_3", f"{c}_lag_12"])

        if extra_drop_cols is not None:
            cols.extend(list(extra_drop_cols))
        return sorted(set(cols))

    def _apply_final_clean(train_part, future_part):
        train_part = train_part.copy()
        future_part = future_part.copy()
        drop_cols = _future_unknown_cols()
        if drop_obvious_duplicates:
            drop_cols.extend(_obvious_duplicate_cols())
        drop_cols = sorted(set(c for c in drop_cols if c in train_part.columns or c in future_part.columns))
        train_part = train_part.drop(columns=drop_cols, errors="ignore")
        future_part = future_part.drop(columns=drop_cols, errors="ignore")

        # Replace infinities only; do not fill all lag NaNs unless requested.
        for df_ in (train_part, future_part):
            num_cols = df_.select_dtypes(include=[np.number]).columns
            df_[num_cols] = df_[num_cols].replace([np.inf, -np.inf], np.nan)
            if fill_history_na:
                df_[num_cols] = df_[num_cols].fillna(fill_value)

        # Align columns in stable order.
        ordered_cols = list(train_part.columns)
        for c in future_part.columns:
            if c not in ordered_cols:
                ordered_cols.append(c)
        for c in ordered_cols:
            if c not in train_part.columns:
                train_part[c] = np.nan
            if c not in future_part.columns:
                future_part[c] = np.nan
        train_part = train_part[ordered_cols]
        future_part = future_part[ordered_cols]
        return train_part, future_part, drop_cols

    def _downcast(df):
        if not downcast_float_features:
            return df
        df = df.copy()
        float_cols = [c for c in df.select_dtypes(include=["float64"]).columns if c != target_col]
        if float_cols:
            df[float_cols] = df[float_cols].astype("float32")
        return df

    def _generate_pair(history_base, future_base, future_name):
        info = {}
        h, f = history_base.copy(), future_base.copy()

        if add_target_history:
            h, f, created = _add_target_history_pair(h, f, future_name)
            info["target_history_features"] = created
        else:
            info["target_history_features"] = []

        if add_category_target_history:
            h, f, created = _add_category_history_pair(h, f, future_name)
            info["category_target_history_features"] = created
        else:
            info["category_target_history_features"] = []

        if add_non_target_monthly_history:
            h, f, created = _add_monthly_history_pair(h, f, future_name)
            info["non_target_monthly_history_features"] = created
        else:
            info["non_target_monthly_history_features"] = []
            if drop_current_monthly_cols:
                h = h.drop(columns=[c for c in current_monthly_cols if c in h.columns], errors="ignore")
                f = f.drop(columns=[c for c in current_monthly_cols if c in f.columns], errors="ignore")
            if drop_current_monthly_derived_cols:
                h = h.drop(columns=[c for c in current_monthly_derived_cols if c in h.columns], errors="ignore")
                f = f.drop(columns=[c for c in current_monthly_derived_cols if c in f.columns], errors="ignore")

        h, f, dropped = _apply_final_clean(h, f)
        info["dropped_final_cols"] = dropped
        h = _downcast(h)
        f = _downcast(f)
        return h, f, info

    # ============================================================
    # Main flow
    # ============================================================

    train_df = _ensure_time_year_month(train_df, "train_df")
    _validate_required(train_df, {id_col, time_col, target_col}, "train_df")
    _validate_unique_panel(train_df, "train_df")

    if make_val_from_train:
        if val_df is not None:
            raise ValueError("Нельзя одновременно передать val_df и make_val_from_train=True.")
        if val_time_idx is None:
            val_time_idx = int(train_df[time_col].max())
        val_df = train_df[train_df[time_col] == val_time_idx].copy()
        train_df = train_df[train_df[time_col] < val_time_idx].copy()
        if verbose:
            print(f"Auto val_time_idx: {val_time_idx}")
            print("Auto train shape:", train_df.shape)
            print("Auto val shape:", val_df.shape)

    if val_df is not None:
        val_df = _ensure_time_year_month(val_df, "val_df")
        _validate_required(val_df, {id_col, time_col}, "val_df")
        _validate_unique_panel(val_df, "val_df")

    if test_df is not None:
        test_df = _ensure_time_year_month(test_df, "test_df")
        _validate_required(test_df, {id_col, time_col}, "test_df")
        _validate_unique_panel(test_df, "test_df")
        if target_col not in test_df.columns:
            test_df[target_col] = np.nan

    output = {
        "train_features": None,
        "val_features": None,
        "test_train_features": None,
        "test_features": None,
        "feature_cols": None,
        "categorical_cols": None,
        "feature_info": {},
    }

    if val_df is not None:
        base_parts, base_info = _prepare_base_parts({"train": train_df, "val": val_df})
        train_features, val_features, val_info = _generate_pair(base_parts["train"], base_parts["val"], "val")
        output["train_features"] = train_features
        output["val_features"] = val_features
        output["feature_info"]["val"] = {**base_info, **val_info}
        if verbose:
            print("Generated train_features:", train_features.shape)
            print("Generated val_features:", val_features.shape)

    if test_df is not None:
        if val_df is not None and use_val_as_test_history:
            test_history = pd.concat([train_df, val_df], axis=0, ignore_index=True, sort=False)
        else:
            test_history = train_df.copy()
        test_history = _ensure_time_year_month(test_history, "test_history")
        _validate_unique_panel(test_history, "test_history")

        base_parts, base_info = _prepare_base_parts({"train": test_history, "test": test_df})
        test_train_features, test_features, test_info = _generate_pair(base_parts["train"], base_parts["test"], "test")
        output["test_train_features"] = test_train_features
        output["test_features"] = test_features
        output["feature_info"]["test"] = {**base_info, **test_info}
        if verbose:
            print("Generated test_train_features:", test_train_features.shape)
            print("Generated test_features:", test_features.shape)

    # Select columns for model.
    ref = output["test_train_features"] if output["test_train_features"] is not None else output["train_features"]
    if ref is not None:
        exclude = {target_col}
        feature_cols = [c for c in ref.columns if c not in exclude]
        categorical_cols = [
            c for c in feature_cols
            if c in ref.columns and (pd.api.types.is_object_dtype(ref[c]) or pd.api.types.is_categorical_dtype(ref[c]))
        ]
        output["feature_cols"] = feature_cols
        output["categorical_cols"] = categorical_cols

    gc.collect()
    return output


def _ensure_time_year_month_external(
    df: pd.DataFrame,
    *,
    year_col: str,
    month_col: str,
    time_col: str,
    min_year: int,
    name: str = "df",
) -> pd.DataFrame:
    """Внешний helper для wrapper/enhancer: гарантирует year/month/time."""
    out = df.copy()
    has_year_month = year_col in out.columns and month_col in out.columns
    has_time = time_col in out.columns

    if not has_year_month and not has_time:
        raise ValueError(f"В {name} нет ни year/month, ни time_col='{time_col}'.")

    if has_year_month:
        out[year_col] = pd.to_numeric(out[year_col], errors="raise").astype(int)
        out[month_col] = pd.to_numeric(out[month_col], errors="raise").astype(int)
        out[time_col] = ((out[year_col] - int(min_year)) * 12 + (out[month_col] - 1)).astype(int)
    else:
        out[time_col] = pd.to_numeric(out[time_col], errors="raise").astype(int)
        out[year_col] = (int(min_year) + out[time_col] // 12).astype(int)
        out[month_col] = (out[time_col] % 12 + 1).astype(int)

    return out


def _calendar_month_from_time(time_s: pd.Series, min_year: int) -> pd.Series:
    return (time_s.astype(int) % 12 + 1).astype(int)


def _year_from_time(time_s: pd.Series, min_year: int) -> pd.Series:
    return (int(min_year) + time_s.astype(int) // 12).astype(int)


def _days_in_month_from_year_month(year_s: pd.Series, month_s: pd.Series) -> pd.Series:
    dt = pd.to_datetime(
        {
            "year": pd.to_numeric(year_s, errors="coerce").astype("Int64"),
            "month": pd.to_numeric(month_s, errors="coerce").astype("Int64"),
            "day": 1,
        },
        errors="coerce",
    )
    return dt.dt.days_in_month.astype("float32")


def _safe_divide_external(a, b, fill=np.nan):
    a = a if isinstance(a, pd.Series) else pd.Series(a)
    b = b if isinstance(b, pd.Series) else pd.Series(b)
    out = a.astype(float) / b.replace(0, np.nan).astype(float)
    out = out.replace([np.inf, -np.inf], np.nan)
    if fill is not None and not (isinstance(fill, float) and np.isnan(fill)):
        out = out.fillna(fill)
    return out


def _safe_log_external(x, eps: float = 1e-9):
    return np.log(np.clip(pd.Series(x).astype(float), eps, None))



def _assign_lag1_rto_size_bin(
    df: pd.DataFrame,
    *,
    time_col: str,
    lag1_col: str,
    out_col: str = "lag1_rto_size_bin",
    n_bins: int = 5,
) -> pd.Series:
    """
    Cross-sectional size bin by pto_lag_1 inside each forecast month.
    Uses only lagged target values, therefore it is safe for future rows.
    """
    result = pd.Series(np.nan, index=df.index, dtype="object")

    if lag1_col not in df.columns:
        return result

    for _, idx in df.groupby(time_col, sort=False).groups.items():
        s = pd.to_numeric(df.loc[idx, lag1_col], errors="coerce")
        valid = s.notna()
        if valid.sum() == 0:
            continue
        if valid.sum() == 1 or s[valid].nunique(dropna=True) == 1:
            result.loc[s[valid].index] = "q3"
            continue
        q = int(min(n_bins, valid.sum(), s[valid].nunique(dropna=True)))
        if q < 2:
            result.loc[s[valid].index] = "q3"
            continue
        try:
            codes = pd.qcut(
                s[valid].rank(method="first"),
                q=q,
                labels=False,
                duplicates="drop",
            )
            result.loc[s[valid].index] = [f"q{int(x) + 1}" if pd.notna(x) else np.nan for x in codes]
        except ValueError:
            result.loc[s[valid].index] = "q3"

    return result


def _expanding_median_factor_from_history(
    combined: pd.DataFrame,
    observed: pd.DataFrame,
    *,
    group_cols: list[str],
    time_col: str,
    obs_col: str,
    out_col: str,
) -> pd.Series:
    """
    For every row in combined returns median(obs_col) over historical observations
    with the same group key and strictly earlier time_col.

    Implementation detail: query rows are sorted before observation rows at the
    same time index, so same-month target is never used for the same row/month.
    """
    keys = list(group_cols) + [time_col]
    result = pd.Series(np.nan, index=combined.index, dtype="float64")

    if not group_cols or obs_col not in observed.columns:
        return result
    if any(c not in combined.columns for c in keys) or any(c not in observed.columns for c in keys):
        return result

    obs = observed[keys + [obs_col]].dropna(subset=[obs_col]).copy()
    if obs.empty:
        return result

    obs_gt = (
        obs.groupby(keys, dropna=False, as_index=False)[obs_col]
        .median()
        .rename(columns={obs_col: "__obs_factor__"})
    )
    if obs_gt.empty:
        return result

    queries = combined[keys].copy()
    queries["__query_index__"] = combined.index
    queries["__obs_factor__"] = np.nan
    queries["__is_query__"] = 1
    queries["__order__"] = 0

    obs_gt["__query_index__"] = -1
    obs_gt["__is_query__"] = 0
    obs_gt["__order__"] = 1

    stack = pd.concat([queries, obs_gt], ignore_index=True, sort=False)
    stack = stack.sort_values(group_cols + [time_col, "__order__"], kind="mergesort")

    stack[out_col] = (
        stack.groupby(group_cols, dropna=False)["__obs_factor__"]
        .transform(lambda s: s.expanding(min_periods=1).median())
    )

    q = stack[stack["__is_query__"].eq(1)][["__query_index__", out_col]].copy()
    q = q.drop_duplicates("__query_index__", keep="last")
    result.loc[q["__query_index__"].astype(int).values] = q[out_col].values
    return result


def _add_expanding_calendar_heuristic_features(
    combined: pd.DataFrame,
    *,
    id_col: str,
    time_col: str,
    target_col: str,
    min_year: int,
    eps: float = 1e-9,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Adds the old expanding calendar heuristic block without target leakage.

    Exposed features:
    - lag1_rto_size_bin;
    - same_month_calendar_factor;
    - same_month_region_calendar_factor;
    - same_month_population_type_calendar_factor;
    - same_month_lag1_size_calendar_factor;
    - calendar_growth_multiplier;
    - calendar_heuristic_*_prediction;
    - log_calendar_heuristic_prediction;
    - log_pto_lag_1_to_calendar_heuristic;
    - log_pto_lag_1_to_day_scaled_lag1.

    Internal observed_calendar_factor is deliberately not returned as a model
    feature, because it uses current target on history rows.
    """
    out = combined.copy()
    created: list[str] = []

    def mark(col: str):
        if col in out.columns and col not in created:
            created.append(col)

    pto_lag_1 = f"{target_col}_lag_1"
    required = {"__part__", time_col, target_col, pto_lag_1, "day_scaled_lag1_prediction", "month_day_ratio_to_previous_month"}
    if not required.issubset(out.columns):
        return out, created

    if "calendar_month_num" not in out.columns:
        out["calendar_month_num"] = _calendar_month_from_time(out[time_col], min_year)
        mark("calendar_month_num")

    out["lag1_rto_size_bin"] = _assign_lag1_rto_size_bin(
        out,
        time_col=time_col,
        lag1_col=pto_lag_1,
        out_col="lag1_rto_size_bin",
        n_bins=5,
    )
    mark("lag1_rto_size_bin")

    # Useful log-ratio against the pure day-scaled anchor.
    if f"log_{target_col}_lag_1" not in out.columns:
        out[f"log_{target_col}_lag_1"] = _safe_log_external(out[pto_lag_1], eps=eps)
        mark(f"log_{target_col}_lag_1")
    if "log_day_scaled_lag1_prediction" not in out.columns:
        out["log_day_scaled_lag1_prediction"] = _safe_log_external(out["day_scaled_lag1_prediction"], eps=eps)
        mark("log_day_scaled_lag1_prediction")

    out["log_pto_lag_1_to_day_scaled_lag1"] = (
        out[f"log_{target_col}_lag_1"] - out["log_day_scaled_lag1_prediction"]
    )
    mark("log_pto_lag_1_to_day_scaled_lag1")

    hist_mask = out["__part__"].eq("history") & out[target_col].notna()
    observed = out.loc[hist_mask].copy()
    observed["__observed_calendar_factor__"] = _safe_divide_external(
        observed[target_col],
        observed["day_scaled_lag1_prediction"],
        fill=np.nan,
    )
    observed["__observed_calendar_factor__"] = observed["__observed_calendar_factor__"].replace(
        [np.inf, -np.inf], np.nan
    )
    observed = observed.dropna(subset=["__observed_calendar_factor__"])

    if observed.empty:
        return out, created

    factor_specs = [
        (["calendar_month_num"], "same_month_calendar_factor", None),
    ]
    if "region" in out.columns:
        factor_specs.append((["calendar_month_num", "region"], "same_month_region_calendar_factor", "same_month_calendar_factor"))
    if "population_city_type" in out.columns:
        factor_specs.append((["calendar_month_num", "population_city_type"], "same_month_population_type_calendar_factor", "same_month_calendar_factor"))
    if "lag1_rto_size_bin" in out.columns:
        factor_specs.append((["calendar_month_num", "lag1_rto_size_bin"], "same_month_lag1_size_calendar_factor", "same_month_calendar_factor"))

    for group_cols, factor_col, fallback_col in factor_specs:
        if any(c not in out.columns for c in group_cols):
            continue
        factor = _expanding_median_factor_from_history(
            out,
            observed,
            group_cols=group_cols,
            time_col=time_col,
            obs_col="__observed_calendar_factor__",
            out_col=factor_col,
        )
        out[factor_col] = factor
        if fallback_col is not None and fallback_col in out.columns:
            out[factor_col] = out[factor_col].fillna(out[fallback_col])
        mark(factor_col)

    if "same_month_calendar_factor" in out.columns:
        out["calendar_growth_multiplier"] = (
            out["month_day_ratio_to_previous_month"] * out["same_month_calendar_factor"]
        )
        out["calendar_heuristic_prediction"] = out[pto_lag_1] * out["calendar_growth_multiplier"]
        out["log_calendar_heuristic_prediction"] = _safe_log_external(
            out["calendar_heuristic_prediction"], eps=eps
        )
        out["log_pto_lag_1_to_calendar_heuristic"] = (
            out[f"log_{target_col}_lag_1"] - out["log_calendar_heuristic_prediction"]
        )
        mark("calendar_growth_multiplier")
        mark("calendar_heuristic_prediction")
        mark("log_calendar_heuristic_prediction")
        mark("log_pto_lag_1_to_calendar_heuristic")

    pred_specs = [
        ("same_month_region_calendar_factor", "calendar_heuristic_region_prediction"),
        ("same_month_population_type_calendar_factor", "calendar_heuristic_population_type_prediction"),
        ("same_month_lag1_size_calendar_factor", "calendar_heuristic_lag1_size_prediction"),
    ]
    for factor_col, pred_col in pred_specs:
        if factor_col not in out.columns:
            continue
        out[pred_col] = out[pto_lag_1] * out["month_day_ratio_to_previous_month"] * out[factor_col]
        out[f"log_{pred_col}"] = _safe_log_external(out[pred_col], eps=eps)
        mark(pred_col)
        mark(f"log_{pred_col}")

    created = sorted(set(c for c in created if c in out.columns))
    out[created] = out[created].replace([np.inf, -np.inf], np.nan)
    return out, created

def _add_integrated_anchor_features_pair(
    history_features: pd.DataFrame,
    future_features: pd.DataFrame,
    history_raw: pd.DataFrame,
    future_raw: pd.DataFrame,
    *,
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    min_year: int = 2023,
    target_lags: tuple[int, ...] = (1, 2, 3, 6, 12, 13),
    add_calendar_anchor_features: bool = True,
    add_expanding_calendar_heuristic_features: bool = True,
    add_daily_anchor_features: bool = True,
    add_transition_anchor_features: bool = True,
    add_march_2024_features: bool = True,
    add_hierarchy_share_features: bool = True,
    add_operational_lag_features: bool = True,
    group_cols_for_transition: list | None = None,
    lagged_monthly_cols: tuple[str, ...] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    clip_growth_low: float = 0.70,
    clip_growth_high: float = 1.45,
    eps: float = 1e-9,
    fill_history_na: bool = False,
    fill_value: float = 0.0,
    downcast_float_features: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """
    Добавляет идеи из build_lag1_features_for_pipeline,
    build_anchor_features_for_pipeline и build_march_hierarchy_ops_features_for_pipeline
    к уже собранным leak-free features.

    Важная логика:
    - лаги target и monthly-cols строятся time-aware merge'ем из history_raw;
    - current target future_features не используется;
    - мартовские коэффициенты 2024 считаются только по history_raw;
    - expanding calendar heuristic считается только по history rows и строго прошлым time_idx;
    - мартовские коэффициенты доступны только для строк после марта 2024, чтобы не
      протаскивать target марта 2024 в более ранние train-строки.
    """
    h_feat = history_features.copy()
    f_feat = future_features.copy()

    h_feat["__part__"] = "history"
    f_feat["__part__"] = "future"
    h_feat["__row_order__"] = np.arange(len(h_feat))
    f_feat["__row_order__"] = np.arange(len(f_feat))

    combined = pd.concat([h_feat, f_feat], ignore_index=True, sort=False)
    combined = _ensure_time_year_month_external(
        combined,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        min_year=min_year,
        name="combined_features",
    )

    history_raw = _ensure_time_year_month_external(
        history_raw,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        min_year=min_year,
        name="history_raw",
    )
    future_raw = _ensure_time_year_month_external(
        future_raw,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        min_year=min_year,
        name="future_raw",
    )

    if id_col not in combined.columns:
        raise ValueError(f"В features нет id_col='{id_col}'.")
    if target_col not in history_raw.columns:
        raise ValueError(f"В history_raw нет target_col='{target_col}'.")

    created: list[str] = []

    def _mark(col):
        if col in combined.columns and col not in created:
            created.append(col)

    def _merge_history_lag(source_history, source_col, lag, feature_name, by_cols):
        nonlocal combined
        if source_col not in source_history.columns:
            return False

        lag_df = source_history[by_cols + [time_col, source_col]].copy()
        lag_df = lag_df.dropna(subset=[source_col])
        lag_df[time_col] = lag_df[time_col].astype(int) + int(lag)
        lag_df = lag_df.rename(columns={source_col: feature_name})
        lag_df = lag_df.drop_duplicates(subset=by_cols + [time_col], keep="last")

        combined = combined.drop(columns=[feature_name], errors="ignore")
        combined = combined.merge(
            lag_df[by_cols + [time_col, feature_name]],
            on=by_cols + [time_col],
            how="left",
        )
        _mark(feature_name)
        return True

    # ------------------------------------------------------------
    # 1. Calendar day features + target lags/daily lags
    # ------------------------------------------------------------
    combined["calendar_month_num"] = _calendar_month_from_time(combined[time_col], min_year)
    combined["calendar_year"] = _year_from_time(combined[time_col], min_year)
    combined["days_in_month"] = _days_in_month_from_year_month(
        combined["calendar_year"], combined["calendar_month_num"]
    )
    _mark("calendar_month_num")
    _mark("days_in_month")

    for lag in target_lags:
        lag = int(lag)
        pto_lag_col = f"{target_col}_lag_{lag}"
        _merge_history_lag(history_raw, target_col, lag, pto_lag_col, [id_col])

        lag_time = combined[time_col].astype(int) - lag
        lag_year = _year_from_time(lag_time, min_year)
        lag_month = _calendar_month_from_time(lag_time, min_year)
        days_lag_col = f"days_lag_{lag}"
        combined[days_lag_col] = _days_in_month_from_year_month(lag_year, lag_month)
        _mark(days_lag_col)

        daily_col = f"daily_{target_col}_lag_{lag}"
        combined[daily_col] = _safe_divide_external(
            combined[pto_lag_col],
            combined[days_lag_col],
            fill=np.nan,
        )
        _mark(daily_col)

        log_pto_col = f"log_{target_col}_lag_{lag}"
        log_daily_col = f"log_daily_{target_col}_lag_{lag}"
        combined[log_pto_col] = _safe_log_external(combined[pto_lag_col], eps=eps)
        combined[log_daily_col] = _safe_log_external(combined[daily_col], eps=eps)
        _mark(log_pto_col)
        _mark(log_daily_col)

    if "days_lag_1" in combined.columns:
        combined["days_in_prev_month"] = combined["days_lag_1"]
        _mark("days_in_prev_month")

    if add_calendar_anchor_features and {f"{target_col}_lag_1", "days_in_month", "days_in_prev_month"}.issubset(combined.columns):
        combined["month_day_ratio_to_previous_month"] = _safe_divide_external(
            combined["days_in_month"],
            combined["days_in_prev_month"],
            fill=np.nan,
        )
        combined["day_scaled_lag1_prediction"] = (
            combined[f"{target_col}_lag_1"] * combined["month_day_ratio_to_previous_month"]
        )
        combined["log_day_scaled_lag1_prediction"] = _safe_log_external(
            combined["day_scaled_lag1_prediction"], eps=eps
        )
        _mark("month_day_ratio_to_previous_month")
        _mark("day_scaled_lag1_prediction")
        _mark("log_day_scaled_lag1_prediction")

        for lag in [2, 3, 6, 12]:
            daily_col = f"daily_{target_col}_lag_{lag}"
            pred_col = f"day_scaled_lag{lag}_prediction"
            if daily_col in combined.columns:
                combined[pred_col] = combined[daily_col] * combined["days_in_month"]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

    if add_calendar_anchor_features and add_expanding_calendar_heuristic_features:
        combined, calendar_created = _add_expanding_calendar_heuristic_features(
            combined,
            id_col=id_col,
            time_col=time_col,
            target_col=target_col,
            min_year=min_year,
            eps=eps,
        )
        for c in calendar_created:
            _mark(c)

    # ------------------------------------------------------------
    # 2. Recent daily baselines + trend anchors + same-transition anchors
    # ------------------------------------------------------------
    if add_daily_anchor_features:
        daily_1 = f"daily_{target_col}_lag_1"
        daily_2 = f"daily_{target_col}_lag_2"
        daily_3 = f"daily_{target_col}_lag_3"

        if {daily_1, daily_2}.issubset(combined.columns):
            combined["recent_daily_mean_2"] = combined[[daily_1, daily_2]].mean(axis=1)
            combined["recent_daily_mean_2_prediction"] = combined["recent_daily_mean_2"] * combined["days_in_month"]
            combined["log_recent_daily_mean_2_prediction"] = _safe_log_external(
                combined["recent_daily_mean_2_prediction"], eps=eps
            )
            _mark("recent_daily_mean_2")
            _mark("recent_daily_mean_2_prediction")
            _mark("log_recent_daily_mean_2_prediction")

        if {daily_1, daily_2, daily_3}.issubset(combined.columns):
            recent_cols = [daily_1, daily_2, daily_3]
            combined["recent_daily_mean_3"] = combined[recent_cols].mean(axis=1)
            combined["recent_daily_median_3"] = combined[recent_cols].median(axis=1)
            combined["recent_daily_weighted_123"] = (
                0.60 * combined[daily_1]
                + 0.25 * combined[daily_2]
                + 0.15 * combined[daily_3]
            )

            for col in ["recent_daily_mean_3", "recent_daily_median_3", "recent_daily_weighted_123"]:
                pred_col = f"{col}_prediction"
                combined[pred_col] = combined[col] * combined["days_in_month"]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(col)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

            combined["daily_growth_1_2"] = _safe_divide_external(combined[daily_1], combined[daily_2], fill=np.nan)
            combined["daily_growth_1_3"] = _safe_divide_external(combined[daily_1], combined[daily_3], fill=np.nan)
            combined["daily_growth_1_2_clipped"] = combined["daily_growth_1_2"].clip(0.70, 1.35)
            combined["daily_growth_1_3_clipped"] = combined["daily_growth_1_3"].clip(0.70, 1.35)
            _mark("daily_growth_1_2")
            _mark("daily_growth_1_3")
            _mark("daily_growth_1_2_clipped")
            _mark("daily_growth_1_3_clipped")

            if "day_scaled_lag1_prediction" in combined.columns:
                combined["trend_day_scaled_lag1_prediction"] = (
                    combined["day_scaled_lag1_prediction"] * combined["daily_growth_1_2_clipped"]
                )
                combined["trend_day_scaled_lag1_prediction_smooth"] = (
                    combined["day_scaled_lag1_prediction"] * np.sqrt(combined["daily_growth_1_2_clipped"])
                )
                combined["log_trend_day_scaled_lag1_prediction"] = _safe_log_external(
                    combined["trend_day_scaled_lag1_prediction"], eps=eps
                )
                combined["log_trend_day_scaled_lag1_prediction_smooth"] = _safe_log_external(
                    combined["trend_day_scaled_lag1_prediction_smooth"], eps=eps
                )
                _mark("trend_day_scaled_lag1_prediction")
                _mark("trend_day_scaled_lag1_prediction_smooth")
                _mark("log_trend_day_scaled_lag1_prediction")
                _mark("log_trend_day_scaled_lag1_prediction_smooth")

    if add_transition_anchor_features:
        pto12, pto13 = f"{target_col}_lag_12", f"{target_col}_lag_13"
        daily1, daily12, daily13 = (
            f"daily_{target_col}_lag_1",
            f"daily_{target_col}_lag_12",
            f"daily_{target_col}_lag_13",
        )

        if {pto12, pto13, f"{target_col}_lag_1"}.issubset(combined.columns):
            combined["same_transition_last_year_factor"] = _safe_divide_external(
                combined[pto12], combined[pto13], fill=np.nan
            )
            combined["same_transition_last_year_factor_clipped"] = (
                combined["same_transition_last_year_factor"].clip(0.75, 1.45)
            )
            combined["same_transition_last_year_prediction"] = (
                combined[f"{target_col}_lag_1"] * combined["same_transition_last_year_factor_clipped"]
            )
            combined["log_same_transition_last_year_prediction"] = _safe_log_external(
                combined["same_transition_last_year_prediction"], eps=eps
            )
            _mark("same_transition_last_year_factor")
            _mark("same_transition_last_year_factor_clipped")
            _mark("same_transition_last_year_prediction")
            _mark("log_same_transition_last_year_prediction")

        if {daily1, daily12, daily13, "days_in_month"}.issubset(combined.columns):
            combined["daily_same_transition_last_year_factor"] = _safe_divide_external(
                combined[daily12], combined[daily13], fill=np.nan
            )
            combined["daily_same_transition_last_year_factor_clipped"] = (
                combined["daily_same_transition_last_year_factor"].clip(0.75, 1.35)
            )
            combined["daily_same_transition_last_year_prediction"] = (
                combined[daily1]
                * combined["daily_same_transition_last_year_factor_clipped"]
                * combined["days_in_month"]
            )
            combined["log_daily_same_transition_last_year_prediction"] = _safe_log_external(
                combined["daily_same_transition_last_year_prediction"], eps=eps
            )
            _mark("daily_same_transition_last_year_factor")
            _mark("daily_same_transition_last_year_factor_clipped")
            _mark("daily_same_transition_last_year_prediction")
            _mark("log_daily_same_transition_last_year_prediction")

            combined["global_daily_transition_factor"] = (
                combined.groupby(time_col, dropna=False)["daily_same_transition_last_year_factor_clipped"]
                .transform("median")
            )
            combined["global_transition_prediction"] = (
                combined[daily1] * combined["global_daily_transition_factor"] * combined["days_in_month"]
            )
            combined["log_global_transition_prediction"] = _safe_log_external(
                combined["global_transition_prediction"], eps=eps
            )
            _mark("global_daily_transition_factor")
            _mark("global_transition_prediction")
            _mark("log_global_transition_prediction")

            if group_cols_for_transition is None:
                group_cols_for_transition = []
                for col in ["region", "trading_area_category", "opening_date_category", "settlement"]:
                    if col in combined.columns:
                        group_cols_for_transition.append([col])
                if {"region", "trading_area_category"}.issubset(combined.columns):
                    group_cols_for_transition.append(["region", "trading_area_category"])
                if {"region", "opening_date_category"}.issubset(combined.columns):
                    group_cols_for_transition.append(["region", "opening_date_category"])

            for group_cols in group_cols_for_transition:
                existing = [c for c in group_cols if c in combined.columns]
                if not existing:
                    continue
                group_name = "__".join(existing)
                factor_col = f"{group_name}_daily_transition_factor"
                pred_col = f"{group_name}_transition_prediction"
                combined[factor_col] = (
                    combined.groupby([time_col] + existing, dropna=False)["daily_same_transition_last_year_factor_clipped"]
                    .transform("median")
                )
                combined[pred_col] = combined[daily1] * combined[factor_col] * combined["days_in_month"]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(factor_col)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

        # Blends
        blend_sources = {
            "blend_day_scaled_and_last_year_prediction": (
                "day_scaled_lag1_prediction",
                "daily_same_transition_last_year_prediction",
                0.50,
                0.50,
            ),
            "blend_day_scaled_and_global_prediction": (
                "day_scaled_lag1_prediction",
                "global_transition_prediction",
                0.50,
                0.50,
            ),
            "blend_day_scaled_and_recent_mean_prediction": (
                "day_scaled_lag1_prediction",
                "recent_daily_mean_3_prediction",
                0.60,
                0.40,
            ),
        }
        for out_col, (c1, c2, w1, w2) in blend_sources.items():
            if {c1, c2}.issubset(combined.columns):
                combined[out_col] = w1 * combined[c1] + w2 * combined[c2]
                combined[f"log_{out_col}"] = _safe_log_external(combined[out_col], eps=eps)
                _mark(out_col)
                _mark(f"log_{out_col}")

        ratio_pairs = {
            "day_scaled_to_lag1_ratio": ("day_scaled_lag1_prediction", f"{target_col}_lag_1"),
            "last_year_anchor_to_day_scaled_ratio": ("daily_same_transition_last_year_prediction", "day_scaled_lag1_prediction"),
            "global_anchor_to_day_scaled_ratio": ("global_transition_prediction", "day_scaled_lag1_prediction"),
            "recent_mean_anchor_to_day_scaled_ratio": ("recent_daily_mean_3_prediction", "day_scaled_lag1_prediction"),
        }
        for out_col, (a, b) in ratio_pairs.items():
            if {a, b}.issubset(combined.columns):
                combined[out_col] = _safe_divide_external(combined[a], combined[b], fill=np.nan)
                _mark(out_col)

        log_diff_pairs = [(1, 2), (1, 3), (1, 6), (1, 12), (2, 3), (3, 6), (12, 13)]
        for left, right in log_diff_pairs:
            a, b = f"{target_col}_lag_{left}", f"{target_col}_lag_{right}"
            if {a, b}.issubset(combined.columns):
                out_col = f"log_{target_col}_diff_lag_{left}_{right}"
                combined[out_col] = _safe_log_external(combined[a], eps=eps) - _safe_log_external(combined[b], eps=eps)
                _mark(out_col)

            da, db = f"daily_{target_col}_lag_{left}", f"daily_{target_col}_lag_{right}"
            if {da, db}.issubset(combined.columns):
                out_col = f"log_daily_{target_col}_diff_lag_{left}_{right}"
                combined[out_col] = _safe_log_external(combined[da], eps=eps) - _safe_log_external(combined[db], eps=eps)
                _mark(out_col)

    # ------------------------------------------------------------
    # 3. March 2024 growth coefficients
    # ------------------------------------------------------------
    if add_march_2024_features:
        march_2024_idx = (2024 - int(min_year)) * 12 + (3 - 1)
        after_march_2024 = combined[time_col].astype(int) > march_2024_idx

        hist = history_raw.copy()
        hist["calendar_month_num"] = _calendar_month_from_time(hist[time_col], min_year)
        hist["calendar_year"] = _year_from_time(hist[time_col], min_year)

        hist_2024 = hist[
            (hist["calendar_year"].astype(int) == 2024)
            & (hist["calendar_month_num"].isin([2, 3]))
            & hist[target_col].notna()
        ].copy()

        if not hist_2024.empty:
            store_pivot = (
                hist_2024.pivot_table(
                    index=id_col,
                    columns="calendar_month_num",
                    values=target_col,
                    aggfunc="first",
                )
                .rename(columns={2: "pto_feb_2024", 3: "pto_mar_2024"})
                .reset_index()
            )
            if {"pto_feb_2024", "pto_mar_2024"}.issubset(store_pivot.columns):
                store_pivot["store_feb_mar_growth_2024"] = _safe_divide_external(
                    store_pivot["pto_mar_2024"], store_pivot["pto_feb_2024"], fill=np.nan
                ).clip(clip_growth_low, clip_growth_high)
                combined = combined.drop(columns=["store_feb_mar_growth_2024"], errors="ignore")
                combined = combined.merge(
                    store_pivot[[id_col, "store_feb_mar_growth_2024"]],
                    on=id_col,
                    how="left",
                )
                combined.loc[~after_march_2024, "store_feb_mar_growth_2024"] = np.nan
                _mark("store_feb_mar_growth_2024")

            def _add_group_growth(group_col, out_col):
                nonlocal combined
                if group_col not in hist_2024.columns or group_col not in combined.columns:
                    return
                group_agg = (
                    hist_2024.groupby([group_col, "calendar_month_num"], dropna=False)[target_col]
                    .sum()
                    .reset_index()
                )
                pivot = (
                    group_agg.pivot_table(
                        index=group_col,
                        columns="calendar_month_num",
                        values=target_col,
                        aggfunc="first",
                    )
                    .rename(columns={2: "pto_feb_2024", 3: "pto_mar_2024"})
                    .reset_index()
                )
                if {"pto_feb_2024", "pto_mar_2024"}.issubset(pivot.columns):
                    pivot[out_col] = _safe_divide_external(
                        pivot["pto_mar_2024"], pivot["pto_feb_2024"], fill=np.nan
                    ).clip(clip_growth_low, clip_growth_high)
                    combined = combined.drop(columns=[out_col], errors="ignore")
                    combined = combined.merge(pivot[[group_col, out_col]], on=group_col, how="left")
                    combined.loc[~after_march_2024, out_col] = np.nan
                    _mark(out_col)

            _add_group_growth("region", "region_feb_mar_growth_2024")
            _add_group_growth("settlement", "settlement_feb_mar_growth_2024")
            _add_group_growth("trading_area_category", "trading_area_feb_mar_growth_2024")

            feb_sum = hist_2024.loc[hist_2024["calendar_month_num"] == 2, target_col].sum()
            mar_sum = hist_2024.loc[hist_2024["calendar_month_num"] == 3, target_col].sum()
            global_growth = np.nan
            if pd.notna(feb_sum) and feb_sum > 0:
                global_growth = np.clip(mar_sum / feb_sum, clip_growth_low, clip_growth_high)

            combined["global_feb_mar_growth_2024"] = np.where(after_march_2024, global_growth, np.nan)
            _mark("global_feb_mar_growth_2024")

            growth_cols = [
                "store_feb_mar_growth_2024",
                "region_feb_mar_growth_2024",
                "settlement_feb_mar_growth_2024",
                "trading_area_category_feb_mar_growth_2024",  # backward-safe typo guard
                "trading_area_feb_mar_growth_2024",
                "global_feb_mar_growth_2024",
            ]
            # unify expected column name from the prompt
            if "trading_area_category_feb_mar_growth_2024" in combined.columns and "trading_area_feb_mar_growth_2024" not in combined.columns:
                combined["trading_area_feb_mar_growth_2024"] = combined["trading_area_category_feb_mar_growth_2024"]
                _mark("trading_area_feb_mar_growth_2024")

            for col in [
                "store_feb_mar_growth_2024",
                "region_feb_mar_growth_2024",
                "settlement_feb_mar_growth_2024",
                "trading_area_feb_mar_growth_2024",
                "global_feb_mar_growth_2024",
            ]:
                if col not in combined.columns or f"{target_col}_lag_1" not in combined.columns:
                    continue
                combined[col] = combined[col].fillna(combined["global_feb_mar_growth_2024"])
                pred_col = f"lag1_x_{col}"
                combined[pred_col] = combined[f"{target_col}_lag_1"] * combined[col]
                combined[f"log_{pred_col}"] = _safe_log_external(combined[pred_col], eps=eps)
                _mark(pred_col)
                _mark(f"log_{pred_col}")

            if {"day_scaled_lag1_prediction", "lag1_x_store_feb_mar_growth_2024"}.issubset(combined.columns):
                combined["blend_day_scaled_and_store_march_growth"] = (
                    0.50 * combined["day_scaled_lag1_prediction"]
                    + 0.50 * combined["lag1_x_store_feb_mar_growth_2024"]
                )
                combined["log_blend_day_scaled_and_store_march_growth"] = _safe_log_external(
                    combined["blend_day_scaled_and_store_march_growth"], eps=eps
                )
                _mark("blend_day_scaled_and_store_march_growth")
                _mark("log_blend_day_scaled_and_store_march_growth")

            if {"day_scaled_lag1_prediction", "lag1_x_region_feb_mar_growth_2024"}.issubset(combined.columns):
                combined["blend_day_scaled_and_region_march_growth"] = (
                    0.50 * combined["day_scaled_lag1_prediction"]
                    + 0.50 * combined["lag1_x_region_feb_mar_growth_2024"]
                )
                combined["log_blend_day_scaled_and_region_march_growth"] = _safe_log_external(
                    combined["blend_day_scaled_and_region_march_growth"], eps=eps
                )
                _mark("blend_day_scaled_and_region_march_growth")
                _mark("log_blend_day_scaled_and_region_march_growth")

    # ------------------------------------------------------------
    # 4. Hierarchical share features
    # ------------------------------------------------------------
    if add_hierarchy_share_features:
        pto_lag_1 = f"{target_col}_lag_1"
        if pto_lag_1 in combined.columns:
            if "region" in combined.columns:
                combined["region_pto_lag1_mean"] = (
                    combined.groupby([time_col, "region"], dropna=False)[pto_lag_1].transform("mean")
                )
                combined["store_share_region_lag1"] = _safe_divide_external(
                    combined[pto_lag_1], combined["region_pto_lag1_mean"], fill=np.nan
                )
                _mark("region_pto_lag1_mean")
                _mark("store_share_region_lag1")

            if "settlement" in combined.columns:
                combined["settlement_pto_lag1_mean"] = (
                    combined.groupby([time_col, "settlement"], dropna=False)[pto_lag_1].transform("mean")
                )
                combined["store_share_settlement_lag1"] = _safe_divide_external(
                    combined[pto_lag_1], combined["settlement_pto_lag1_mean"], fill=np.nan
                )
                _mark("settlement_pto_lag1_mean")
                _mark("store_share_settlement_lag1")

            if "trading_area_category" in combined.columns:
                combined["area_category_pto_lag1_mean"] = (
                    combined.groupby([time_col, "trading_area_category"], dropna=False)[pto_lag_1].transform("mean")
                )
                combined["store_share_area_category_lag1"] = _safe_divide_external(
                    combined[pto_lag_1], combined["area_category_pto_lag1_mean"], fill=np.nan
                )
                _mark("area_category_pto_lag1_mean")
                _mark("store_share_area_category_lag1")

            if "region" in combined.columns:
                for lag in [2, 3]:
                    lag_col = f"{target_col}_lag_{lag}"
                    if lag_col not in combined.columns:
                        continue
                    mean_col = f"region_pto_lag{lag}_mean"
                    share_col = f"store_share_region_lag{lag}"
                    combined[mean_col] = (
                        combined.groupby([time_col, "region"], dropna=False)[lag_col].transform("mean")
                    )
                    combined[share_col] = _safe_divide_external(combined[lag_col], combined[mean_col], fill=np.nan)
                    _mark(mean_col)
                    _mark(share_col)

                share_cols = [c for c in ["store_share_region_lag1", "store_share_region_lag2", "store_share_region_lag3"] if c in combined.columns]
                if share_cols:
                    combined["store_share_region_mean_3"] = combined[share_cols].mean(axis=1)
                    _mark("store_share_region_mean_3")
                if {"store_share_region_lag1", "store_share_region_lag2"}.issubset(combined.columns):
                    combined["store_share_region_change_1_2"] = _safe_divide_external(
                        combined["store_share_region_lag1"], combined["store_share_region_lag2"], fill=np.nan
                    )
                    combined["store_share_region_change_1_2_clipped"] = (
                        combined["store_share_region_change_1_2"].clip(0.70, 1.35)
                    )
                    _mark("store_share_region_change_1_2")
                    _mark("store_share_region_change_1_2_clipped")

            if "settlement" in combined.columns:
                for lag in [2, 3]:
                    lag_col = f"{target_col}_lag_{lag}"
                    if lag_col not in combined.columns:
                        continue
                    mean_col = f"settlement_pto_lag{lag}_mean"
                    share_col = f"store_share_settlement_lag{lag}"
                    combined[mean_col] = (
                        combined.groupby([time_col, "settlement"], dropna=False)[lag_col].transform("mean")
                    )
                    combined[share_col] = _safe_divide_external(combined[lag_col], combined[mean_col], fill=np.nan)
                    _mark(mean_col)
                    _mark(share_col)

                share_cols = [
                    c for c in ["store_share_settlement_lag1", "store_share_settlement_lag2", "store_share_settlement_lag3"]
                    if c in combined.columns
                ]
                if share_cols:
                    combined["store_share_settlement_mean_3"] = combined[share_cols].mean(axis=1)
                    _mark("store_share_settlement_mean_3")

            if {"store_share_region_mean_3", "lag1_x_region_feb_mar_growth_2024"}.issubset(combined.columns):
                combined["region_anchor_x_store_share_mean_3"] = (
                    combined["lag1_x_region_feb_mar_growth_2024"]
                    * combined["store_share_region_mean_3"].clip(0.50, 1.80)
                )
                combined["log_region_anchor_x_store_share_mean_3"] = _safe_log_external(
                    combined["region_anchor_x_store_share_mean_3"], eps=eps
                )
                _mark("region_anchor_x_store_share_mean_3")
                _mark("log_region_anchor_x_store_share_mean_3")

            if {"store_share_region_change_1_2_clipped", "day_scaled_lag1_prediction"}.issubset(combined.columns):
                combined["day_scaled_x_region_share_change"] = (
                    combined["day_scaled_lag1_prediction"]
                    * combined["store_share_region_change_1_2_clipped"]
                )
                combined["log_day_scaled_x_region_share_change"] = _safe_log_external(
                    combined["day_scaled_x_region_share_change"], eps=eps
                )
                _mark("day_scaled_x_region_share_change")
                _mark("log_day_scaled_x_region_share_change")

    # ------------------------------------------------------------
    # 5. Operational lag features
    # ------------------------------------------------------------
    if add_operational_lag_features:
        # lag_1, lag_3, lag_12 из исходных operational cols
        for col in lagged_monthly_cols:
            if col not in history_raw.columns:
                continue
            for lag in [1, 3, 12]:
                _merge_history_lag(history_raw, col, lag, f"{col}_lag_{lag}", [id_col])

            # rolling mean/median по прошлым 3 наблюдениям.
            raw_combined = pd.concat(
                [
                    history_raw.assign(__part_roll__="history"),
                    future_raw.assign(__part_roll__="future"),
                ],
                axis=0,
                ignore_index=True,
                sort=False,
            )
            raw_combined = _ensure_time_year_month_external(
                raw_combined,
                year_col=year_col,
                month_col=month_col,
                time_col=time_col,
                min_year=min_year,
                name=f"raw_combined_for_{col}",
            )
            raw_combined = raw_combined.sort_values([id_col, time_col], kind="mergesort")
            raw_combined["__x__"] = np.where(raw_combined["__part_roll__"].eq("history"), raw_combined[col], np.nan)
            shifted = raw_combined.groupby(id_col, sort=False)["__x__"].shift(1)
            raw_combined[f"{col}_roll_mean_3"] = (
                shifted.groupby(raw_combined[id_col], sort=False)
                .rolling(3, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )
            raw_combined[f"{col}_roll_median_3"] = (
                shifted.groupby(raw_combined[id_col], sort=False)
                .rolling(3, min_periods=1)
                .median()
                .reset_index(level=0, drop=True)
            )
            roll_map = raw_combined[[id_col, time_col, f"{col}_roll_mean_3", f"{col}_roll_median_3"]].drop_duplicates(
                [id_col, time_col],
                keep="last",
            )
            combined = combined.drop(columns=[f"{col}_roll_mean_3", f"{col}_roll_median_3"], errors="ignore")
            combined = combined.merge(roll_map, on=[id_col, time_col], how="left")
            _mark(f"{col}_roll_mean_3")
            _mark(f"{col}_roll_median_3")

        if {f"{target_col}_lag_1", "working_hours_per_day_lag_1"}.issubset(combined.columns):
            combined["pto_per_working_hour_lag1"] = _safe_divide_external(
                combined[f"{target_col}_lag_1"], combined["working_hours_per_day_lag_1"], fill=np.nan
            )
            combined["log_pto_per_working_hour_lag1"] = _safe_log_external(
                combined["pto_per_working_hour_lag1"], eps=eps
            )
            _mark("pto_per_working_hour_lag1")
            _mark("log_pto_per_working_hour_lag1")

        if {f"{target_col}_lag_1", "number_of_cash_registers"}.issubset(combined.columns):
            combined["pto_per_cash_register_lag1"] = _safe_divide_external(
                combined[f"{target_col}_lag_1"],
                combined["number_of_cash_registers"].astype(float),
                fill=np.nan,
            )
            combined["log_pto_per_cash_register_lag1"] = _safe_log_external(
                combined["pto_per_cash_register_lag1"], eps=eps
            )
            _mark("pto_per_cash_register_lag1")
            _mark("log_pto_per_cash_register_lag1")

        if {f"{target_col}_lag_1", "working_hours_per_day_lag_1", "number_of_cash_registers"}.issubset(combined.columns):
            denom = combined["working_hours_per_day_lag_1"] * combined["number_of_cash_registers"].astype(float)
            combined["pto_per_cash_register_hour_lag1"] = _safe_divide_external(
                combined[f"{target_col}_lag_1"], denom, fill=np.nan
            )
            combined["log_pto_per_cash_register_hour_lag1"] = _safe_log_external(
                combined["pto_per_cash_register_hour_lag1"], eps=eps
            )
            _mark("pto_per_cash_register_hour_lag1")
            _mark("log_pto_per_cash_register_hour_lag1")

    # ------------------------------------------------------------
    # Final cleanup and restore parts
    # ------------------------------------------------------------
    created = sorted(set(c for c in created if c in combined.columns))
    combined[created] = combined[created].replace([np.inf, -np.inf], np.nan)

    if fill_history_na and created:
        combined[created] = combined[created].fillna(fill_value)

    combined = combined.drop(columns=["calendar_year"], errors="ignore")

    history_out = (
        combined[combined["__part__"] == "history"]
        .sort_values("__row_order__")
        .drop(columns=["__part__", "__row_order__"], errors="ignore")
        .reset_index(drop=True)
    )
    future_out = (
        combined[combined["__part__"] == "future"]
        .sort_values("__row_order__")
        .drop(columns=["__part__", "__row_order__"], errors="ignore")
        .reset_index(drop=True)
    )

    # Align columns
    ordered_cols = list(history_out.columns)
    for c in future_out.columns:
        if c not in ordered_cols:
            ordered_cols.append(c)
    for c in ordered_cols:
        if c not in history_out.columns:
            history_out[c] = np.nan
        if c not in future_out.columns:
            future_out[c] = np.nan
    history_out = history_out[ordered_cols]
    future_out = future_out[ordered_cols]

    if downcast_float_features:
        for df_ in (history_out, future_out):
            float_cols = [c for c in df_.select_dtypes(include=["float64"]).columns if c != target_col]
            if float_cols:
                df_[float_cols] = df_[float_cols].astype("float32")

    return history_out, future_out, created


def build_retail_features_with_split(
    df: pd.DataFrame,
    *,
    # split config
    split_mode: Literal["train_val", "train_val_test", "train_test"] = "train_val",
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    target_col: str = "pto",
    min_year: int = 2023,
    val_time_idx: Optional[int] = None,
    val_year: Optional[int] = None,
    val_month: Optional[int] = None,
    test_time_idx: Optional[int] = None,
    test_year: Optional[int] = None,
    test_month: Optional[int] = None,
    create_test_if_missing: bool = True,
    expected_test_size: Optional[int] = 18657,
    future_unknown_cols: Optional[Sequence[str]] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    # feature config
    use_val_as_test_history: bool = True,
    add_integrated_anchor_features: bool = True,
    add_calendar_anchor_features: bool = True,
    add_expanding_calendar_heuristic_features: bool = True,
    add_daily_anchor_features: bool = True,
    add_transition_anchor_features: bool = True,
    add_march_2024_features: bool = True,
    add_hierarchy_share_features: bool = True,
    add_operational_lag_features: bool = True,
    group_cols_for_transition: list | None = None,
    lagged_monthly_cols: tuple[str, ...] = (
        "avg_promo_items_per_receipt",
        "avg_items_per_receipt",
        "avg_cancellations",
        "working_hours_per_day",
    ),
    clip_growth_low: float = 0.70,
    clip_growth_high: float = 1.45,
    eps: float = 1e-9,
    # pass-through to base generator
    base_feature_kwargs: dict | None = None,
    verbose: bool = True,
) -> dict:
    """
    Единая функция:
    1) делает leakage-safe split;
    2) создаёт artificial test month, если нужно;
    3) запускает базовый generate_retail_forecast_features_leakfree;
    4) добавляет все новые идеи:
       - month_day_ratio_to_previous_month;
       - day_scaled_lag1_prediction;
       - expanding calendar heuristic factors/predictions;
       - daily_pto_lag_*;
       - recent daily baselines;
       - trend-corrected anchors;
       - same-transition-last-year anchors;
       - global/group transition anchors;
       - March 2024 growth coefficients;
       - hierarchical share features;
       - lagged operational features.
    """
    base_feature_kwargs = {} if base_feature_kwargs is None else dict(base_feature_kwargs)
    # Defaults chosen to cover the full feature-idea corpus. User can override them in base_feature_kwargs.
    base_feature_kwargs.setdefault("target_rolling_windows", (3, 6, 12))
    base_feature_kwargs.setdefault("monthly_cross_lags", (1, 3, 12))
    base_feature_kwargs.setdefault(
        "monthly_cross_feature_types",
        ("promo_share", "promo_x_items", "items_x_hours", "cancellations_per_item", "cancellations_per_hour"),
    )

    split_result = make_panel_time_split(
        df,
        group_col=id_col,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        target_col=target_col,
        split_mode=split_mode,
        val_time_idx=val_time_idx,
        val_year=val_year,
        val_month=val_month,
        test_time_idx=test_time_idx,
        test_year=test_year,
        test_month=test_month,
        min_year=min_year,
        create_test_if_missing=create_test_if_missing,
        future_unknown_cols=future_unknown_cols,
        expected_test_size=expected_test_size if split_mode in {"train_val_test", "train_test"} else None,
        verbose=verbose,
    )

    if split_mode == "train_val":
        train_df, val_df = split_result
        test_df = None
    elif split_mode == "train_val_test":
        train_df, val_df, test_df = split_result
    else:
        train_df, test_df = split_result
        val_df = None

    output = generate_retail_forecast_features_leakfree(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        id_col=id_col,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        target_col=target_col,
        min_year=min_year,
        use_val_as_test_history=use_val_as_test_history,
        verbose=verbose,
        **base_feature_kwargs,
    )

    output["raw_splits"] = {
        "train": train_df,
        "val": val_df,
        "test": test_df,
    }

    if add_integrated_anchor_features:
        # validation pair: history = train, future = val
        if output.get("train_features") is not None and output.get("val_features") is not None:
            train_feat, val_feat, created = _add_integrated_anchor_features_pair(
                output["train_features"],
                output["val_features"],
                train_df,
                val_df,
                id_col=id_col,
                year_col=year_col,
                month_col=month_col,
                time_col=time_col,
                target_col=target_col,
                min_year=min_year,
                add_calendar_anchor_features=add_calendar_anchor_features,
                add_expanding_calendar_heuristic_features=add_expanding_calendar_heuristic_features,
                add_daily_anchor_features=add_daily_anchor_features,
                add_transition_anchor_features=add_transition_anchor_features,
                add_march_2024_features=add_march_2024_features,
                add_hierarchy_share_features=add_hierarchy_share_features,
                add_operational_lag_features=add_operational_lag_features,
                group_cols_for_transition=group_cols_for_transition,
                lagged_monthly_cols=lagged_monthly_cols,
                clip_growth_low=clip_growth_low,
                clip_growth_high=clip_growth_high,
                eps=eps,
                fill_history_na=base_feature_kwargs.get("fill_history_na", False),
                fill_value=base_feature_kwargs.get("fill_value", 0.0),
                downcast_float_features=base_feature_kwargs.get("downcast_float_features", True),
            )
            if base_feature_kwargs.get("drop_year_col", True):
                train_feat = train_feat.drop(columns=[year_col], errors="ignore")
                val_feat = val_feat.drop(columns=[year_col], errors="ignore")
            output["train_features"] = train_feat
            output["val_features"] = val_feat
            output.setdefault("feature_info", {}).setdefault("integrated_anchors", {})["val_created"] = created
            if verbose:
                print("Added integrated anchor features to train/val:", len(created))

        # test pair: history = train + val if requested and val exists, otherwise train
        if output.get("test_train_features") is not None and output.get("test_features") is not None:
            if val_df is not None and use_val_as_test_history:
                test_history_raw = pd.concat([train_df, val_df], axis=0, ignore_index=True, sort=False)
            else:
                test_history_raw = train_df.copy()

            test_train_feat, test_feat, created = _add_integrated_anchor_features_pair(
                output["test_train_features"],
                output["test_features"],
                test_history_raw,
                test_df,
                id_col=id_col,
                year_col=year_col,
                month_col=month_col,
                time_col=time_col,
                target_col=target_col,
                min_year=min_year,
                add_calendar_anchor_features=add_calendar_anchor_features,
                add_expanding_calendar_heuristic_features=add_expanding_calendar_heuristic_features,
                add_daily_anchor_features=add_daily_anchor_features,
                add_transition_anchor_features=add_transition_anchor_features,
                add_march_2024_features=add_march_2024_features,
                add_hierarchy_share_features=add_hierarchy_share_features,
                add_operational_lag_features=add_operational_lag_features,
                group_cols_for_transition=group_cols_for_transition,
                lagged_monthly_cols=lagged_monthly_cols,
                clip_growth_low=clip_growth_low,
                clip_growth_high=clip_growth_high,
                eps=eps,
                fill_history_na=base_feature_kwargs.get("fill_history_na", False),
                fill_value=base_feature_kwargs.get("fill_value", 0.0),
                downcast_float_features=base_feature_kwargs.get("downcast_float_features", True),
            )
            if base_feature_kwargs.get("drop_year_col", True):
                test_train_feat = test_train_feat.drop(columns=[year_col], errors="ignore")
                test_feat = test_feat.drop(columns=[year_col], errors="ignore")
            output["test_train_features"] = test_train_feat
            output["test_features"] = test_feat
            output.setdefault("feature_info", {}).setdefault("integrated_anchors", {})["test_created"] = created
            if verbose:
                print("Added integrated anchor features to train/test:", len(created))

    # Recompute model columns after enhancement.
    ref = output["test_train_features"] if output.get("test_train_features") is not None else output.get("train_features")
    if ref is not None:
        exclude = {target_col}
        output["feature_cols"] = [c for c in ref.columns if c not in exclude]
        output["categorical_cols"] = [
            c for c in output["feature_cols"]
            if c in ref.columns and (
                pd.api.types.is_object_dtype(ref[c]) or pd.api.types.is_categorical_dtype(ref[c])
            )
        ]

    gc.collect()
    return output

import numpy as np
import pandas as pd
from typing import Optional, Sequence, Literal


def prepare_lgbm_matrices_with_time_aware_catboost_encoding(
    feature_output: dict,
    *,
    features_to_use: Optional[Sequence[str]] = None,
    target_mode: Literal[
        "original",
        "growth_log",
        "day_scaled_residual_log",
        "calendar_residual_log",
        "daily_rate_log",
        "direct_log1p",
    ] = "direct_log1p",
    target_col: str = "pto",
    id_col: str = "new_id",
    time_col: str = "year_month_index",
    cat_features: Optional[Sequence[str]] = None,
    add_default_cat_features: bool = True,
    add_numeric_cat_copies: bool = True,
    numeric_cat_source_cols: Sequence[str] = (
        "new_id",
        "alcohol_license_flag",
        "year",
        "month",
        "prev_month",
        "calendar_month_num",
        "marketplaces_deliveries_parcel_lockers_100m",
        "medical_institutions_and_pharmacies_300m",
        "schools_300m",
        "public_transport_stops_300m",
        "grocery_stores_500m",
        "pyaterochka_stores_500m",
        "number_of_cash_registers",
        "working_hours_per_day_lag_1",
        "working_hours_per_day_lag_3",
        "working_hours_per_day_lag_12",
    ),
    cb_suffix: str = "_cb",
    smoothing: float = 20.0,
    min_count: int = 1,
    drop_original_cat_features: bool = True,
    drop_numeric_cat_source_cols: bool = False,
    include_all_detected_cat_features: bool = True,
    keep_time_col: bool = True,
    keep_id_col: bool = True,
    strict_train_time_order: bool = True,
    first_block_value: Literal["nan", "global"] = "nan",
    unknown_value: Literal["global", "nan"] = "global",
    verbose: bool = True,
) -> dict:
    """
    Готовит матрицы для LightGBM/числовой модели с time-aware ordered target encoding
    для всех категориальных признаков.

    Основной контракт против leakage:
    - validation: encoder fit только на train_features; val только transform;
    - test: encoder fit только на test_train_features; test только transform;
    - train-строка месяца t кодируется только статистиками месяцев < t;
    - same-month target никогда не используется для кодировки train-строк этого же месяца;
    - fallback prior внутри train также time-aware: для месяца t берётся среднее только по месяцам < t;
    - для первого train-блока fallback по умолчанию NaN, а не mean по всему train.

    Почему это не нативный CatBoost:
    - это ручной CatBoost-like / ordered target encoding;
    - на выходе не остаётся object/category/string колонок;
    - исходные категориальные признаки по умолчанию удаляются, остаются *_cb.

    Важные параметры:
    - include_all_detected_cat_features=True добавляет в признаки все найденные
      категориальные колонки, даже если features_to_use их не содержал;
    - first_block_value="nan" максимально строго исключает temporal leakage;
      если нужна менее строгая, но более плотная матрица, можно поставить "global";
    - unknown_value="global" для val/test безопасен, потому что global считается
      только по fit-history.
    """

    # ------------------------------------------------------------
    # Basic checks
    # ------------------------------------------------------------
    if smoothing < 0:
        raise ValueError("smoothing должен быть >= 0.")
    if min_count < 1:
        raise ValueError("min_count должен быть >= 1.")
    if first_block_value not in {"nan", "global"}:
        raise ValueError("first_block_value должен быть 'nan' или 'global'.")
    if unknown_value not in {"global", "nan"}:
        raise ValueError("unknown_value должен быть 'global' или 'nan'.")
    if not strict_train_time_order:
        raise ValueError(
            "strict_train_time_order=False отключал бы защиту от temporal leakage. "
            "Для этой задачи оставляем только strict_train_time_order=True."
        )

    def _check_required_functions():
        if "make_forecast_target" not in globals():
            raise NameError(
                "Не найдена функция make_forecast_target. "
                "Она должна быть определена до prepare_lgbm_matrices_with_time_aware_catboost_encoding."
            )

    def _copy_df(df):
        return df.copy() if df is not None else None

    def _is_non_numeric_dtype(s: pd.Series) -> bool:
        return (
            pd.api.types.is_object_dtype(s)
            or pd.api.types.is_categorical_dtype(s)
            or pd.api.types.is_string_dtype(s)
        )

    def _as_cat_string(s: pd.Series) -> pd.Series:
        # Единая нормализация категорий для train/future.
        # Все пропуски и строковые варианты NA схлопываются в один токен.
        out = s.astype("string")
        out = out.fillna("__missing__")
        out = out.replace(
            {
                "<NA>": "__missing__",
                "nan": "__missing__",
                "NaN": "__missing__",
                "None": "__missing__",
                "none": "__missing__",
                "NULL": "__missing__",
                "null": "__missing__",
            }
        )
        return out

    def _safe_numeric_frame(df: pd.DataFrame, *, name: str) -> pd.DataFrame:
        bad_cols = [c for c in df.columns if _is_non_numeric_dtype(df[c])]
        if bad_cols:
            raise ValueError(
                f"[{name}] После ordered target encoding остались нечисловые колонки: "
                f"{bad_cols[:50]}. Обычно это значит, что колонка не попала в cat_features "
                "или drop_original_cat_features=False."
            )
        return df

    def _ordered_unique(items) -> list:
        seen = set()
        result = []
        for x in items:
            if x is None:
                continue
            if x not in seen:
                seen.add(x)
                result.append(x)
        return result

    # ------------------------------------------------------------
    # Categorical feature construction / discovery
    # ------------------------------------------------------------
    def _add_numeric_copies_to_parts(parts: dict[str, Optional[pd.DataFrame]]):
        """
        Создаёт *_cat копии для числовых/дискретных признаков.
        Если source есть в train, но отсутствует в future, future получает __missing__.
        Это важно, чтобы transform не падал на разных наборах колонок.
        """
        created = []
        if not add_numeric_cat_copies:
            return {k: _copy_df(v) for k, v in parts.items()}, created

        out_parts = {k: _copy_df(v) for k, v in parts.items()}

        for src in numeric_cat_source_cols:
            dst = f"{src}_cat"
            exists_somewhere = any(
                df_part is not None and (src in df_part.columns or dst in df_part.columns)
                for df_part in out_parts.values()
            )
            if not exists_somewhere:
                continue

            for _, df_part in out_parts.items():
                if df_part is None:
                    continue
                if dst in df_part.columns:
                    df_part[dst] = _as_cat_string(df_part[dst])
                elif src in df_part.columns:
                    df_part[dst] = _as_cat_string(df_part[src])
                else:
                    df_part[dst] = "__missing__"

            created.append(dst)

        return out_parts, sorted(set(created))

    def _default_cat_candidates() -> list[str]:
        return [
            "opening_date_category",
            "trading_area_category",
            "settlement",
            "region",
            "month_name",
            "holiday_month_type",
            "population_city_type",
            "season",
            "lag1_rto_size_bin",
            "new_id_cat",
            "alcohol_license_flag_cat",
            "year_cat",
            "month_cat",
            "prev_month_cat",
            "calendar_month_num_cat",
            "marketplaces_deliveries_parcel_lockers_100m_cat",
            "medical_institutions_and_pharmacies_300m_cat",
            "schools_300m_cat",
            "public_transport_stops_300m_cat",
            "grocery_stores_500m_cat",
            "pyaterochka_stores_500m_cat",
            "number_of_cash_registers_cat",
            "working_hours_per_day_lag_1_cat",
            "working_hours_per_day_lag_3_cat",
            "working_hours_per_day_lag_12_cat",
        ]

    def _collect_all_cat_candidates(ref_df: pd.DataFrame, numeric_cat_copies: list[str]) -> list[str]:
        candidates = []
        if cat_features is not None:
            candidates.extend(list(cat_features))
        candidates.extend(feature_output.get("categorical_cols") or [])
        candidates.extend([c for c in ref_df.columns if _is_non_numeric_dtype(ref_df[c])])
        candidates.extend(numeric_cat_copies)
        if add_default_cat_features:
            candidates.extend(_default_cat_candidates())
        return [c for c in _ordered_unique(candidates) if c in ref_df.columns and c != target_col]

    def _build_base_feature_list(train_df: pd.DataFrame, numeric_cat_copies: list[str]) -> list[str]:
        if features_to_use is None:
            base_features = list(feature_output.get("feature_cols") or [])
            if not base_features:
                base_features = [c for c in train_df.columns if c != target_col]
        else:
            base_features = list(features_to_use)

        base_features = [c for c in base_features if c != target_col]
        base_features = [c for c in base_features if c in train_df.columns]

        # Newly created *_cat должны попасть в список признаков, иначе encoding не будет создан.
        for c in numeric_cat_copies:
            if c in train_df.columns and c not in base_features:
                base_features.append(c)

        # По требованию: все обнаруженные категориальные признаки пропускаем через ordered TE.
        # Если features_to_use был строгим selected-list, это поведение можно выключить
        # include_all_detected_cat_features=False.
        if include_all_detected_cat_features:
            for c in _collect_all_cat_candidates(train_df, numeric_cat_copies):
                if c in train_df.columns and c not in base_features:
                    base_features.append(c)

        return _ordered_unique(base_features)

    def _collect_cat_features(ref_df: pd.DataFrame, feature_cols: list[str], numeric_cat_copies: list[str]) -> list[str]:
        feature_set = set(feature_cols)
        candidates = _collect_all_cat_candidates(ref_df, numeric_cat_copies)
        result = []
        for c in candidates:
            if c in ref_df.columns and c in feature_set and c not in result:
                result.append(c)
        return result

    # ------------------------------------------------------------
    # Ordered target encoding core
    # ------------------------------------------------------------
    def _smoothed_value(sum_y: float, cnt: int, prior: float) -> float:
        if cnt < min_count:
            return prior
        return (float(sum_y) + float(smoothing) * float(prior)) / (int(cnt) + float(smoothing))

    def _make_future_encoding(
        future_col: pd.Series,
        stats: pd.DataFrame,
        *,
        global_prior: float,
    ) -> pd.Series:
        cats = _as_cat_string(future_col)

        if stats.empty:
            fill = global_prior if unknown_value == "global" else np.nan
            return pd.Series(fill, index=future_col.index, dtype="float32")

        sums = cats.map(stats["sum"]).astype(float)
        cnts = cats.map(stats["count"]).astype(float)

        if unknown_value == "global":
            prior = global_prior
        else:
            prior = np.nan

        enc = (sums + smoothing * global_prior) / (cnts + smoothing)
        enc = enc.astype(float)
        enc = enc.where(cnts.fillna(0) >= min_count, prior)
        enc = enc.fillna(prior)
        return enc.astype("float32")

    def _time_aware_cb_encode_pair(
        train_df: pd.DataFrame,
        future_df: Optional[pd.DataFrame],
        y_train: pd.Series,
        *,
        feature_cols: list[str],
        cat_cols: list[str],
        pair_name: str,
    ):
        """
        Block-wise ordered target encoding:
        - train month t uses only rows with time < t;
        - fallback prior for train month t is also computed only from time < t;
        - future uses all train rows, which are historical relative to that future block.
        """
        train_df = train_df.copy()
        future_df = future_df.copy() if future_df is not None else None

        if time_col not in train_df.columns:
            raise ValueError(f"[{pair_name}] В train_df нет time_col='{time_col}'.")
        if train_df[time_col].isna().any():
            raise ValueError(f"[{pair_name}] В train_df['{time_col}'] есть NaN.")

        y_train = pd.Series(y_train, index=train_df.index)
        y_train = pd.to_numeric(y_train, errors="coerce").astype(float)
        valid_y_mask = y_train.notna() & np.isfinite(y_train)
        if valid_y_mask.sum() == 0:
            raise ValueError(f"[{pair_name}] y_train полностью NaN/inf после make_forecast_target.")

        # Full-history mean безопасен только для future transform.
        full_history_global_mean = float(y_train.loc[valid_y_mask].mean())

        # Гарантируем наличие cat_cols в train/future. Если в future нет колонки – это unseen/missing категория.
        for col in cat_cols:
            if col not in train_df.columns:
                train_df[col] = "__missing__"
            train_df[col] = _as_cat_string(train_df[col])
            if future_df is not None:
                if col not in future_df.columns:
                    future_df[col] = "__missing__"
                future_df[col] = _as_cat_string(future_df[col])

        encoded_cols = []
        ordered_times = sorted(pd.to_numeric(train_df[time_col], errors="raise").astype(int).unique())

        # Предрасчёт индексов по времени. Так меньше риска случайно использовать same-month target.
        time_values = pd.to_numeric(train_df[time_col], errors="raise").astype(int)
        indices_by_time = {t: train_df.index[time_values.eq(t)] for t in ordered_times}

        for col in cat_cols:
            enc_col = f"{col}{cb_suffix}"
            encoded_cols.append(enc_col)
            train_df[enc_col] = np.nan

            sum_by_cat: dict[str, float] = {}
            cnt_by_cat: dict[str, int] = {}
            global_sum = 0.0
            global_cnt = 0

            for t in ordered_times:
                idx_t = indices_by_time[t]
                cats_t = train_df.loc[idx_t, col]

                if global_cnt > 0:
                    time_prior = global_sum / global_cnt
                elif first_block_value == "global":
                    # Не идеально строго, но оставлено как управляемый режим совместимости.
                    time_prior = full_history_global_mean
                else:
                    # Максимально строгий режим: нет прошлой истории -> NaN.
                    time_prior = np.nan

                vals = []
                for cat in cats_t:
                    n = cnt_by_cat.get(cat, 0)
                    if n < min_count or pd.isna(time_prior):
                        vals.append(time_prior)
                    else:
                        vals.append(_smoothed_value(sum_by_cat.get(cat, 0.0), n, time_prior))

                train_df.loc[idx_t, enc_col] = vals

                # Обновляем статистики только после кодирования всего time-блока.
                y_t = y_train.loc[idx_t]
                valid_t = y_t.notna() & np.isfinite(y_t)
                if valid_t.any():
                    tmp = pd.DataFrame(
                        {
                            "__cat__": cats_t.loc[valid_t].values,
                            "__y__": y_t.loc[valid_t].values,
                        }
                    )
                    agg = tmp.groupby("__cat__", dropna=False)["__y__"].agg(["sum", "count"])

                    for cat, row in agg.iterrows():
                        sum_by_cat[cat] = sum_by_cat.get(cat, 0.0) + float(row["sum"])
                        cnt_by_cat[cat] = cnt_by_cat.get(cat, 0) + int(row["count"])

                    global_sum += float(y_t.loc[valid_t].sum())
                    global_cnt += int(valid_t.sum())

            train_df[enc_col] = pd.to_numeric(train_df[enc_col], errors="coerce").astype("float32")

        # Full train history stats for future transform. This is safe because future is later than fit history.
        full_stats = {}
        for col in cat_cols:
            tmp = pd.DataFrame(
                {
                    "__cat__": train_df.loc[valid_y_mask, col].values,
                    "__y__": y_train.loc[valid_y_mask].values,
                }
            )
            full_stats[col] = tmp.groupby("__cat__", dropna=False)["__y__"].agg(["sum", "count"])

        if future_df is not None:
            for col in cat_cols:
                enc_col = f"{col}{cb_suffix}"
                future_df[enc_col] = _make_future_encoding(
                    future_df[col],
                    full_stats[col],
                    global_prior=full_history_global_mean,
                )

        final_features = list(feature_cols)

        # Добавляем созданные encoded-признаки.
        for enc_col in encoded_cols:
            if enc_col not in final_features:
                final_features.append(enc_col)

        if drop_original_cat_features:
            cat_set = set(cat_cols)
            final_features = [c for c in final_features if c not in cat_set]

        if drop_numeric_cat_source_cols and add_numeric_cat_copies:
            source_cols_to_drop = set()
            for src in numeric_cat_source_cols:
                if f"{src}_cat" in cat_cols:
                    source_cols_to_drop.add(src)
            final_features = [c for c in final_features if c not in source_cols_to_drop]

        if not keep_time_col:
            final_features = [c for c in final_features if c != time_col]
        if not keep_id_col:
            final_features = [c for c in final_features if c != id_col]

        final_features = _ordered_unique([c for c in final_features if c in train_df.columns])

        X_train = train_df[final_features].copy()
        X_future = None
        if future_df is not None:
            for c in final_features:
                if c not in future_df.columns:
                    future_df[c] = np.nan
            X_future = future_df[final_features].copy()

        X_train = _safe_numeric_frame(X_train, name=f"{pair_name}:X_train")
        if X_future is not None:
            X_future = _safe_numeric_frame(X_future, name=f"{pair_name}:X_future")

        num_cols = X_train.select_dtypes(include=[np.number]).columns
        X_train[num_cols] = X_train[num_cols].replace([np.inf, -np.inf], np.nan)
        if X_future is not None:
            num_cols_f = X_future.select_dtypes(include=[np.number]).columns
            X_future[num_cols_f] = X_future[num_cols_f].replace([np.inf, -np.inf], np.nan)

        # Служебная диагностика: доля NaN в train encoding показывает первый блок и новые категории.
        encoded_nan_share = {}
        for c in encoded_cols:
            if c in X_train.columns:
                encoded_nan_share[c] = float(X_train[c].isna().mean())

        info = {
            "pair_name": pair_name,
            "cat_features_used": cat_cols,
            "encoded_features_created": encoded_cols,
            "final_features": final_features,
            "global_mean_for_future_transform": full_history_global_mean,
            "smoothing": smoothing,
            "min_count": min_count,
            "first_block_value": first_block_value,
            "unknown_value": unknown_value,
            "strict_train_time_order": strict_train_time_order,
            "encoded_train_nan_share": encoded_nan_share,
            "n_train_rows": int(len(train_df)),
            "n_future_rows": int(0 if future_df is None else len(future_df)),
            "n_time_blocks_train": int(len(ordered_times)),
            "first_time_block": int(ordered_times[0]) if ordered_times else None,
            "last_time_block": int(ordered_times[-1]) if ordered_times else None,
        }

        return X_train, X_future, info

    # ------------------------------------------------------------
    # Pair preparation
    # ------------------------------------------------------------
    def _prepare_pair(train_key: str, future_key: Optional[str], pair_name: str):
        train_df = feature_output.get(train_key)
        future_df = feature_output.get(future_key) if future_key is not None else None

        if train_df is None:
            return None

        parts = {
            "train": train_df,
            "future": future_df,
        }
        parts, numeric_cat_copies = _add_numeric_copies_to_parts(parts)
        train_df = parts["train"]
        future_df = parts["future"]

        base_features = _build_base_feature_list(train_df, numeric_cat_copies)
        cat_cols = _collect_cat_features(train_df, base_features, numeric_cat_copies)

        # Если cat_features явно передали, но они не нашлись после feature construction – лучше явно сообщить.
        if cat_features is not None:
            missing_explicit = [c for c in cat_features if c not in train_df.columns]
            if missing_explicit and verbose:
                print(f"[{pair_name}] explicit cat_features отсутствуют в train_df и будут пропущены:", missing_explicit[:30])

        y_train = make_forecast_target(
            train_df,
            mode=target_mode,
            target_col=target_col,
        )

        y_future = None
        if future_df is not None and target_col in future_df.columns and future_df[target_col].notna().any():
            y_future = make_forecast_target(
                future_df,
                mode=target_mode,
                target_col=target_col,
            )

        X_train, X_future, info = _time_aware_cb_encode_pair(
            train_df=train_df,
            future_df=future_df,
            y_train=y_train,
            feature_cols=base_features,
            cat_cols=cat_cols,
            pair_name=pair_name,
        )

        return {
            "X_train": X_train,
            "y_train": y_train,
            "X_future": X_future,
            "y_future": y_future,
            "info": info,
        }

    # ------------------------------------------------------------
    # Main
    # ------------------------------------------------------------
    _check_required_functions()

    result = {
        "validation": None,
        "test": None,
        "catboost_encoding_info": {},
    }

    # validation: train -> val
    if feature_output.get("train_features") is not None and feature_output.get("val_features") is not None:
        val_pack = _prepare_pair(
            train_key="train_features",
            future_key="val_features",
            pair_name="validation_train_to_val",
        )
        result["validation"] = {
            "X_train": val_pack["X_train"],
            "y_train": val_pack["y_train"],
            "X_val": val_pack["X_future"],
            "y_val": val_pack["y_future"],
        }
        result["catboost_encoding_info"]["validation"] = val_pack["info"]

    # test/final: test_train -> test
    if feature_output.get("test_train_features") is not None and feature_output.get("test_features") is not None:
        test_pack = _prepare_pair(
            train_key="test_train_features",
            future_key="test_features",
            pair_name="final_train_to_test",
        )
        result["test"] = {
            "X_train": test_pack["X_train"],
            "y_train": test_pack["y_train"],
            "X_test": test_pack["X_future"],
        }
        result["catboost_encoding_info"]["test"] = test_pack["info"]

    if verbose:
        for block_name, info in result["catboost_encoding_info"].items():
            print(f"\n[catboost-encoding:{block_name}]")
            print("cat features used:", len(info["cat_features_used"]))
            print("first cat features:", info["cat_features_used"][:30])
            print("encoded features created:", len(info["encoded_features_created"]))
            print("first encoded features:", info["encoded_features_created"][:30])
            print("final features:", len(info["final_features"]))
            print("strict train time order:", info["strict_train_time_order"])
            print("first_block_value:", info["first_block_value"])
            print("unknown_value:", info["unknown_value"])

    return result

# ============================================================
# Target engineering utilities
# ============================================================

def make_forecast_target(
    df: pd.DataFrame,
    *,
    mode: Literal[
        "original",
        "growth_log",
        "day_scaled_residual_log",
        "calendar_residual_log",
        "daily_rate_log",
        "direct_log1p",
    ] = "growth_log",
    target_col: str = "pto",
    eps: float = 1e-9,
) -> pd.Series:
    """
    Builds model target for the main target formulations used in the experiments.

    Required feature columns by mode:
    - growth_log: pto_lag_1;
    - day_scaled_residual_log: day_scaled_lag1_prediction;
    - calendar_residual_log: calendar_heuristic_prediction;
    - daily_rate_log: n_days_in_month or days_in_month;
    - direct_log1p: only target_col.
    """
    if target_col not in df.columns:
        raise ValueError(f"В df нет target_col='{target_col}'.")

    y = pd.to_numeric(df[target_col], errors="coerce").astype(float)

    if mode == "original":
        return y

    if mode == "direct_log1p":
        return np.log1p(y.clip(lower=0))

    if mode == "growth_log":
        anchor_col = f"{target_col}_lag_1"
        if anchor_col not in df.columns:
            raise ValueError(f"Для mode='growth_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(df[anchor_col], errors="coerce").astype(float)
        return np.log(np.clip(y, eps, None)) - np.log(np.clip(anchor, eps, None))

    if mode == "day_scaled_residual_log":
        anchor_col = "day_scaled_lag1_prediction"
        if anchor_col not in df.columns:
            raise ValueError(f"Для mode='day_scaled_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(df[anchor_col], errors="coerce").astype(float)
        return np.log(np.clip(y, eps, None)) - np.log(np.clip(anchor, eps, None))

    if mode == "calendar_residual_log":
        anchor_col = "calendar_heuristic_prediction"
        if anchor_col not in df.columns:
            raise ValueError(f"Для mode='calendar_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(df[anchor_col], errors="coerce").astype(float)
        return np.log(np.clip(y, eps, None)) - np.log(np.clip(anchor, eps, None))

    if mode == "daily_rate_log":
        day_col = "n_days_in_month" if "n_days_in_month" in df.columns else "days_in_month"
        if day_col not in df.columns:
            raise ValueError("Для mode='daily_rate_log' нужна колонка 'n_days_in_month' или 'days_in_month'.")
        days = pd.to_numeric(df[day_col], errors="coerce").astype(float)
        daily_rate = y / days.replace(0, np.nan)
        return np.log(np.clip(daily_rate, eps, None))

    raise ValueError(f"Неизвестный mode='{mode}'.")


def inverse_forecast_target(
    raw_prediction,
    features: pd.DataFrame,
    *,
    mode: Literal[
        "original",
        "growth_log",
        "day_scaled_residual_log",
        "calendar_residual_log",
        "daily_rate_log",
        "direct_log1p",
    ] = "growth_log",
    target_col: str = "pto",
    residual_shrinkage: float = 1.0,
    clip_min: float = 0.0,
) -> np.ndarray:
    """
    Converts model output back to RTO forecast.

    residual_shrinkage is useful for residual modes, e.g. 0.35 or 0.50.
    """
    pred = np.asarray(raw_prediction, dtype=float)

    if mode == "original":
        out = pred
    elif mode == "direct_log1p":
        out = np.expm1(pred)
    elif mode == "growth_log":
        anchor_col = f"{target_col}_lag_1"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='growth_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(pred)
    elif mode == "day_scaled_residual_log":
        anchor_col = "day_scaled_lag1_prediction"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='day_scaled_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(residual_shrinkage * pred)
    elif mode == "calendar_residual_log":
        anchor_col = "calendar_heuristic_prediction"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='calendar_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(residual_shrinkage * pred)
    elif mode == "daily_rate_log":
        day_col = "n_days_in_month" if "n_days_in_month" in features.columns else "days_in_month"
        if day_col not in features.columns:
            raise ValueError("Для mode='daily_rate_log' нужна колонка 'n_days_in_month' или 'days_in_month'.")
        days = pd.to_numeric(features[day_col], errors="coerce").astype(float).to_numpy()
        out = days * np.exp(pred)
    else:
        raise ValueError(f"Неизвестный mode='{mode}'.")

    out = np.asarray(out, dtype=float)
    out = np.where(np.isfinite(out), out, np.nan)
    if clip_min is not None:
        out = np.maximum(out, clip_min)
    return out

# Функция для отбора признаков

In [9]:
# -*- coding: utf-8 -*-
"""
LightGBM ranked-add feature selection for the X5/RTO monthly panel pipeline.

This module is synchronized with the current leak-free feature pipeline:
- features are built fold-by-fold through build_retail_features_with_split(...);
- all categorical features are converted by
  prepare_lgbm_matrices_with_time_aware_catboost_encoding(...);
- inside each temporal fold, ordered target encoding is fit only on that fold train history;
- the selector trains LightGBM on fully numeric matrices;
- default LightGBM tree depth is max_depth=5, with num_leaves<=2**5-1.

Expected usage in notebook:
    # 1) Define/import build_retail_features_with_split and
    #    prepare_lgbm_matrices_with_time_aware_catboost_encoding first.
    # 2) Then run:
    result_lgbm_fs = run_x5_lgbm_ranked_add_feature_selection_v1(
        df=df,
        feature_builder_func=build_retail_features_with_split,
        vladimir_features=vladimir_features,
        candidate_features=candidate_features,
        max_features_to_add=50,
        patience_no_improvement=100,
        verbose=True,
    )

Main outputs:
    result["selected_features"]             # final encoded numeric feature names for LightGBM
    result["selected_raw_feature_aliases"] # best-effort raw aliases mapped to selected encoded names
    result["selection_history"]
    result["trial_history"]
    result["candidate_importance"]
    result["fold_cache"]                   # optional, if return_fold_cache=True
"""

from __future__ import annotations

import gc
import os
import warnings
from dataclasses import dataclass
from typing import Any, Callable, Literal, Optional, Sequence

import numpy as np
import pandas as pd


# =============================================================================
# Metrics
# =============================================================================


def mape_score(
    y_true,
    y_pred,
    *,
    eps: float = 1e-9,
    clip_pred_lower: float | None = 0.0,
) -> float:
    """MAPE in percent. Example: 5.23 means 5.23%."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if clip_pred_lower is not None:
        y_pred = np.maximum(y_pred, clip_pred_lower)

    denom = np.where(np.abs(y_true) > eps, np.abs(y_true), np.nan)
    ape = np.abs(y_pred - y_true) / denom
    if np.all(np.isnan(ape)):
        return np.nan
    return float(np.nanmean(ape) * 100.0)


def competition_score_from_mape(mape: float) -> float:
    """Competition score: 100 * ((100 - min(MAPE, 100)) / 100) ** 2."""
    if pd.isna(mape):
        return np.nan
    return float(100.0 * ((100.0 - min(float(mape), 100.0)) / 100.0) ** 2)


def _unique_keep_order(values: Sequence[Any] | None) -> list:
    if values is None:
        return []
    seen = set()
    out = []
    for x in values:
        if x is None:
            continue
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def _month_index(year: int, month: int, min_year: int) -> int:
    year = int(year)
    month = int(month)
    if not 1 <= month <= 12:
        raise ValueError(f"month должен быть от 1 до 12, получено: {month}")
    return (year - int(min_year)) * 12 + (month - 1)


# =============================================================================
# Folds
# =============================================================================


def make_march_2025_feature_selection_folds(normalize: bool = True) -> list[dict]:
    """
    Temporal folds oriented to March-2025 final forecasting.

    Raw priorities:
        march_2024    = 0.6
        february_2025 = 0.2
        january_2025  = 0.2
        april_2024    = 0.3
        may_2024      = 0.3
    """
    folds = [
        {"fold_name": "march_2024", "name": "march_2024", "valid_year": 2024, "valid_month": 3, "weight": 0.6},
        {"fold_name": "february_2025", "name": "february_2025", "valid_year": 2025, "valid_month": 2, "weight": 0.2},
        {"fold_name": "january_2025", "name": "january_2025", "valid_year": 2025, "valid_month": 1, "weight": 0.2},
        {"fold_name": "april_2024", "name": "april_2024", "valid_year": 2024, "valid_month": 4, "weight": 0.3},
        {"fold_name": "may_2024", "name": "may_2024", "valid_year": 2024, "valid_month": 5, "weight": 0.3},
    ]
    return normalize_fold_weights(folds) if normalize else folds


def normalize_fold_weights(folds: Sequence[dict]) -> list[dict]:
    folds_out = [dict(f) for f in folds]
    for i, f in enumerate(folds_out):
        if "fold_name" not in f and "name" in f:
            f["fold_name"] = str(f["name"])
        if "name" not in f and "fold_name" in f:
            f["name"] = str(f["fold_name"])
        if "fold_name" not in f:
            f["fold_name"] = f"{int(f['valid_year'])}_{int(f['valid_month']):02d}"
            f["name"] = f["fold_name"]
        for k in ("valid_year", "valid_month"):
            if k not in f:
                raise ValueError(f"В fold #{i} отсутствует ключ '{k}'.")
        f["valid_year"] = int(f["valid_year"])
        f["valid_month"] = int(f["valid_month"])
        f["weight"] = float(f.get("weight", 1.0))

    total = float(np.sum([f["weight"] for f in folds_out]))
    if not np.isfinite(total) or total <= 0:
        raise ValueError("Сумма весов folds должна быть положительной и конечной.")
    for f in folds_out:
        f["weight"] = float(f["weight"] / total)
    return folds_out


def compute_weighted_score(fold_results: pd.DataFrame) -> dict:
    if fold_results.empty:
        raise ValueError("fold_results пустой.")
    weights = fold_results["weight"].astype(float).to_numpy()
    mapes = fold_results["mape"].astype(float).to_numpy()
    weighted_mape = np.nan if np.any(pd.isna(mapes)) else float(np.sum(weights * mapes) / np.sum(weights))
    return {
        "weighted_mape": weighted_mape,
        "competition_score": competition_score_from_mape(weighted_mape),
    }


# =============================================================================
# Pipeline callable resolution
# =============================================================================


def _resolve_pipeline_callable(func: Callable | None, name: str) -> Callable:
    """
    Resolve a callable either from explicit argument, current globals, or the companion
    pipeline module x5_retail_time_aware_catboost_encoding.py.
    """
    if func is not None:
        return func
    if name in globals() and callable(globals()[name]):
        return globals()[name]
    try:
        import x5_retail_time_aware_catboost_encoding as pipe  # type: ignore
        obj = getattr(pipe, name)
        if callable(obj):
            return obj
    except Exception:
        pass
    raise NameError(
        f"Не найдена функция '{name}'. Передай её явно или импортируй/выполни pipeline-блок до этого модуля."
    )


# =============================================================================
# Target inverse utilities compatible with prepare_lgbm_matrices...
# =============================================================================


def inverse_forecast_target_for_lgbm(
    raw_prediction,
    features: pd.DataFrame,
    *,
    mode: Literal[
        "original",
        "growth_log",
        "day_scaled_residual_log",
        "calendar_residual_log",
        "daily_rate_log",
        "direct_log1p",
    ] = "direct_log1p",
    target_col: str = "pto",
    residual_shrinkage: float = 1.0,
    clip_min: float | None = 0.0,
) -> np.ndarray:
    """Convert model-space LightGBM predictions back to raw RTO scale."""
    pred = np.asarray(raw_prediction, dtype=float)

    if mode == "original":
        out = pred
    elif mode == "direct_log1p":
        out = np.expm1(pred)
    elif mode == "growth_log":
        anchor_col = f"{target_col}_lag_1"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='growth_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(pred)
    elif mode == "day_scaled_residual_log":
        anchor_col = "day_scaled_lag1_prediction"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='day_scaled_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(float(residual_shrinkage) * pred)
    elif mode == "calendar_residual_log":
        anchor_col = "calendar_heuristic_prediction"
        if anchor_col not in features.columns:
            raise ValueError(f"Для mode='calendar_residual_log' нужна колонка '{anchor_col}'.")
        anchor = pd.to_numeric(features[anchor_col], errors="coerce").astype(float).to_numpy()
        out = anchor * np.exp(float(residual_shrinkage) * pred)
    elif mode == "daily_rate_log":
        day_col = "n_days_in_month" if "n_days_in_month" in features.columns else "days_in_month"
        if day_col not in features.columns:
            raise ValueError("Для mode='daily_rate_log' нужна колонка 'n_days_in_month' или 'days_in_month'.")
        days = pd.to_numeric(features[day_col], errors="coerce").astype(float).to_numpy()
        out = days * np.exp(pred)
    else:
        raise ValueError(f"Неизвестный mode='{mode}'.")

    out = np.asarray(out, dtype=float)
    out = np.where(np.isfinite(out), out, np.nan)
    if clip_min is not None:
        out = np.maximum(out, float(clip_min))
    return out


# =============================================================================
# LightGBM params
# =============================================================================


def make_default_lgbm_params(
    *,
    random_state: int = 42,
    n_estimators: int = 700,
    learning_rate: float = 0.05,
    max_depth: int = 5,
    num_leaves: int | None = None,
    objective: str = "regression_l1",
    metric: str = "mae",
    n_jobs: int = -1,
) -> dict:
    """
    Default LightGBM proxy model for feature selection.

    Depth policy:
    - max_depth defaults to 5;
    - num_leaves defaults to min(31, 2**max_depth - 1), so it is consistent with depth 5.
    """
    max_depth = int(max_depth)
    if num_leaves is None:
        if max_depth > 0:
            num_leaves = min(31, 2 ** max_depth - 1)
        else:
            num_leaves = 31

    return {
        "objective": objective,
        "metric": metric,
        "n_estimators": int(n_estimators),
        "learning_rate": float(learning_rate),
        "max_depth": max_depth,
        "num_leaves": int(num_leaves),
        "min_child_samples": 80,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.10,
        "reg_lambda": 5.0,
        "random_state": int(random_state),
        "n_jobs": int(n_jobs),
        "verbosity": -1,
        "force_col_wise": True,
    }


def normalize_lgbm_params(params: dict | None, *, random_state: int = 42) -> dict:
    out = make_default_lgbm_params(random_state=random_state) if params is None else dict(params)
    out.setdefault("random_state", random_state)
    out.setdefault("verbosity", -1)
    out.setdefault("n_jobs", -1)
    out.setdefault("objective", "regression_l1")
    out.setdefault("metric", "mae")
    out.setdefault("max_depth", 5)
    out["max_depth"] = int(out["max_depth"])

    # Keep num_leaves consistent with max_depth when max_depth is positive.
    if "num_leaves" not in out or out["num_leaves"] is None:
        out["num_leaves"] = min(31, 2 ** out["max_depth"] - 1) if out["max_depth"] > 0 else 31
    elif out["max_depth"] > 0:
        max_leaves = 2 ** out["max_depth"] - 1
        out["num_leaves"] = min(int(out["num_leaves"]), int(max_leaves))

    return out


# =============================================================================
# X5 feature-builder kwargs and categorical source columns
# =============================================================================


def make_x5_lgbm_feature_selection_builder_kwargs() -> dict:
    """Feature-builder kwargs synchronized with the current final holdout pipeline."""
    return {
        "base_feature_kwargs": {
            "target_history_scale": "log",
            "monthly_history_scale": "raw",
            "target_rolling_windows": (3, 6, 12),
            "monthly_cross_lags": (1, 3, 12),
            "monthly_cross_feature_types": (
                "promo_share",
                "promo_x_items",
                "items_x_hours",
                "cancellations_per_item",
                "cancellations_per_hour",
            ),
            "drop_current_monthly_cols": True,
            "drop_current_monthly_derived_cols": True,
            # Keep year so year/year_cat can be encoded/scored.
            "drop_year_col": False,
            "drop_obvious_duplicates": False,
            "fill_history_na": False,
            "downcast_float_features": True,
        }
    }


def make_x5_lgbm_numeric_cat_source_cols(
    *,
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    include_working_hours_lag_categoricals: bool = True,
) -> tuple[str, ...]:
    cols = [
        id_col,
        "alcohol_license_flag",
        year_col,
        month_col,
        "prev_month",
        "calendar_month_num",
        "marketplaces_deliveries_parcel_lockers_100m",
        "medical_institutions_and_pharmacies_300m",
        "schools_300m",
        "public_transport_stops_300m",
        "grocery_stores_500m",
        "pyaterochka_stores_500m",
        "number_of_cash_registers",
    ]
    if include_working_hours_lag_categoricals:
        cols.extend([
            "working_hours_per_day_lag_1",
            "working_hours_per_day_lag_3",
            "working_hours_per_day_lag_12",
        ])
    return tuple(_unique_keep_order(cols))


# =============================================================================
# Fold cache with encoded matrices
# =============================================================================


@dataclass
class EncodedFold:
    fold_name: str
    valid_year: int
    valid_month: int
    weight: float
    X_train: pd.DataFrame
    y_train: pd.Series
    X_valid: pd.DataFrame
    y_valid: pd.Series | None
    y_valid_raw: np.ndarray
    encoding_info: dict


def _safe_builder_kwargs(feature_builder_kwargs: dict | None) -> dict:
    feature_builder_kwargs = {} if feature_builder_kwargs is None else dict(feature_builder_kwargs)
    reserved = {
        "df", "split_mode", "id_col", "year_col", "month_col", "time_col",
        "target_col", "min_year", "val_year", "val_month", "verbose",
    }
    return {k: v for k, v in feature_builder_kwargs.items() if k not in reserved}


def _build_feature_output_for_fold(
    df: pd.DataFrame,
    fold: dict,
    *,
    feature_builder_func: Callable,
    feature_builder_kwargs: dict | None,
    id_col: str,
    year_col: str,
    month_col: str,
    time_col: str,
    target_col: str,
    min_year: int,
) -> dict:
    safe_kwargs = _safe_builder_kwargs(feature_builder_kwargs)
    built = feature_builder_func(
        df,
        split_mode="train_val",
        id_col=id_col,
        year_col=year_col,
        month_col=month_col,
        time_col=time_col,
        target_col=target_col,
        min_year=min_year,
        val_year=int(fold["valid_year"]),
        val_month=int(fold["valid_month"]),
        expected_test_size=None,
        verbose=False,
        **safe_kwargs,
    )
    if not isinstance(built, dict):
        raise TypeError("feature_builder_func должен возвращать dict.")
    if built.get("train_features") is None or built.get("val_features") is None:
        raise ValueError("feature_builder_func должен вернуть train_features и val_features.")
    return built


def _infer_raw_feature_universe(
    feature_output: dict,
    *,
    target_col: str,
    forbidden_features: set[str],
) -> list[str]:
    cols = list(feature_output.get("feature_cols") or [])
    if not cols:
        ref = feature_output.get("train_features")
        if isinstance(ref, pd.DataFrame):
            cols = [c for c in ref.columns if c != target_col]
    return [c for c in _unique_keep_order(cols) if c not in forbidden_features]


def build_lgbm_encoded_fold_cache(
    df: pd.DataFrame,
    *,
    folds: Sequence[dict],
    feature_builder_func: Callable,
    matrix_preparer_func: Callable | None = None,
    feature_builder_kwargs: dict | None = None,
    raw_features_to_use: Sequence[str] | None = None,
    encoding_kwargs: dict | None = None,
    target_mode: Literal[
        "original",
        "growth_log",
        "day_scaled_residual_log",
        "calendar_residual_log",
        "daily_rate_log",
        "direct_log1p",
    ] = "direct_log1p",
    target_col: str = "pto",
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    min_year: int = 2023,
    forbidden_features: Sequence[str] | None = None,
    verbose: bool = True,
) -> tuple[list[EncodedFold], dict, pd.DataFrame]:
    """
    Build fold cache with fully numeric LightGBM matrices.

    The key leakage guarantee is delegated to
    prepare_lgbm_matrices_with_time_aware_catboost_encoding(...), called separately
    inside every temporal fold.
    """
    matrix_preparer_func = _resolve_pipeline_callable(
        matrix_preparer_func,
        "prepare_lgbm_matrices_with_time_aware_catboost_encoding",
    )
    encoding_kwargs = {} if encoding_kwargs is None else dict(encoding_kwargs)

    forbidden = set(_unique_keep_order(forbidden_features))
    forbidden.update({target_col, "__part__", "__row_order__"})

    fold_cache: list[EncodedFold] = []
    encoding_records = []
    raw_to_encoded: dict[str, str] = {}

    for fold in folds:
        fold_name = str(fold.get("fold_name") or fold.get("name"))
        if verbose:
            print(f"[fold-build:lgbm] {fold_name}: valid={fold['valid_year']}-{fold['valid_month']:02d}")

        built = _build_feature_output_for_fold(
            df,
            fold,
            feature_builder_func=feature_builder_func,
            feature_builder_kwargs=feature_builder_kwargs,
            id_col=id_col,
            year_col=year_col,
            month_col=month_col,
            time_col=time_col,
            target_col=target_col,
            min_year=min_year,
        )

        if raw_features_to_use is None:
            fold_raw_features = _infer_raw_feature_universe(
                built,
                target_col=target_col,
                forbidden_features=forbidden,
            )
        else:
            fold_raw_features = [f for f in _unique_keep_order(raw_features_to_use) if f not in forbidden]

        matrices = matrix_preparer_func(
            built,
            features_to_use=fold_raw_features,
            target_mode=target_mode,
            target_col=target_col,
            id_col=id_col,
            time_col=time_col,
            verbose=False,
            **encoding_kwargs,
        )

        val_pack = matrices.get("validation")
        if val_pack is None:
            raise ValueError(f"Для fold={fold_name} matrix_preparer не вернул validation pack.")

        X_train = val_pack["X_train"].copy()
        y_train = pd.Series(val_pack["y_train"]).reset_index(drop=True)
        X_valid = val_pack["X_val"].copy()
        y_valid = None if val_pack.get("y_val") is None else pd.Series(val_pack["y_val"]).reset_index(drop=True)

        val_features = built["val_features"].reset_index(drop=True)
        if target_col not in val_features.columns:
            raise ValueError(f"В val_features нет target_col='{target_col}'.")
        y_valid_raw = pd.to_numeric(val_features[target_col], errors="coerce").astype(float).to_numpy()

        info = matrices.get("catboost_encoding_info", {}).get("validation", {})
        cat_used = list(info.get("cat_features_used") or [])
        enc_created = list(info.get("encoded_features_created") or [])
        final_features = list(info.get("final_features") or list(X_train.columns))

        for cat_col, enc_col in zip(cat_used, enc_created):
            raw_to_encoded[cat_col] = enc_col
            if cat_col.endswith("_cat"):
                raw_to_encoded[cat_col[:-4]] = enc_col

        for c in final_features:
            raw_to_encoded.setdefault(c, c)

        for cat_col, enc_col in zip(cat_used, enc_created):
            encoding_records.append(
                {
                    "fold_name": fold_name,
                    "cat_feature": cat_col,
                    "encoded_feature": enc_col,
                    "encoded_train_nan_share": (info.get("encoded_train_nan_share") or {}).get(enc_col, np.nan),
                    "first_block_value": info.get("first_block_value"),
                    "unknown_value": info.get("unknown_value"),
                    "smoothing": info.get("smoothing"),
                    "min_count": info.get("min_count"),
                }
            )

        fold_cache.append(
            EncodedFold(
                fold_name=fold_name,
                valid_year=int(fold["valid_year"]),
                valid_month=int(fold["valid_month"]),
                weight=float(fold["weight"]),
                X_train=X_train.reset_index(drop=True),
                y_train=y_train,
                X_valid=X_valid.reset_index(drop=True),
                y_valid=y_valid,
                y_valid_raw=y_valid_raw,
                encoding_info=info,
            )
        )

        if verbose:
            print(f"    X_train={X_train.shape}, X_valid={X_valid.shape}, encoded_cats={len(cat_used)}")

    # Align all fold matrices to union of encoded final columns.
    all_cols = []
    for fd in fold_cache:
        all_cols.extend(list(fd.X_train.columns))
        all_cols.extend(list(fd.X_valid.columns))
    all_cols = _unique_keep_order(all_cols)

    for fd in fold_cache:
        for c in all_cols:
            if c not in fd.X_train.columns:
                fd.X_train[c] = np.nan
            if c not in fd.X_valid.columns:
                fd.X_valid[c] = np.nan
        fd.X_train = fd.X_train[all_cols].copy()
        fd.X_valid = fd.X_valid[all_cols].copy()

    encoding_meta = pd.DataFrame(encoding_records)
    feature_map = {
        "raw_to_encoded": raw_to_encoded,
        "all_encoded_features": all_cols,
    }
    return fold_cache, feature_map, encoding_meta


# =============================================================================
# Feature name resolution
# =============================================================================


def resolve_features_for_lgbm_matrix(
    features: Sequence[str] | None,
    *,
    feature_map: dict,
    available_features: Sequence[str],
    forbidden_features: Sequence[str] | None = None,
    strict: bool = False,
) -> list[str]:
    """
    Map raw pipeline names to encoded matrix names.

    Examples:
        region      -> region_cb
        new_id      -> new_id_cat_cb, if new_id was represented as new_id_cat
        pto_lag_1   -> pto_lag_1
    """
    if features is None:
        return []

    available = set(available_features)
    forbidden = set(_unique_keep_order(forbidden_features))
    raw_to_encoded = feature_map.get("raw_to_encoded") or {}

    resolved = []
    missing = []
    for f in _unique_keep_order(features):
        if f in forbidden:
            continue
        candidates = []
        if f in available:
            candidates.append(f)
        mapped = raw_to_encoded.get(f)
        if mapped is not None:
            candidates.append(mapped)
        if f.endswith("_cat"):
            mapped2 = raw_to_encoded.get(f[:-4])
            if mapped2 is not None:
                candidates.append(mapped2)
        else:
            mapped3 = raw_to_encoded.get(f"{f}_cat")
            if mapped3 is not None:
                candidates.append(mapped3)
        # If user already passed encoded name.
        if f.endswith("_cb") and f in available:
            candidates.append(f)

        chosen = None
        for c in candidates:
            if c in available:
                chosen = c
                break
        if chosen is None:
            missing.append(f)
            continue
        if chosen not in resolved:
            resolved.append(chosen)

    if missing and strict:
        raise ValueError(f"Не удалось сопоставить признаки с LGBM-матрицей: {missing[:50]}")
    if missing and not strict:
        warnings.warn(
            "Некоторые признаки не найдены в encoded LGBM-матрицах и будут пропущены: "
            + str(missing[:50])
            + (" ..." if len(missing) > 50 else "")
        )
    return resolved


def make_raw_alias_table(feature_map: dict) -> pd.DataFrame:
    raw_to_encoded = feature_map.get("raw_to_encoded") or {}
    return pd.DataFrame(
        [{"raw_feature": k, "encoded_feature": v} for k, v in raw_to_encoded.items()]
    ).drop_duplicates().sort_values(["encoded_feature", "raw_feature"]).reset_index(drop=True)


# =============================================================================
# LightGBM fitting/evaluation
# =============================================================================


def _make_lgbm_model(params: dict):
    try:
        from lightgbm import LGBMRegressor
    except ImportError as e:
        raise ImportError("LightGBM не установлен. Установи его через `pip install lightgbm`.") from e
    return LGBMRegressor(**params)


def _fit_lgbm(
    X_train: pd.DataFrame,
    y_train,
    *,
    params: dict,
):
    model = _make_lgbm_model(params)
    model.fit(X_train, y_train)
    return model


def _predict_constant_baseline(
    y_train_model,
    X_valid: pd.DataFrame,
    *,
    target_mode: str,
    target_col: str,
    constant_strategy: Literal["median", "mean"] = "median",
    clip_pred_lower: float | None = 0.0,
) -> np.ndarray:
    y_train_model = np.asarray(y_train_model, dtype=float)
    if constant_strategy == "median":
        const = float(np.nanmedian(y_train_model))
    elif constant_strategy == "mean":
        const = float(np.nanmean(y_train_model))
    else:
        raise ValueError("constant_strategy должен быть 'median' или 'mean'.")
    pred_model = np.full(len(X_valid), const, dtype=float)
    return inverse_forecast_target_for_lgbm(
        pred_model,
        X_valid,
        mode=target_mode,
        target_col=target_col,
        clip_min=clip_pred_lower,
    )


def evaluate_lgbm_feature_set(
    fold_cache: Sequence[EncodedFold],
    features: Sequence[str],
    *,
    lgbm_params: dict,
    target_mode: str = "direct_log1p",
    target_col: str = "pto",
    clip_pred_lower: float | None = 0.0,
    constant_strategy: Literal["median", "mean"] = "median",
    return_models: bool = False,
    verbose: bool = False,
) -> dict:
    features = _unique_keep_order(features)
    records = []
    models = {}

    for fd in fold_cache:
        available = set(fd.X_train.columns).intersection(fd.X_valid.columns)
        fold_features = [f for f in features if f in available]

        if not fold_features:
            pred_raw = _predict_constant_baseline(
                fd.y_train,
                fd.X_valid,
                target_mode=target_mode,
                target_col=target_col,
                constant_strategy=constant_strategy,
                clip_pred_lower=clip_pred_lower,
            )
            model = None
        else:
            X_tr = fd.X_train[fold_features].copy()
            X_va = fd.X_valid[fold_features].copy()
            model = _fit_lgbm(X_tr, fd.y_train, params=lgbm_params)
            pred_model = model.predict(X_va)
            pred_raw = inverse_forecast_target_for_lgbm(
                pred_model,
                X_va,
                mode=target_mode,
                target_col=target_col,
                clip_min=clip_pred_lower,
            )

        mape = mape_score(fd.y_valid_raw, pred_raw, clip_pred_lower=clip_pred_lower)
        rec = {
            "fold_name": fd.fold_name,
            "valid_year": fd.valid_year,
            "valid_month": fd.valid_month,
            "weight": fd.weight,
            "mape": mape,
            "competition_score": competition_score_from_mape(mape),
            "n_train": len(fd.X_train),
            "n_valid": len(fd.X_valid),
            "n_features": len(fold_features),
            "target_mode": target_mode,
        }
        records.append(rec)
        if return_models and model is not None:
            models[fd.fold_name] = model

        if verbose:
            print(
                f"    [{fd.fold_name}] MAPE={mape:.6f}, "
                f"score={rec['competition_score']:.6f}, n_features={len(fold_features)}"
            )

        del model
        gc.collect()

    fold_results = pd.DataFrame(records)
    weighted = compute_weighted_score(fold_results)
    out = {
        "weighted_mape": weighted["weighted_mape"],
        "competition_score": weighted["competition_score"],
        "fold_results": fold_results,
    }
    if return_models:
        out["models"] = models
    return out


def _fold_mape_map(results: dict) -> dict[str, float]:
    fr = results.get("fold_results")
    if fr is None or fr.empty:
        return {}
    return {str(r["fold_name"]): float(r["mape"]) for _, r in fr.iterrows()}


def _count_improved_folds(prev_results: dict, new_results: dict, *, min_fold_improvement: float = 0.0) -> int:
    prev = _fold_mape_map(prev_results)
    new = _fold_mape_map(new_results)
    return int(sum(prev[k] - new[k] > float(min_fold_improvement) for k in prev if k in new))


def _fold_degradation_info(prev_results: dict, new_results: dict) -> tuple[float, dict]:
    prev = _fold_mape_map(prev_results)
    new = _fold_mape_map(new_results)
    common = sorted(set(prev).intersection(new))
    deltas = {k: float(new[k] - prev[k]) for k in common}
    positive = [v for v in deltas.values() if np.isfinite(v) and v > 0]
    return (float(max(positive)) if positive else 0.0), deltas


def _check_acceptance(
    prev_results: dict,
    trial_results: dict,
    *,
    min_improvement: float,
    min_improved_folds: int,
    max_allowed_fold_degradation: float | None,
    max_allowed_degradation_by_fold: dict | None,
) -> tuple[bool, list[str], dict]:
    prev_mape = float(prev_results["weighted_mape"])
    new_mape = float(trial_results["weighted_mape"])
    improvement = prev_mape - new_mape
    n_improved = _count_improved_folds(prev_results, trial_results)
    max_degradation, degradation_by_fold = _fold_degradation_info(prev_results, trial_results)

    eligible = True
    fail = []
    if not np.isfinite(improvement) or improvement < float(min_improvement):
        eligible = False
        fail.append("min_improvement")
    if int(n_improved) < int(min_improved_folds):
        eligible = False
        fail.append("min_improved_folds")
    if max_allowed_fold_degradation is not None and max_degradation > float(max_allowed_fold_degradation):
        eligible = False
        fail.append("max_allowed_fold_degradation")
    if max_allowed_degradation_by_fold:
        for name, allowed in max_allowed_degradation_by_fold.items():
            if str(name) in degradation_by_fold and degradation_by_fold[str(name)] > float(allowed):
                eligible = False
                fail.append(f"max_degradation_by_fold:{name}")

    meta = {
        "previous_weighted_mape": prev_mape,
        "new_weighted_mape": new_mape,
        "improvement": float(improvement),
        "n_improved_folds": int(n_improved),
        "max_fold_degradation": float(max_degradation),
        "degradation_by_fold": degradation_by_fold,
    }
    return eligible, fail, meta


def _extract_named_fold_values(results: dict, prefix: str = "mape") -> dict:
    out = {}
    fr = results.get("fold_results")
    if fr is None:
        return out
    for _, row in fr.iterrows():
        out[f"{prefix}_{row['fold_name']}"] = float(row["mape"])
    return out


def _extract_delta_values(prev_results: dict, new_results: dict, prefix: str = "delta") -> dict:
    prev = _fold_mape_map(prev_results)
    new = _fold_mape_map(new_results)
    return {f"{prefix}_{k}": float(prev[k] - new[k]) for k in prev if k in new}


# =============================================================================
# Proxy ranking by LightGBM gain importance
# =============================================================================


def get_lgbm_proxy_feature_importance(
    fold_cache: Sequence[EncodedFold],
    *,
    fixed_features: Sequence[str],
    candidate_features: Sequence[str],
    lgbm_params: dict,
    importance_type: Literal["gain", "split"] = "gain",
    max_proxy_features: int | None = None,
    verbose: bool = False,
) -> pd.DataFrame:
    fixed_features = _unique_keep_order(fixed_features)
    candidate_features = _unique_keep_order(candidate_features)
    all_features = _unique_keep_order(fixed_features + candidate_features)

    if max_proxy_features is not None and len(candidate_features) > int(max_proxy_features):
        candidate_features = candidate_features[: int(max_proxy_features)]
        all_features = _unique_keep_order(fixed_features + candidate_features)

    if not candidate_features:
        return pd.DataFrame(columns=["feature", "importance_mean", "importance_std", "importance_by_fold", "is_fixed"])

    fold_frames = []
    for fd in fold_cache:
        available = set(fd.X_train.columns).intersection(fd.X_valid.columns)
        fold_features = [f for f in all_features if f in available]
        if not fold_features:
            imp = pd.DataFrame({"feature": all_features, f"importance_{fd.fold_name}": 0.0})
            fold_frames.append(imp)
            continue

        model = _fit_lgbm(fd.X_train[fold_features], fd.y_train, params=lgbm_params)
        booster = model.booster_
        values = booster.feature_importance(importance_type=importance_type)
        imp_fold = pd.DataFrame({"feature": fold_features, f"importance_{fd.fold_name}": values.astype(float)})

        # Add absent features with zero importance for this fold.
        absent = [f for f in all_features if f not in fold_features]
        if absent:
            imp_fold = pd.concat(
                [imp_fold, pd.DataFrame({"feature": absent, f"importance_{fd.fold_name}": 0.0})],
                ignore_index=True,
            )
        fold_frames.append(imp_fold)
        del model
        gc.collect()

        if verbose:
            print(f"[proxy:lgbm] fold={fd.fold_name}, n_features={len(fold_features)}")

    importance_df = fold_frames[0]
    for part in fold_frames[1:]:
        importance_df = importance_df.merge(part, on="feature", how="outer")

    imp_cols = [c for c in importance_df.columns if c.startswith("importance_")]
    importance_df[imp_cols] = importance_df[imp_cols].fillna(0.0)
    importance_df["importance_mean"] = importance_df[imp_cols].mean(axis=1)
    importance_df["importance_std"] = importance_df[imp_cols].std(axis=1)
    importance_df["importance_min"] = importance_df[imp_cols].min(axis=1)
    importance_df["importance_median"] = importance_df[imp_cols].median(axis=1)
    importance_df["n_positive_folds"] = (importance_df[imp_cols] > 0).sum(axis=1).astype(int)
    importance_df["positive_fold_share"] = importance_df["n_positive_folds"] / max(1, len(imp_cols))
    importance_df["importance_by_fold"] = importance_df.apply(
        lambda row: {c.replace("importance_", ""): float(row[c]) for c in imp_cols},
        axis=1,
    )
    importance_df["is_fixed"] = importance_df["feature"].isin(set(fixed_features))
    return importance_df


def add_stability_columns(
    importance_df: pd.DataFrame,
    *,
    stability_std_penalty: float = 0.50,
    positive_fold_bonus: float = 0.25,
) -> pd.DataFrame:
    out = importance_df.copy()
    if "importance_mean" not in out.columns:
        out["importance_mean"] = 0.0
    if "importance_std" not in out.columns:
        out["importance_std"] = 0.0
    if "positive_fold_share" not in out.columns:
        out["positive_fold_share"] = 0.0
    out["importance_mean"] = pd.to_numeric(out["importance_mean"], errors="coerce").fillna(0.0)
    out["importance_std"] = pd.to_numeric(out["importance_std"], errors="coerce").fillna(0.0)
    out["positive_fold_share"] = pd.to_numeric(out["positive_fold_share"], errors="coerce").fillna(0.0)
    out["stability_score"] = (
        out["importance_mean"] * (1.0 + float(positive_fold_bonus) * out["positive_fold_share"])
        - float(stability_std_penalty) * out["importance_std"]
    )
    return out


# =============================================================================
# Backward pruning
# =============================================================================


def backward_prune_once_lgbm(
    *,
    fold_cache: Sequence[EncodedFold],
    current_features: list[str],
    current_results: dict,
    added_features: list[str],
    lgbm_params: dict,
    target_mode: str,
    target_col: str,
    clip_pred_lower: float | None,
    constant_strategy: str,
    protected_features: Sequence[str],
    initial_features: Sequence[str],
    pruning_min_improvement: float,
    pruning_allow_neutral: bool,
    pruning_neutral_tolerance: float,
    pruning_max_allowed_fold_degradation: float | None,
    pruning_order: Literal["reverse", "forward"] = "reverse",
    prune_initial_features: bool = False,
    verbose: bool = True,
) -> tuple[list[str], dict, list[str], list[dict]]:
    protected = set(protected_features)
    initial = set(initial_features)
    source = list(current_features) if prune_initial_features else list(added_features)
    prunable = []
    for f in source:
        if f not in current_features:
            continue
        if f in protected:
            continue
        if (not prune_initial_features) and f in initial:
            continue
        if f not in prunable:
            prunable.append(f)

    if pruning_order == "reverse":
        prunable = list(reversed(prunable))
    elif pruning_order != "forward":
        raise ValueError("pruning_order должен быть 'reverse' или 'forward'.")

    records = []
    if verbose:
        print(f"[backward-pruning:lgbm] features before={len(current_features)}, prunable={len(prunable)}")

    for feature in prunable:
        trial_features = [f for f in current_features if f != feature]
        trial_results = evaluate_lgbm_feature_set(
            fold_cache,
            trial_features,
            lgbm_params=lgbm_params,
            target_mode=target_mode,
            target_col=target_col,
            clip_pred_lower=clip_pred_lower,
            constant_strategy=constant_strategy,
            return_models=False,
            verbose=False,
        )
        prev_mape = float(current_results["weighted_mape"])
        new_mape = float(trial_results["weighted_mape"])
        improvement = prev_mape - new_mape
        n_improved = _count_improved_folds(current_results, trial_results)
        max_deg, deg_by_fold = _fold_degradation_info(current_results, trial_results)

        positive_accept = np.isfinite(improvement) and improvement >= float(pruning_min_improvement)
        neutral_accept = bool(pruning_allow_neutral) and np.isfinite(improvement) and improvement >= -float(pruning_neutral_tolerance)
        eligible = bool(positive_accept or neutral_accept)
        fail = []
        if not eligible:
            fail.append("pruning_improvement")
        if pruning_max_allowed_fold_degradation is not None and max_deg > float(pruning_max_allowed_fold_degradation):
            eligible = False
            fail.append("pruning_max_allowed_fold_degradation")

        rec = {
            "pruned_feature": feature,
            "action": "removed" if eligible else "kept",
            "previous_weighted_mape": prev_mape,
            "new_weighted_mape": new_mape,
            "improvement": float(improvement),
            "n_improved_folds": int(n_improved),
            "max_fold_degradation": float(max_deg),
            "fail_reasons": ",".join(fail),
            "features_count_before": len(current_features),
            "features_count_after_trial": len(trial_features),
        }
        rec.update(_extract_named_fold_values(trial_results, prefix="mape"))
        rec.update(_extract_delta_values(current_results, trial_results, prefix="delta"))
        records.append(rec)

        if eligible:
            current_features = trial_features
            current_results = trial_results
            added_features = [f for f in added_features if f != feature]
            if verbose:
                print(f"[backward-pruning:lgbm] REMOVED {feature}: {prev_mape:.6f}->{new_mape:.6f}")
        elif verbose:
            print(f"[backward-pruning:lgbm] kept {feature}: {prev_mape:.6f}->{new_mape:.6f}; {rec['fail_reasons']}")

    return current_features, current_results, added_features, records


# =============================================================================
# Main ranked-add selector
# =============================================================================


def ranked_add_features_with_time_aware_lgbm_v1(
    *,
    df: pd.DataFrame,
    feature_builder_func: Callable,
    matrix_preparer_func: Callable | None = None,
    base_features: Sequence[str] | None = None,
    protected_features: Sequence[str] | None = None,
    candidate_features: Sequence[str] | None = None,
    feature_builder_kwargs: dict | None = None,
    encoding_kwargs: dict | None = None,
    fold_cache: Sequence[EncodedFold] | None = None,
    feature_map: dict | None = None,

    target_col: str = "pto",
    id_col: str = "new_id",
    year_col: str = "year",
    month_col: str = "month",
    time_col: str = "year_month_index",
    min_year: int = 2023,
    target_mode: Literal[
        "original",
        "growth_log",
        "day_scaled_residual_log",
        "calendar_residual_log",
        "daily_rate_log",
        "direct_log1p",
    ] = "direct_log1p",

    folds: Sequence[dict] | None = None,
    normalize_fold_weights: bool = True,

    lgbm_params: dict | None = None,
    random_state: int = 42,

    start_from_empty: bool = False,
    force_include_protected: bool = False,

    use_proxy_filter: bool = True,
    top_k_candidates: int | None = 150,
    max_proxy_features: int | None = None,
    importance_type: Literal["gain", "split"] = "gain",
    ranking_score_col: str = "stability_score",
    stability_std_penalty: float = 0.50,
    positive_fold_bonus: float = 0.25,
    min_proxy_importance: float | None = None,
    min_proxy_importance_on: str = "importance_mean",

    enable_periodic_reranking: bool = True,
    rerank_after_n_added: int = 5,
    rerank_after_n_checked: int | None = None,

    min_improvement: float = 0.005,
    min_improved_folds: int = 2,
    max_features_to_add: int = 40,
    max_candidates_to_check: int | None = None,
    patience_no_improvement: int = 25,
    max_allowed_fold_degradation: float | None = 0.05,
    max_allowed_degradation_by_fold: dict | None = None,

    enable_backward_pruning: bool = True,
    backward_pruning_every_n_added: int = 5,
    backward_pruning_min_improvement: float = 0.0,
    backward_pruning_allow_neutral: bool = True,
    backward_pruning_neutral_tolerance: float = 0.002,
    backward_pruning_max_allowed_fold_degradation: float | None = 0.03,
    backward_pruning_order: Literal["reverse", "forward"] = "reverse",
    backward_pruning_include_initial_features: bool = False,

    forbidden_features: Sequence[str] | None = None,
    clip_pred_lower: float | None = 0.0,
    constant_strategy: Literal["median", "mean"] = "median",
    verbose: bool = True,
    gc_after_each_trial: bool = True,
    return_fold_cache: bool = False,
) -> dict:
    """
    Adaptive ranked-add feature selection on LightGBM matrices produced by
    time-aware CatBoost-like ordered target encoding.

    Feature semantics:
    - input base/protected/candidate features may be raw pipeline names;
    - raw categorical names are mapped to encoded *_cb matrix features;
    - selected_features are final numeric LightGBM matrix columns.
    """
    if folds is None:
        folds = make_march_2025_feature_selection_folds(normalize=False)
    folds = globals()["normalize_fold_weights"](folds) if normalize_fold_weights else [dict(f) for f in folds]

    lgbm_params = normalize_lgbm_params(lgbm_params, random_state=random_state)

    forbidden = set(_unique_keep_order(forbidden_features))
    forbidden.update({target_col, "__part__", "__row_order__"})

    base_features = _unique_keep_order(base_features)
    protected_features = _unique_keep_order(protected_features)
    candidate_features = _unique_keep_order(candidate_features)

    # Build raw universe for encoding. Include all user-visible candidate lists.
    raw_features_to_use = _unique_keep_order(base_features + protected_features + candidate_features)
    if not raw_features_to_use:
        raw_features_to_use = None

    if fold_cache is None:
        fold_cache, feature_map_built, encoding_meta = build_lgbm_encoded_fold_cache(
            df,
            folds=folds,
            feature_builder_func=feature_builder_func,
            matrix_preparer_func=matrix_preparer_func,
            feature_builder_kwargs=feature_builder_kwargs,
            raw_features_to_use=raw_features_to_use,
            encoding_kwargs=encoding_kwargs,
            target_mode=target_mode,
            target_col=target_col,
            id_col=id_col,
            year_col=year_col,
            month_col=month_col,
            time_col=time_col,
            min_year=min_year,
            forbidden_features=forbidden,
            verbose=verbose,
        )
        feature_map = feature_map_built
    else:
        fold_cache = list(fold_cache)
        if feature_map is None:
            all_cols = []
            for fd in fold_cache:
                all_cols.extend(list(fd.X_train.columns))
            feature_map = {"raw_to_encoded": {c: c for c in _unique_keep_order(all_cols)}, "all_encoded_features": _unique_keep_order(all_cols)}
        encoding_meta = pd.DataFrame()

    available_features = feature_map.get("all_encoded_features") or list(fold_cache[0].X_train.columns)
    alias_table = make_raw_alias_table(feature_map)

    base_encoded = resolve_features_for_lgbm_matrix(
        base_features,
        feature_map=feature_map,
        available_features=available_features,
        forbidden_features=forbidden,
        strict=False,
    )
    protected_encoded = resolve_features_for_lgbm_matrix(
        protected_features,
        feature_map=feature_map,
        available_features=available_features,
        forbidden_features=forbidden,
        strict=False,
    )

    if candidate_features:
        candidate_encoded = resolve_features_for_lgbm_matrix(
            candidate_features,
            feature_map=feature_map,
            available_features=available_features,
            forbidden_features=forbidden,
            strict=False,
        )
    else:
        candidate_encoded = [c for c in available_features if c not in forbidden]

    if start_from_empty:
        if force_include_protected:
            initial_features = list(protected_encoded)
            candidate_pool = _unique_keep_order(base_encoded + candidate_encoded)
        else:
            initial_features = []
            candidate_pool = _unique_keep_order(protected_encoded + base_encoded + candidate_encoded)
    else:
        initial_features = _unique_keep_order(protected_encoded + base_encoded)
        candidate_pool = list(candidate_encoded)

    initial_set = set(initial_features)
    candidate_pool = [f for f in _unique_keep_order(candidate_pool) if f not in initial_set and f in available_features and f not in forbidden]

    if verbose:
        print("=" * 100)
        print("ADAPTIVE RANKED-ADD FEATURE SELECTION WITH TIME-AWARE ENCODING + LIGHTGBM")
        print("=" * 100)
        print(f"target_mode:              {target_mode}")
        print(f"lgbm objective:           {lgbm_params.get('objective')}")
        print(f"lgbm metric:              {lgbm_params.get('metric')}")
        print(f"lgbm max_depth:           {lgbm_params.get('max_depth')}")
        print(f"lgbm num_leaves:          {lgbm_params.get('num_leaves')}")
        print(f"start_from_empty:         {start_from_empty}")
        print(f"initial_features:         {len(initial_features)}")
        print(f"candidate_pool:           {len(candidate_pool)}")
        print(f"encoded matrix features:  {len(available_features)}")
        print(f"ranking_score_col:        {ranking_score_col}")
        print("[folds]")
        print(pd.DataFrame(folds)[["fold_name", "valid_year", "valid_month", "weight"]].to_string(index=False))
        print()

    baseline_results = evaluate_lgbm_feature_set(
        fold_cache,
        initial_features,
        lgbm_params=lgbm_params,
        target_mode=target_mode,
        target_col=target_col,
        clip_pred_lower=clip_pred_lower,
        constant_strategy=constant_strategy,
        return_models=False,
        verbose=False,
    )

    current_features = list(initial_features)
    current_results = baseline_results
    added_features = []
    remaining_candidates = list(candidate_pool)

    if verbose:
        print(
            f"[baseline:lgbm] weighted_mape={baseline_results['weighted_mape']:.6f}, "
            f"score={baseline_results['competition_score']:.6f}"
        )
        print(baseline_results["fold_results"][["fold_name", "mape", "competition_score", "n_features"]].to_string(index=False))
        print()

    selection_records = []
    trial_records = []
    pruning_records_all = []
    candidate_importance_history = []

    no_improvement_streak = 0
    checked_candidates = 0
    ranking_round = 0
    last_candidate_importance = pd.DataFrame()
    stop_reason = None

    while remaining_candidates:
        if len(added_features) >= int(max_features_to_add):
            stop_reason = "max_features_to_add"
            break
        if no_improvement_streak >= int(patience_no_improvement):
            stop_reason = "patience_no_improvement"
            break
        if max_candidates_to_check is not None and checked_candidates >= int(max_candidates_to_check):
            stop_reason = "max_candidates_to_check"
            break

        ranking_round += 1

        if use_proxy_filter:
            candidate_importance = get_lgbm_proxy_feature_importance(
                fold_cache,
                fixed_features=current_features,
                candidate_features=remaining_candidates,
                lgbm_params=lgbm_params,
                importance_type=importance_type,
                max_proxy_features=max_proxy_features,
                verbose=False,
            )
        else:
            candidate_importance = pd.DataFrame(
                {
                    "feature": list(current_features) + list(remaining_candidates),
                    "importance_mean": 0.0,
                    "importance_std": 0.0,
                    "importance_by_fold": [{} for _ in range(len(current_features) + len(remaining_candidates))],
                    "is_fixed": [True] * len(current_features) + [False] * len(remaining_candidates),
                }
            )

        candidate_importance = add_stability_columns(
            candidate_importance,
            stability_std_penalty=stability_std_penalty,
            positive_fold_bonus=positive_fold_bonus,
        )
        candidate_importance["ranking_round"] = int(ranking_round)
        candidate_importance["n_fixed_features_at_ranking"] = int(len(current_features))

        candidate_rows = candidate_importance[candidate_importance["feature"].isin(remaining_candidates)].copy()
        if min_proxy_importance is not None:
            if min_proxy_importance_on not in candidate_rows.columns:
                raise ValueError(f"min_proxy_importance_on='{min_proxy_importance_on}' отсутствует.")
            candidate_rows[min_proxy_importance_on] = pd.to_numeric(candidate_rows[min_proxy_importance_on], errors="coerce")
            candidate_rows = candidate_rows[candidate_rows[min_proxy_importance_on] >= float(min_proxy_importance)].copy()

        if ranking_score_col not in candidate_rows.columns:
            raise ValueError(f"ranking_score_col='{ranking_score_col}' отсутствует в proxy importance.")
        candidate_rows = candidate_rows.sort_values(
            by=[ranking_score_col, "importance_mean"],
            ascending=[False, False],
            kind="mergesort",
        )
        if top_k_candidates is not None:
            candidate_rows = candidate_rows.head(int(top_k_candidates)).copy()
        ranked_features_round = candidate_rows["feature"].tolist()
        ranked_set = set(ranked_features_round)
        candidate_importance["selected_for_ranked_add"] = candidate_importance["feature"].isin(ranked_set)
        candidate_importance_history.append(candidate_importance.copy())
        last_candidate_importance = candidate_importance

        if verbose:
            print(f"[proxy-ranking:lgbm round {ranking_round}] current={len(current_features)}, remaining={len(remaining_candidates)}, ranked={len(ranked_features_round)}")
            if ranked_features_round:
                show_cols = [
                    "feature", "importance_mean", "importance_median", "importance_min",
                    "importance_std", "n_positive_folds", "positive_fold_share", "stability_score",
                ]
                print(candidate_rows[[c for c in show_cols if c in candidate_rows.columns]].head(20).to_string(index=False))
                print()

        if not ranked_features_round:
            stop_reason = "no_ranked_candidates"
            break

        added_this_round = 0
        checked_this_round = 0

        for rank_in_round, feature in enumerate(ranked_features_round, start=1):
            if feature not in remaining_candidates:
                continue
            if len(added_features) >= int(max_features_to_add):
                stop_reason = "max_features_to_add"
                break
            if no_improvement_streak >= int(patience_no_improvement):
                stop_reason = "patience_no_improvement"
                break
            if max_candidates_to_check is not None and checked_candidates >= int(max_candidates_to_check):
                stop_reason = "max_candidates_to_check"
                break

            checked_candidates += 1
            checked_this_round += 1
            prev_weighted_mape = float(current_results["weighted_mape"])
            trial_features = current_features + [feature]

            if verbose:
                print("-" * 100)
                print(
                    f"[ranked-add:lgbm] round={ranking_round}, rank={rank_in_round}, "
                    f"checked={checked_candidates}, feature={feature}"
                )
                print(
                    f"current_features={len(current_features)}, current_weighted_mape={prev_weighted_mape:.6f}, "
                    f"no_improvement_streak={no_improvement_streak}"
                )

            try:
                trial_results = evaluate_lgbm_feature_set(
                    fold_cache,
                    trial_features,
                    lgbm_params=lgbm_params,
                    target_mode=target_mode,
                    target_col=target_col,
                    clip_pred_lower=clip_pred_lower,
                    constant_strategy=constant_strategy,
                    return_models=False,
                    verbose=False,
                )
                eligible, fail_reasons, meta = _check_acceptance(
                    current_results,
                    trial_results,
                    min_improvement=min_improvement,
                    min_improved_folds=min_improved_folds,
                    max_allowed_fold_degradation=max_allowed_fold_degradation,
                    max_allowed_degradation_by_fold=max_allowed_degradation_by_fold,
                )
                action = "added" if eligible else "skipped"
                record = {
                    "ranking_round": ranking_round,
                    "rank_in_round": rank_in_round,
                    "checked_candidate_number": checked_candidates,
                    "trial_feature": feature,
                    "action": action,
                    "previous_weighted_mape": meta["previous_weighted_mape"],
                    "new_weighted_mape": meta["new_weighted_mape"],
                    "improvement": meta["improvement"],
                    "n_improved_folds": meta["n_improved_folds"],
                    "max_fold_degradation": meta["max_fold_degradation"],
                    "eligible": eligible,
                    "fail_reasons": ",".join(fail_reasons),
                    "features_count_before": len(current_features),
                    "features_count_after_trial": len(trial_features),
                    "no_improvement_streak_before": no_improvement_streak,
                    "target_mode": target_mode,
                }
                record.update(_extract_named_fold_values(trial_results, prefix="mape"))
                record.update(_extract_delta_values(current_results, trial_results, prefix="delta"))

                if eligible:
                    current_features.append(feature)
                    added_features.append(feature)
                    current_results = trial_results
                    remaining_candidates = [f for f in remaining_candidates if f != feature]
                    added_this_round += 1
                    no_improvement_streak = 0

                    hist_rec = {
                        "add_order": len(added_features),
                        "ranking_round": ranking_round,
                        "rank_in_round": rank_in_round,
                        "added_feature": feature,
                        "features_count": len(current_features),
                        "previous_weighted_mape": meta["previous_weighted_mape"],
                        "new_weighted_mape": meta["new_weighted_mape"],
                        "improvement": meta["improvement"],
                        "n_improved_folds": meta["n_improved_folds"],
                        "max_fold_degradation": meta["max_fold_degradation"],
                    }
                    for k, v in record.items():
                        if k.startswith("mape_") or k.startswith("delta_"):
                            hist_rec[k] = v
                    selection_records.append(hist_rec)

                    if verbose:
                        print(
                            f"[ranked-add:lgbm] ADDED: {feature}\n"
                            f"    weighted_mape: {meta['previous_weighted_mape']:.6f} -> {meta['new_weighted_mape']:.6f}\n"
                            f"    improvement: {meta['improvement']:.6f}\n"
                            f"    improved_folds: {meta['n_improved_folds']}\n"
                            f"    max_fold_degradation: {meta['max_fold_degradation']:.6f}"
                        )

                    if (
                        enable_backward_pruning
                        and backward_pruning_every_n_added is not None
                        and int(backward_pruning_every_n_added) > 0
                        and len(added_features) > 0
                        and len(added_features) % int(backward_pruning_every_n_added) == 0
                    ):
                        current_features, current_results, added_features, pruning_records = backward_prune_once_lgbm(
                            fold_cache=fold_cache,
                            current_features=current_features,
                            current_results=current_results,
                            added_features=added_features,
                            lgbm_params=lgbm_params,
                            target_mode=target_mode,
                            target_col=target_col,
                            clip_pred_lower=clip_pred_lower,
                            constant_strategy=constant_strategy,
                            protected_features=protected_encoded,
                            initial_features=initial_features,
                            pruning_min_improvement=backward_pruning_min_improvement,
                            pruning_allow_neutral=backward_pruning_allow_neutral,
                            pruning_neutral_tolerance=backward_pruning_neutral_tolerance,
                            pruning_max_allowed_fold_degradation=backward_pruning_max_allowed_fold_degradation,
                            pruning_order=backward_pruning_order,
                            prune_initial_features=backward_pruning_include_initial_features,
                            verbose=verbose,
                        )
                        for pr in pruning_records:
                            pr["trigger_add_order"] = len(added_features)
                            pr["ranking_round"] = ranking_round
                            pruning_records_all.append(pr)
                else:
                    remaining_candidates = [f for f in remaining_candidates if f != feature]
                    no_improvement_streak += 1
                    if verbose:
                        print(
                            f"[ranked-add:lgbm] SKIPPED: {feature}\n"
                            f"    weighted_mape: {meta['previous_weighted_mape']:.6f} -> {meta['new_weighted_mape']:.6f}\n"
                            f"    improvement: {meta['improvement']:.6f}\n"
                            f"    fail_reasons: {record['fail_reasons']}\n"
                            f"    no_improvement_streak: {no_improvement_streak}"
                        )

                record["no_improvement_streak_after"] = no_improvement_streak
                trial_records.append(record)

            except Exception as e:
                remaining_candidates = [f for f in remaining_candidates if f != feature]
                no_improvement_streak += 1
                record = {
                    "ranking_round": ranking_round,
                    "rank_in_round": rank_in_round,
                    "checked_candidate_number": checked_candidates,
                    "trial_feature": feature,
                    "action": "error",
                    "previous_weighted_mape": prev_weighted_mape,
                    "new_weighted_mape": np.nan,
                    "improvement": np.nan,
                    "n_improved_folds": 0,
                    "max_fold_degradation": np.nan,
                    "eligible": False,
                    "fail_reasons": f"error: {repr(e)}",
                    "features_count_before": len(current_features),
                    "features_count_after_trial": len(trial_features),
                    "no_improvement_streak_before": no_improvement_streak - 1,
                    "no_improvement_streak_after": no_improvement_streak,
                    "target_mode": target_mode,
                }
                trial_records.append(record)
                if verbose:
                    print(f"[ranked-add:lgbm] ERROR: {feature}; error={repr(e)}")

            if gc_after_each_trial:
                gc.collect()

            if not enable_periodic_reranking:
                continue

            need_rerank_by_added = (
                rerank_after_n_added is not None
                and int(rerank_after_n_added) > 0
                and added_this_round >= int(rerank_after_n_added)
            )
            need_rerank_by_checked = (
                rerank_after_n_checked is not None
                and int(rerank_after_n_checked) > 0
                and checked_this_round >= int(rerank_after_n_checked)
            )
            if need_rerank_by_added or need_rerank_by_checked:
                if verbose:
                    print(
                        f"[ranked-add:lgbm] trigger re-ranking: "
                        f"added_this_round={added_this_round}, checked_this_round={checked_this_round}"
                    )
                break

        if stop_reason is not None:
            break
        if checked_this_round == 0:
            stop_reason = "no_candidates_checked_in_round"
            break
        if not enable_periodic_reranking:
            stop_reason = "single_pass_finished"
            break

    if stop_reason is None:
        stop_reason = "candidate_pool_exhausted"

    if enable_backward_pruning and added_features:
        if verbose:
            print("[final-pruning:lgbm] running final backward pruning pass...")
        current_features, current_results, added_features, pruning_records = backward_prune_once_lgbm(
            fold_cache=fold_cache,
            current_features=current_features,
            current_results=current_results,
            added_features=added_features,
            lgbm_params=lgbm_params,
            target_mode=target_mode,
            target_col=target_col,
            clip_pred_lower=clip_pred_lower,
            constant_strategy=constant_strategy,
            protected_features=protected_encoded,
            initial_features=initial_features,
            pruning_min_improvement=backward_pruning_min_improvement,
            pruning_allow_neutral=backward_pruning_allow_neutral,
            pruning_neutral_tolerance=backward_pruning_neutral_tolerance,
            pruning_max_allowed_fold_degradation=backward_pruning_max_allowed_fold_degradation,
            pruning_order=backward_pruning_order,
            prune_initial_features=backward_pruning_include_initial_features,
            verbose=verbose,
        )
        for pr in pruning_records:
            pr["trigger_add_order"] = len(added_features)
            pr["ranking_round"] = ranking_round
            pr["is_final_pruning"] = True
            pruning_records_all.append(pr)

    selection_history = pd.DataFrame(selection_records)
    trial_history = pd.DataFrame(trial_records)
    pruning_history = pd.DataFrame(pruning_records_all)
    candidate_importance = (
        pd.concat(candidate_importance_history, ignore_index=True, sort=False)
        if candidate_importance_history else pd.DataFrame()
    )
    checked_set = set(trial_history["trial_feature"]) if not trial_history.empty else set()
    added_set = set(added_features)
    remaining_features = [
        f for f in remaining_candidates
        if f not in checked_set and f not in added_set and f not in set(current_features)
    ]

    fold_cache_meta = pd.DataFrame(
        [
            {
                "fold_name": fd.fold_name,
                "valid_year": fd.valid_year,
                "valid_month": fd.valid_month,
                "weight": fd.weight,
                "train_shape": fd.X_train.shape,
                "valid_shape": fd.X_valid.shape,
                "n_encoded_features": fd.X_train.shape[1],
                "n_encoded_categoricals": len(fd.encoding_info.get("encoded_features_created") or []),
            }
            for fd in fold_cache
        ]
    )

    selected_alias_rows = []
    raw_to_encoded = feature_map.get("raw_to_encoded") or {}
    for raw, enc in raw_to_encoded.items():
        if enc in current_features:
            selected_alias_rows.append({"raw_feature": raw, "encoded_feature": enc})
    selected_raw_feature_aliases = pd.DataFrame(selected_alias_rows).drop_duplicates() if selected_alias_rows else pd.DataFrame(columns=["raw_feature", "encoded_feature"])

    if verbose:
        print("=" * 100)
        print("[done:lgbm]")
        print(f"stop_reason:          {stop_reason}")
        print(f"ranking_rounds:       {ranking_round}")
        print(f"initial features:     {len(initial_features)}")
        print(f"checked candidates:   {checked_candidates}")
        print(f"added features:       {len(added_features)}")
        print(f"final features:       {len(current_features)}")
        print(f"remaining candidates: {len(remaining_features)}")
        print(f"weighted_mape:        {baseline_results['weighted_mape']:.6f} -> {current_results['weighted_mape']:.6f}")
        print(f"score:                {baseline_results['competition_score']:.6f} -> {current_results['competition_score']:.6f}")
        print("=" * 100)

    result = {
        "selected_features": current_features,
        "selected_raw_feature_aliases": selected_raw_feature_aliases,
        "initial_features": initial_features,
        "added_features": added_features,
        "remaining_features": remaining_features,
        "selection_history": selection_history,
        "trial_history": trial_history,
        "pruning_history": pruning_history,
        "candidate_importance": candidate_importance,
        "last_candidate_importance": last_candidate_importance,
        "baseline_results": baseline_results,
        "final_results": current_results,
        "fold_cache_meta": fold_cache_meta,
        "encoding_meta": encoding_meta,
        "feature_alias_table": alias_table,
        "feature_map": feature_map,
        "lgbm_params_used": lgbm_params,
        "target_mode": target_mode,
        "stop_reason": stop_reason,
        "ranking_rounds": ranking_round,
    }
    if return_fold_cache:
        result["fold_cache"] = fold_cache
    return result


# =============================================================================
# Convenience wrapper for current notebook
# =============================================================================


def run_x5_lgbm_ranked_add_feature_selection_v1(
    *,
    df: pd.DataFrame,
    feature_builder_func: Callable,
    matrix_preparer_func: Callable | None = None,
    vladimir_features: Sequence[str] | None = None,
    hard_protected_features: Sequence[str] | None = None,
    candidate_features: Sequence[str] | None = None,
    feature_builder_kwargs: dict | None = None,
    encoding_kwargs: dict | None = None,
    lgbm_params: dict | None = None,
    target_mode: Literal[
        "original",
        "growth_log",
        "day_scaled_residual_log",
        "calendar_residual_log",
        "daily_rate_log",
        "direct_log1p",
    ] = "direct_log1p",
    max_depth: int = 5,
    verbose: bool = True,
    return_fold_cache: bool = True,
    **selector_kwargs,
) -> dict:
    """
    Safe launch wrapper synchronized with the current X5/RTO pipeline.

    Defaults are intentionally close to the previous ranked-add workflow, but the
    estimator is LightGBM and all categoricals are handled through time-aware ordered TE.
    """
    if feature_builder_kwargs is None:
        feature_builder_kwargs = make_x5_lgbm_feature_selection_builder_kwargs()

    if encoding_kwargs is None:
        encoding_kwargs = {
            "smoothing": 30.0,
            "min_count": 2,
            "drop_original_cat_features": True,
            "drop_numeric_cat_source_cols": True,
            "include_all_detected_cat_features": True,
            "add_default_cat_features": True,
            "add_numeric_cat_copies": True,
            "numeric_cat_source_cols": make_x5_lgbm_numeric_cat_source_cols(),
            "keep_id_col": False,
            "keep_time_col": True,
            "strict_train_time_order": True,
            "first_block_value": "nan",
            "unknown_value": "global",
        }
    else:
        encoding_kwargs = dict(encoding_kwargs)
        encoding_kwargs.setdefault("smoothing", 30.0)
        encoding_kwargs.setdefault("min_count", 2)
        encoding_kwargs.setdefault("drop_original_cat_features", True)
        encoding_kwargs.setdefault("drop_numeric_cat_source_cols", True)
        encoding_kwargs.setdefault("include_all_detected_cat_features", True)
        encoding_kwargs.setdefault("add_default_cat_features", True)
        encoding_kwargs.setdefault("add_numeric_cat_copies", True)
        encoding_kwargs.setdefault("numeric_cat_source_cols", make_x5_lgbm_numeric_cat_source_cols())
        encoding_kwargs.setdefault("keep_id_col", False)
        encoding_kwargs.setdefault("keep_time_col", True)
        encoding_kwargs.setdefault("strict_train_time_order", True)
        encoding_kwargs.setdefault("first_block_value", "nan")
        encoding_kwargs.setdefault("unknown_value", "global")

    if lgbm_params is None:
        lgbm_params = make_default_lgbm_params(max_depth=max_depth)
    else:
        lgbm_params = dict(lgbm_params)
        lgbm_params.setdefault("max_depth", max_depth)
        lgbm_params = normalize_lgbm_params(lgbm_params)

    call_kwargs = dict(
        df=df,
        feature_builder_func=feature_builder_func,
        matrix_preparer_func=matrix_preparer_func,
        base_features=_unique_keep_order(vladimir_features),
        protected_features=_unique_keep_order(hard_protected_features),
        candidate_features=candidate_features,
        feature_builder_kwargs=feature_builder_kwargs,
        encoding_kwargs=encoding_kwargs,
        target_col="pto",
        id_col="new_id",
        year_col="year",
        month_col="month",
        time_col="year_month_index",
        min_year=2023,
        target_mode=target_mode,
        folds=None,
        normalize_fold_weights=True,
        lgbm_params=lgbm_params,
        random_state=int(lgbm_params.get("random_state", 42)),
        start_from_empty=False,
        force_include_protected=False,
        use_proxy_filter=True,
        top_k_candidates=150,
        max_proxy_features=None,
        importance_type="gain",
        ranking_score_col="stability_score",
        stability_std_penalty=0.50,
        positive_fold_bonus=0.25,
        min_proxy_importance=None,
        enable_periodic_reranking=True,
        rerank_after_n_added=5,
        rerank_after_n_checked=None,
        min_improvement=0.005,
        min_improved_folds=2,
        max_features_to_add=40,
        max_candidates_to_check=None,
        patience_no_improvement=25,
        max_allowed_fold_degradation=0.05,
        enable_backward_pruning=True,
        backward_pruning_every_n_added=5,
        backward_pruning_min_improvement=0.0,
        backward_pruning_allow_neutral=True,
        backward_pruning_neutral_tolerance=0.002,
        backward_pruning_max_allowed_fold_degradation=0.03,
        backward_pruning_order="reverse",
        backward_pruning_include_initial_features=False,
        clip_pred_lower=0.0,
        constant_strategy="median",
        verbose=verbose,
        return_fold_cache=return_fold_cache,
    )
    call_kwargs.update(selector_kwargs)
    return ranked_add_features_with_time_aware_lgbm_v1(**call_kwargs)


# Backward-compatible aliases for experiments.
run_x5_lgbm_ranked_add_feature_selection = run_x5_lgbm_ranked_add_feature_selection_v1
ranked_add_features_with_time_aware_lgbm = ranked_add_features_with_time_aware_lgbm_v1

In [10]:
# # ============================================================
# # 0. Проверка, что все нужные функции уже выполнены в ноутбуке
# # ============================================================

# required_funcs = [
#     "build_retail_features_with_split",
#     "prepare_lgbm_matrices_with_time_aware_catboost_encoding",
#     "make_forecast_target",
#     "inverse_forecast_target",
#     "run_x5_lgbm_ranked_add_feature_selection_v1",
#     "make_default_lgbm_params",
# ]

# missing = [f for f in required_funcs if f not in globals()]
# print("Missing functions:", missing)

# if missing:
#     raise RuntimeError(f"Сначала выполни ячейки с функциями: {missing}")

# # ============================================================
# # 1. Явно фиксируем списки фичей
# # ============================================================

# base_features_for_lgbm_fs = globals().get("vladimir_features", [])
# protected_features_for_lgbm_fs = globals().get("hard_protected_features", [])
# candidate_features_for_lgbm_fs = globals().get("candidate_features", None)

# print("base_features:", len(base_features_for_lgbm_fs))
# print("protected_features:", len(protected_features_for_lgbm_fs))
# print(
#     "candidate_features:",
#     None if candidate_features_for_lgbm_fs is None else len(candidate_features_for_lgbm_fs)
# )



# # ============================================================
# # 2. LightGBM params: depth около 5
# # ============================================================

# lgbm_params = make_default_lgbm_params(
#     max_depth=5,
#     n_estimators=700,
#     learning_rate=0.05,
# )

# # Если хочешь GPU, раскомментируй этот блок.
# # Если LightGBM в среде собран без GPU, будет ошибка — тогда просто закомментируй обратно.
# lgbm_params.update({
#     "device_type": "gpu",
#     "gpu_use_dp": False,
#     "max_bin": 63,
#     "verbosity": -1,
# })

# lgbm_params

# # ============================================================
# # 3. Корректный запуск отбора
# # ============================================================

# result_lgbm_fs = run_x5_lgbm_ranked_add_feature_selection_v1(
#     df=df,

#     # Твой feature-pipeline
#     feature_builder_func=build_retail_features_with_split,

#     # Подготовка LGBM-матриц с time-aware ordered target encoding категорий
#     matrix_preparer_func=prepare_lgbm_matrices_with_time_aware_catboost_encoding,

#     # Стартовые / protected / candidate фичи
#     vladimir_features=base_features_for_lgbm_fs,
#     hard_protected_features=protected_features_for_lgbm_fs,
#     candidate_features=candidate_features_for_lgbm_fs,

#     # LightGBM
#     lgbm_params=lgbm_params,
#     max_depth=5,

#     # Forward selection
#     top_k_candidates=30,
#     max_features_to_add=50,
#     patience_no_improvement=400,

#     # Adaptive reranking
#     enable_periodic_reranking=True,
#     rerank_after_n_added=5,
#     rerank_after_n_checked=None,

#     # Условия принятия фичи
#     min_improvement=0.0001,
#     min_improved_folds=2,
#     max_allowed_fold_degradation=2.0,

#     # Backward pruning
#     enable_backward_pruning=False,
#     backward_pruning_every_n_added=5,
#     backward_pruning_include_initial_features=False,
#     backward_pruning_min_improvement=0.0,
#     backward_pruning_allow_neutral=True,
#     backward_pruning_neutral_tolerance=0.002,
#     backward_pruning_max_allowed_fold_degradation=0.05,

#     verbose=True,
#     return_fold_cache=True,
# )

# selected_lgbm_features = result_lgbm_fs["selected_features"]

# print("Stop reason:", result_lgbm_fs["stop_reason"])
# print("Selected features:", len(selected_lgbm_features))
# print("Added features:", len(result_lgbm_fs["added_features"]))
# print("Final weighted MAPE:", result_lgbm_fs["final_results"]["weighted_mape"])

# selected_lgbm_features[:50]

In [11]:
selected_features_light = ['daily_pto_lag_1',
 'log_day_scaled_lag1_prediction',
 'day_scaled_lag1_prediction',
 'is_year_start_month',
 'day_scaled_x_region_share_change',
 'month_cos',
 'recent_daily_mean_2',
 'blend_day_scaled_and_recent_mean_prediction',
 'trend_day_scaled_lag1_prediction',
 'month_sin',
 'log_blend_day_scaled_and_recent_mean_prediction',
 'pto_roll_mean_12',
 'settlement_number_of_cash_registers_mean',
 'number_of_households',
 'region_number_of_cash_registers_mean',
 'avg_items_per_receipt_lag_3',
 'pto_lag_1_to_region_pto_mean_lag_1',
 'trend_day_scaled_lag1_prediction_smooth',
 'log_pto_lag_1',
 'n_working_days',
 'store_share_region_change_1_2',
 'pto_roll_cv_12',
 'recent_daily_median_3',
 'households_per_cash_register',
 'day_scaled_lag2_prediction',
 'log_population_size',
 'log_daily_pto_diff_lag_1_3',
 'days_lag_2',
 'daily_growth_1_3_clipped',
 'daily_growth_1_3',
 'month_sin_2',
 'competition_per_1000_households',
 'store_share_settlement_lag2',
 'area_category_pto_lag1_mean']

# Обучение модели с отобранными признаками

In [12]:
# # ============================================================
# # Fixed-feature LightGBM quality probe with early stopping
# # ============================================================

# from __future__ import annotations

# import gc
# import time
# import warnings
# from typing import Optional, Sequence, Literal

# import numpy as np
# import pandas as pd


# # ============================================================
# # 0. Encoding kwargs для текущего X5 LightGBM pipeline
# # ============================================================

# def make_lgbm_fixed_probe_encoding_kwargs() -> dict:
#     """
#     Encoding kwargs синхронизированы с текущим LightGBM feature-selection pipeline.

#     Важно:
#     - time-aware CatBoost encoding;
#     - strict_train_time_order=True;
#     - drop_original_cat_features=True;
#     - keep_time_col=True, потому что он нужен для ordered encoding / target transform.
#     """
#     return {
#         "smoothing": 30.0,
#         "min_count": 2,

#         "drop_original_cat_features": True,
#         "drop_numeric_cat_source_cols": True,

#         "include_all_detected_cat_features": True,
#         "add_default_cat_features": True,
#         "add_numeric_cat_copies": True,
#         "numeric_cat_source_cols": make_x5_lgbm_numeric_cat_source_cols(),

#         "keep_id_col": False,
#         "keep_time_col": True,

#         "strict_train_time_order": True,
#         "first_block_value": "nan",
#         "unknown_value": "global",
#     }


# # ============================================================
# # 1. Качественные параметры LightGBM
# # ============================================================

# def make_lgbm_fixed_probe_quality_params(
#     *,
#     random_state: int = 42,
#     learning_rate: float = 0.01,
#     n_estimators: int = 12_000,
#     max_depth: int = 5,
#     num_leaves: Optional[int] = None,
#     use_gpu: bool = True,
# ) -> dict:
#     """
#     Параметры для качественного fixed-feature обучения.

#     Логика:
#     - lr=0.01;
#     - много деревьев;
#     - реальное число деревьев выбирает early stopping;
#     - objective='regression_l1' => обучение на MAE;
#     - metric='mae' => early stopping тоже по MAE в model-space.
#     """
#     if num_leaves is None:
#         num_leaves = min(31, 2 ** int(max_depth) - 1)

#     params = make_default_lgbm_params(
#         random_state=random_state,
#         n_estimators=n_estimators,
#         learning_rate=learning_rate,
#         max_depth=max_depth,
#         num_leaves=num_leaves,
#         objective="regression_l1",
#         metric="mae",
#         n_jobs=-1,
#     )

#     params.update({
#         # tree complexity
#         "num_leaves": int(num_leaves),
#         "max_depth": int(max_depth),
#         "min_child_samples": 80,
#         "min_child_weight": 1e-3,

#         # stochasticity
#         "subsample": 0.85,
#         "subsample_freq": 1,
#         "colsample_bytree": 0.85,

#         # regularization
#         "reg_alpha": 0.05,
#         "reg_lambda": 8.0,

#         # stable output
#         "verbosity": -1,
#     })

#     if use_gpu:
#         params.update({
#             "device_type": "gpu",
#             "gpu_use_dp": False,
#             # Для GPU LightGBM обычно лучше небольшой max_bin.
#             "max_bin": 63,
#         })
#     else:
#         params.pop("device_type", None)
#         params.pop("gpu_use_dp", None)

#     # Иногда force_col_wise конфликтует с GPU.
#     params.pop("force_col_wise", None)

#     params = normalize_lgbm_params(params, random_state=random_state)

#     return params


# # ============================================================
# # 2. Resolve selected features + report
# # ============================================================

# def resolve_lgbm_selected_features_with_report(
#     selected_features: Sequence[str],
#     *,
#     feature_map: dict,
#     available_features: Sequence[str],
#     strict: bool = True,
# ) -> tuple[list[str], list[str], pd.DataFrame]:
#     """
#     Сопоставляет selected_features с реальными колонками LGBM-матрицы.

#     Например:
#     - числовые признаки остаются как есть;
#     - категориальные raw-признаки могут стать *_cb;
#     - если признак уже есть в матрице, он берётся напрямую.
#     """
#     selected_features = _unique_keep_order(selected_features)

#     available = set(available_features)
#     raw_to_encoded = feature_map.get("raw_to_encoded") or {}

#     resolved = []
#     missing = []
#     rows = []

#     for f in selected_features:
#         candidates = []

#         if f in available:
#             candidates.append(f)

#         mapped = raw_to_encoded.get(f)
#         if mapped is not None:
#             candidates.append(mapped)

#         if f.endswith("_cat"):
#             mapped2 = raw_to_encoded.get(f[:-4])
#             if mapped2 is not None:
#                 candidates.append(mapped2)
#         else:
#             mapped3 = raw_to_encoded.get(f"{f}_cat")
#             if mapped3 is not None:
#                 candidates.append(mapped3)

#         if f.endswith("_cb") and f in available:
#             candidates.append(f)

#         chosen = None
#         for c in candidates:
#             if c in available:
#                 chosen = c
#                 break

#         if chosen is None:
#             missing.append(f)
#             rows.append({
#                 "raw_feature": f,
#                 "resolved_feature": None,
#                 "status": "missing",
#             })
#         else:
#             if chosen not in resolved:
#                 resolved.append(chosen)

#             rows.append({
#                 "raw_feature": f,
#                 "resolved_feature": chosen,
#                 "status": "ok",
#             })

#     report = pd.DataFrame(rows)

#     if missing and strict:
#         raise RuntimeError(
#             "В selected_features есть признаки, которых нет после feature engineering / encoding:\n"
#             + "\n".join(missing[:80])
#             + (f"\n... и ещё {len(missing) - 80}" if len(missing) > 80 else "")
#         )

#     if missing and not strict:
#         warnings.warn(
#             "Часть selected_features не найдена и будет пропущена:\n"
#             + "\n".join(missing[:50])
#             + (f"\n... и ещё {len(missing) - 50}" if len(missing) > 50 else "")
#         )

#     return resolved, missing, report


# # ============================================================
# # 3. Fit LightGBM with early stopping
# # ============================================================

# def fit_lgbm_with_early_stopping(
#     X_train: pd.DataFrame,
#     y_train,
#     X_valid: pd.DataFrame,
#     y_valid,
#     *,
#     params: dict,
#     early_stopping_rounds: int = 800,
#     log_evaluation_period: int = 300,
# ):
#     """
#     Обучает LGBMRegressor с early stopping.

#     Early stopping идёт по model-target:
#     - при target_mode='direct_log1p' это MAE по log1p(pto);
#     - финальная MAPE ниже считается уже после inverse transform в исходный РТО.
#     """
#     import lightgbm as lgb

#     params = dict(params)
#     model = _make_lgbm_model(params)

#     callbacks = [
#         lgb.early_stopping(
#             stopping_rounds=int(early_stopping_rounds),
#             first_metric_only=True,
#             verbose=False,
#         )
#     ]

#     if log_evaluation_period is not None and int(log_evaluation_period) > 0:
#         callbacks.append(
#             lgb.log_evaluation(period=int(log_evaluation_period))
#         )

#     model.fit(
#         X_train,
#         y_train,
#         eval_set=[(X_valid, y_valid)],
#         eval_metric=params.get("metric", "mae"),
#         callbacks=callbacks,
#     )

#     return model


# # ============================================================
# # 4. Evaluate fixed feature set with early stopping
# # ============================================================

# def evaluate_lgbm_feature_set_with_early_stopping(
#     fold_cache: Sequence,
#     features: Sequence[str],
#     *,
#     lgbm_params: dict,
#     target_mode: str = "direct_log1p",
#     target_col: str = "pto",
#     clip_pred_lower: Optional[float] = 0.0,
#     constant_strategy: Literal["median", "mean"] = "median",
#     return_models: bool = True,
#     early_stopping_rounds: int = 800,
#     log_evaluation_period: int = 300,
#     verbose: bool = True,
# ) -> dict:
#     """
#     Полный evaluator для fixed-feature LightGBM с early stopping.
#     """
#     features = _unique_keep_order(features)

#     records = []
#     models = {}
#     predictions = {}

#     for fd in fold_cache:
#         available = set(fd.X_train.columns).intersection(fd.X_valid.columns)
#         fold_features = [f for f in features if f in available]

#         if verbose:
#             print("-" * 100)
#             print(
#                 f"[fit:lgbm-es] {fd.fold_name}: "
#                 f"valid={fd.valid_year}-{fd.valid_month:02d}, "
#                 f"n_features={len(fold_features)}"
#             )

#         if not fold_features:
#             pred_raw = _predict_constant_baseline(
#                 fd.y_train,
#                 fd.X_valid,
#                 target_mode=target_mode,
#                 target_col=target_col,
#                 constant_strategy=constant_strategy,
#                 clip_pred_lower=clip_pred_lower,
#             )
#             model = None
#             best_iteration = None

#         else:
#             X_tr = fd.X_train[fold_features].copy()
#             X_va = fd.X_valid[fold_features].copy()

#             if fd.y_valid is None:
#                 raise ValueError(
#                     f"Для fold={fd.fold_name} нет y_valid. "
#                     "Early stopping требует validation target."
#                 )

#             model = fit_lgbm_with_early_stopping(
#                 X_train=X_tr,
#                 y_train=fd.y_train,
#                 X_valid=X_va,
#                 y_valid=fd.y_valid,
#                 params=lgbm_params,
#                 early_stopping_rounds=early_stopping_rounds,
#                 log_evaluation_period=log_evaluation_period,
#             )

#             best_iteration = getattr(model, "best_iteration_", None)

#             if best_iteration is not None and int(best_iteration) > 0:
#                 pred_model = model.predict(X_va, num_iteration=int(best_iteration))
#             else:
#                 pred_model = model.predict(X_va)

#             pred_raw = inverse_forecast_target_for_lgbm(
#                 pred_model,
#                 X_va,
#                 mode=target_mode,
#                 target_col=target_col,
#                 clip_min=clip_pred_lower,
#             )

#         mape = mape_score(
#             fd.y_valid_raw,
#             pred_raw,
#             clip_pred_lower=clip_pred_lower,
#         )

#         rec = {
#             "fold_name": fd.fold_name,
#             "valid_year": fd.valid_year,
#             "valid_month": fd.valid_month,
#             "weight": fd.weight,
#             "mape": float(mape),
#             "competition_score": float(competition_score_from_mape(mape)),
#             "n_train": int(len(fd.X_train)),
#             "n_valid": int(len(fd.X_valid)),
#             "n_features": int(len(fold_features)),
#             "best_iteration": None if best_iteration is None else int(best_iteration),
#             "target_mode": target_mode,
#         }
#         records.append(rec)

#         predictions[fd.fold_name] = pd.DataFrame({
#             "y_true": fd.y_valid_raw,
#             "y_pred": pred_raw,
#             "abs_pct_error": np.abs(pred_raw - fd.y_valid_raw) / np.maximum(np.abs(fd.y_valid_raw), 1e-9) * 100,
#         })

#         if return_models and model is not None:
#             models[fd.fold_name] = model

#         if verbose:
#             print(
#                 f"[done:lgbm-es] {fd.fold_name}: "
#                 f"MAPE={rec['mape']:.6f}, "
#                 f"score={rec['competition_score']:.6f}, "
#                 f"best_iter={rec['best_iteration']}"
#             )

#         gc.collect()

#     fold_results = pd.DataFrame(records)
#     weighted = compute_weighted_score(fold_results)

#     out = {
#         "weighted_mape": float(weighted["weighted_mape"]),
#         "competition_score": float(weighted["competition_score"]),
#         "fold_results": fold_results,
#         "predictions": predictions,
#     }

#     if return_models:
#         out["models"] = models

#     return out


# # ============================================================
# # 5. Importance collection
# # ============================================================

# def collect_lgbm_fold_importance(
#     models: dict,
#     *,
#     importance_type: Literal["gain", "split"] = "gain",
# ) -> pd.DataFrame:
#     """
#     Собирает importance по всем fold-моделям.
#     """
#     if not models:
#         return pd.DataFrame()

#     frames = []

#     for fold_name, model in models.items():
#         booster = model.booster_
#         feature_names = list(booster.feature_name())
#         importance = booster.feature_importance(importance_type=importance_type)

#         frames.append(
#             pd.DataFrame({
#                 "feature": feature_names,
#                 f"importance_{fold_name}": importance.astype(float),
#             })
#         )

#     out = frames[0]
#     for part in frames[1:]:
#         out = out.merge(part, on="feature", how="outer")

#     imp_cols = [c for c in out.columns if c.startswith("importance_")]
#     out[imp_cols] = out[imp_cols].fillna(0.0)

#     out["importance_mean"] = out[imp_cols].mean(axis=1)
#     out["importance_median"] = out[imp_cols].median(axis=1)
#     out["importance_std"] = out[imp_cols].std(axis=1)
#     out["importance_min"] = out[imp_cols].min(axis=1)
#     out["n_positive_folds"] = (out[imp_cols] > 0).sum(axis=1)
#     out["positive_fold_share"] = out["n_positive_folds"] / max(1, len(imp_cols))

#     out = out.sort_values(
#         ["importance_mean", "positive_fold_share"],
#         ascending=[False, False],
#     ).reset_index(drop=True)

#     return out


# # ============================================================
# # 6. Main wrapper: fixed-feature quality probe
# # ============================================================

# def run_x5_lgbm_fixed_feature_quality_probe(
#     *,
#     df: pd.DataFrame,
#     selected_features: Sequence[str],

#     feature_builder_func=None,
#     matrix_preparer_func=None,

#     feature_builder_kwargs: Optional[dict] = None,
#     encoding_kwargs: Optional[dict] = None,
#     lgbm_params: Optional[dict] = None,

#     target_mode: Literal[
#         "original",
#         "growth_log",
#         "day_scaled_residual_log",
#         "calendar_residual_log",
#         "daily_rate_log",
#         "direct_log1p",
#     ] = "direct_log1p",

#     folds: Optional[Sequence[dict]] = None,
#     normalize_weights: bool = True,

#     random_state: int = 42,

#     # quality-training params
#     learning_rate: float = 0.01,
#     n_estimators: int = 12_000,
#     max_depth: int = 5,
#     num_leaves: Optional[int] = None,
#     early_stopping_rounds: int = 800,
#     log_evaluation_period: int = 300,

#     use_gpu: bool = True,
#     fallback_to_cpu: bool = True,

#     strict_features: bool = True,
#     return_models: bool = True,
#     verbose: bool = True,
# ) -> dict:
#     """
#     Полная функция пробного обучения LightGBM на фиксированном списке признаков.

#     Использует текущий notebook pipeline:
#     - build_retail_features_with_split;
#     - prepare_lgbm_matrices_with_time_aware_catboost_encoding;
#     - build_lgbm_encoded_fold_cache;
#     - inverse_forecast_target_for_lgbm;
#     - temporal folds для мартовской задачи.
#     """

#     required_funcs = [
#         "build_retail_features_with_split",
#         "prepare_lgbm_matrices_with_time_aware_catboost_encoding",
#         "build_lgbm_encoded_fold_cache",
#         "make_march_2025_feature_selection_folds",
#         "normalize_fold_weights",
#         "make_x5_lgbm_feature_selection_builder_kwargs",
#         "make_x5_lgbm_numeric_cat_source_cols",
#         "make_default_lgbm_params",
#         "normalize_lgbm_params",
#         "_make_lgbm_model",
#         "_predict_constant_baseline",
#         "inverse_forecast_target_for_lgbm",
#         "mape_score",
#         "competition_score_from_mape",
#         "compute_weighted_score",
#         "_unique_keep_order",
#     ]

#     missing_funcs = [name for name in required_funcs if name not in globals()]
#     if missing_funcs:
#         raise RuntimeError(
#             "Сначала выполни ячейки с основными функциями LightGBM pipeline. "
#             "Не найдены функции:\n"
#             + "\n".join(missing_funcs)
#         )

#     if feature_builder_func is None:
#         feature_builder_func = build_retail_features_with_split

#     if matrix_preparer_func is None:
#         matrix_preparer_func = prepare_lgbm_matrices_with_time_aware_catboost_encoding

#     selected_features = _unique_keep_order(selected_features)

#     if not selected_features:
#         raise ValueError("selected_features пустой.")

#     if feature_builder_kwargs is None:
#         feature_builder_kwargs = make_x5_lgbm_feature_selection_builder_kwargs()
#     else:
#         feature_builder_kwargs = dict(feature_builder_kwargs)

#     if encoding_kwargs is None:
#         encoding_kwargs = make_lgbm_fixed_probe_encoding_kwargs()
#     else:
#         default_encoding_kwargs = make_lgbm_fixed_probe_encoding_kwargs()
#         default_encoding_kwargs.update(dict(encoding_kwargs))
#         encoding_kwargs = default_encoding_kwargs

#     if folds is None:
#         folds = make_march_2025_feature_selection_folds(normalize=False)
#     else:
#         folds = [dict(f) for f in folds]

#     if normalize_weights:
#         folds = normalize_fold_weights(folds)

#     if lgbm_params is None:
#         lgbm_params = make_lgbm_fixed_probe_quality_params(
#             random_state=random_state,
#             learning_rate=learning_rate,
#             n_estimators=n_estimators,
#             max_depth=max_depth,
#             num_leaves=num_leaves,
#             use_gpu=use_gpu,
#         )
#     else:
#         lgbm_params = normalize_lgbm_params(dict(lgbm_params), random_state=random_state)

#     if verbose:
#         print("=" * 100)
#         print("X5 LIGHTGBM FIXED-FEATURE QUALITY PROBE")
#         print("=" * 100)
#         print(f"raw selected features:   {len(selected_features)}")
#         print(f"target_mode:             {target_mode}")
#         print(f"objective:               {lgbm_params.get('objective')}")
#         print(f"metric:                  {lgbm_params.get('metric')}")
#         print(f"learning_rate:           {lgbm_params.get('learning_rate')}")
#         print(f"n_estimators:            {lgbm_params.get('n_estimators')}")
#         print(f"early_stopping_rounds:   {early_stopping_rounds}")
#         print(f"max_depth:               {lgbm_params.get('max_depth')}")
#         print(f"num_leaves:              {lgbm_params.get('num_leaves')}")
#         print(f"min_child_samples:       {lgbm_params.get('min_child_samples')}")
#         print(f"subsample:               {lgbm_params.get('subsample')}")
#         print(f"colsample_bytree:        {lgbm_params.get('colsample_bytree')}")
#         print(f"reg_alpha:               {lgbm_params.get('reg_alpha')}")
#         print(f"reg_lambda:              {lgbm_params.get('reg_lambda')}")
#         print(f"device_type:             {lgbm_params.get('device_type', 'cpu')}")
#         print()
#         print("[folds]")
#         print(
#             pd.DataFrame(folds)[
#                 ["fold_name", "valid_year", "valid_month", "weight"]
#             ].to_string(index=False)
#         )
#         print()

#     t0 = time.time()

#     # --------------------------------------------------------
#     # 1. Build encoded fold cache
#     # --------------------------------------------------------
#     fold_cache, feature_map, encoding_meta = build_lgbm_encoded_fold_cache(
#         df=df,
#         folds=folds,
#         feature_builder_func=feature_builder_func,
#         matrix_preparer_func=matrix_preparer_func,
#         feature_builder_kwargs=feature_builder_kwargs,

#         # Важно: строим матрицы только под выбранные признаки,
#         # а не под все 300+ кандидатов.
#         raw_features_to_use=selected_features,

#         encoding_kwargs=encoding_kwargs,
#         target_mode=target_mode,
#         target_col="pto",
#         id_col="new_id",
#         year_col="year",
#         month_col="month",
#         time_col="year_month_index",
#         min_year=2023,
#         forbidden_features=("pto", "__part__", "__row_order__"),
#         verbose=verbose,
#     )

#     available_features = feature_map.get("all_encoded_features") or list(fold_cache[0].X_train.columns)

#     selected_encoded, missing_selected, feature_resolution = resolve_lgbm_selected_features_with_report(
#         selected_features,
#         feature_map=feature_map,
#         available_features=available_features,
#         strict=strict_features,
#     )

#     if verbose:
#         print()
#         print("[feature resolution]")
#         print(f"raw selected:      {len(selected_features)}")
#         print(f"encoded selected:  {len(selected_encoded)}")
#         print(f"missing selected:  {len(missing_selected)}")
#         if missing_selected:
#             print("missing selected:")
#             for f in missing_selected[:50]:
#                 print(f"  - {f}")
#         print()

#     # --------------------------------------------------------
#     # 2. Fit / evaluate with GPU, fallback CPU if needed
#     # --------------------------------------------------------
#     fallback_used = False
#     used_lgbm_params = dict(lgbm_params)

#     try:
#         eval_result = evaluate_lgbm_feature_set_with_early_stopping(
#             fold_cache=fold_cache,
#             features=selected_encoded,
#             lgbm_params=used_lgbm_params,
#             target_mode=target_mode,
#             target_col="pto",
#             clip_pred_lower=0.0,
#             constant_strategy="median",
#             return_models=return_models,
#             early_stopping_rounds=early_stopping_rounds,
#             log_evaluation_period=log_evaluation_period,
#             verbose=verbose,
#         )

#     except Exception as e:
#         if not (use_gpu and fallback_to_cpu):
#             raise

#         print()
#         print("=" * 100)
#         print("[GPU FAILED] LightGBM GPU не запустился. Перезапускаю обучение на CPU.")
#         print("Original error:", repr(e))
#         print("=" * 100)
#         print()

#         cpu_params = dict(lgbm_params)
#         cpu_params.pop("device_type", None)
#         cpu_params.pop("gpu_use_dp", None)

#         # max_bin=63 можно оставить, но для CPU чаще лучше не ограничивать слишком жёстко.
#         # Если хочешь максимально сопоставимо с GPU, закомментируй следующую строку.
#         cpu_params.pop("max_bin", None)

#         used_lgbm_params = cpu_params
#         fallback_used = True

#         eval_result = evaluate_lgbm_feature_set_with_early_stopping(
#             fold_cache=fold_cache,
#             features=selected_encoded,
#             lgbm_params=used_lgbm_params,
#             target_mode=target_mode,
#             target_col="pto",
#             clip_pred_lower=0.0,
#             constant_strategy="median",
#             return_models=return_models,
#             early_stopping_rounds=early_stopping_rounds,
#             log_evaluation_period=log_evaluation_period,
#             verbose=verbose,
#         )

#     # --------------------------------------------------------
#     # 3. Importance
#     # --------------------------------------------------------
#     models = eval_result.get("models", {}) if return_models else {}

#     importance_gain = collect_lgbm_fold_importance(
#         models,
#         importance_type="gain",
#     ) if models else pd.DataFrame()

#     importance_split = collect_lgbm_fold_importance(
#         models,
#         importance_type="split",
#     ) if models else pd.DataFrame()

#     elapsed_sec = time.time() - t0

#     if verbose:
#         print()
#         print("=" * 100)
#         print("FINAL FIXED-FEATURE LIGHTGBM RESULT")
#         print("=" * 100)
#         print(f"weighted_mape:       {eval_result['weighted_mape']:.6f}")
#         print(f"competition_score:   {eval_result['competition_score']:.6f}")
#         print(f"fallback_used:       {fallback_used}")
#         print(f"elapsed_sec:         {elapsed_sec:.1f}")
#         print()
#         print("[fold results]")
#         print(eval_result["fold_results"].to_string(index=False))

#         if not importance_gain.empty:
#             print()
#             print("[top gain importance]")
#             display(importance_gain.head(40))

#     gc.collect()

#     return {
#         "selected_raw_features": selected_features,
#         "selected_encoded_features": selected_encoded,
#         "missing_selected_features": missing_selected,
#         "feature_resolution": feature_resolution,

#         "weighted_mape": eval_result["weighted_mape"],
#         "competition_score": eval_result["competition_score"],
#         "fold_results": eval_result["fold_results"],
#         "predictions": eval_result.get("predictions", {}),

#         "models": models,
#         "importance_gain": importance_gain,
#         "importance_split": importance_split,

#         "fold_cache": fold_cache,
#         "feature_map": feature_map,
#         "encoding_meta": encoding_meta,

#         "lgbm_params": used_lgbm_params,
#         "fallback_used": fallback_used,
#         "target_mode": target_mode,
#         "elapsed_sec": elapsed_sec,
#     }

In [13]:
# # ============================================================
# # Запуск качественного пробного обучения LightGBM
# # selected_features_light + lr=0.01 + много деревьев + early stopping
# # ============================================================

# probe_lgbm_light_quality = run_x5_lgbm_fixed_feature_quality_probe(
#     df=df,
#     selected_features=selected_features_light,

#     target_mode="direct_log1p",

#     learning_rate=0.01,
#     n_estimators=12_000,
#     max_depth=6,
#     num_leaves=31,

#     early_stopping_rounds=50,
#     log_evaluation_period=100,

#     random_state=42,

#     use_gpu=True,
#     fallback_to_cpu=True,

#     strict_features=True,
#     return_models=True,
#     verbose=True,
# )

In [14]:
# # ============================================================
# # FINAL SUBMIT PIPELINE: LightGBM -> март 2025 -> test.csv
# # ============================================================

# from pathlib import Path
# import gc
# import numpy as np
# import pandas as pd


# # ============================================================
# # 0. Берём n_estimators из CV early stopping
# # ============================================================

# best_iters = (
#     probe_lgbm_light_quality["fold_results"]["best_iteration"]
#     .dropna()
#     .astype(int)
# )

# if len(best_iters) == 0:
#     raise RuntimeError("Нет best_iteration в probe_lgbm_light_quality['fold_results'].")

# final_n_estimators = int(np.ceil(best_iters.median() * 1.05))

# print("=" * 100)
# print("FINAL N_ESTIMATORS FROM CV")
# print("=" * 100)
# print("best_iterations:", best_iters.tolist())
# print("median:", best_iters.median())
# print("final_n_estimators:", final_n_estimators)
# print()


# # ============================================================
# # 1. Настройки сабмита
# # ============================================================

# TEST_YEAR = 2025
# TEST_MONTH = 3
# EXPECTED_TEST_SIZE = 18_657
# OUTPUT_PATH = "test.csv"
# RANDOM_STATE = 42

# TARGET_MODE = "direct_log1p"


# # ============================================================
# # 2. Строим train/test признаки на март 2025
# # ============================================================

# builder_kwargs = make_x5_lgbm_feature_selection_builder_kwargs()

# built_submit = build_retail_features_with_split(
#     df,
#     split_mode="train_test",

#     test_year=TEST_YEAR,
#     test_month=TEST_MONTH,
#     create_test_if_missing=True,
#     expected_test_size=EXPECTED_TEST_SIZE,

#     id_col="new_id",
#     year_col="year",
#     month_col="month",
#     time_col="year_month_index",
#     target_col="pto",
#     min_year=2023,

#     verbose=True,
#     **builder_kwargs,
# )


# # ============================================================
# # 3. Готовим LightGBM matrices
# # ============================================================

# matrices_submit = prepare_lgbm_matrices_with_time_aware_catboost_encoding(
#     built_submit,

#     features_to_use=selected_features_light,

#     target_mode=TARGET_MODE,
#     target_col="pto",
#     id_col="new_id",
#     time_col="year_month_index",

#     smoothing=30.0,
#     min_count=2,

#     drop_original_cat_features=True,
#     drop_numeric_cat_source_cols=True,

#     include_all_detected_cat_features=True,
#     add_default_cat_features=True,
#     add_numeric_cat_copies=True,
#     numeric_cat_source_cols=make_x5_lgbm_numeric_cat_source_cols(),

#     keep_id_col=False,
#     keep_time_col=True,

#     strict_train_time_order=True,
#     first_block_value="nan",
#     unknown_value="global",

#     verbose=True,
# )

# test_pack = matrices_submit["test"]

# X_train_submit = test_pack["X_train"].copy()
# y_train_submit = pd.Series(test_pack["y_train"]).reset_index(drop=True)
# X_test_submit = test_pack["X_test"].copy()

# test_features_submit = built_submit["test_features"].reset_index(drop=True)
# test_ids = test_features_submit["new_id"].reset_index(drop=True)


# # ============================================================
# # 4. Проверяем признаки
# # ============================================================

# available_features = set(X_train_submit.columns).intersection(X_test_submit.columns)

# missing_features = [
#     f for f in selected_features_light
#     if f not in available_features
# ]

# if missing_features:
#     raise RuntimeError(
#         "В финальной train/test матрице нет части selected_features_light:\n"
#         + "\n".join(missing_features[:100])
#         + (f"\n... и ещё {len(missing_features) - 100}" if len(missing_features) > 100 else "")
#     )

# final_features = [
#     f for f in selected_features_light
#     if f in available_features
# ]

# if len(final_features) == 0:
#     raise RuntimeError("После проверки не осталось ни одного признака.")

# print("=" * 100)
# print("FINAL TRAIN/TEST SHAPES")
# print("=" * 100)
# print("X_train:", X_train_submit[final_features].shape)
# print("X_test: ", X_test_submit[final_features].shape)
# print("features:", len(final_features))
# print("test ids:", len(test_ids))
# print()


# # ============================================================
# # 5. Параметры финальной LightGBM-модели
# # ============================================================

# lgbm_submit_params = make_default_lgbm_params(
#     max_depth=5,
#     num_leaves=31,

#     # ВАЖНО: не вручную 8000, а из CV early stopping
#     n_estimators=final_n_estimators,

#     learning_rate=0.01,
#     objective="regression_l1",
#     metric="mae",

#     random_state=RANDOM_STATE,
#     n_jobs=-1,
# )

# lgbm_submit_params.update({
#     "min_child_samples": 80,
#     "min_child_weight": 1e-3,

#     "subsample": 0.85,
#     "subsample_freq": 1,
#     "colsample_bytree": 0.85,

#     "reg_alpha": 0.05,
#     "reg_lambda": 8.0,

#     "verbosity": -1,

#     # GPU
#     "device_type": "gpu",
#     "gpu_use_dp": False,
#     "max_bin": 63,
# })

# lgbm_submit_params.pop("force_col_wise", None)

# lgbm_submit_params = normalize_lgbm_params(
#     lgbm_submit_params,
#     random_state=RANDOM_STATE,
# )

# print("=" * 100)
# print("FINAL LGBM PARAMS")
# print("=" * 100)
# print(lgbm_submit_params)
# print()


# # ============================================================
# # 6. Обучаем финальную модель на всей истории
# # ============================================================

# final_lgbm_submit_model = _make_lgbm_model(lgbm_submit_params)

# try:
#     final_lgbm_submit_model.fit(
#         X_train_submit[final_features],
#         y_train_submit,
#     )
#     submit_fallback_used = False

# except Exception as e:
#     print("[GPU failed] Перезапускаю финальное обучение на CPU.")
#     print("Original error:", repr(e))

#     lgbm_submit_params_cpu = dict(lgbm_submit_params)
#     lgbm_submit_params_cpu.pop("device_type", None)
#     lgbm_submit_params_cpu.pop("gpu_use_dp", None)
#     lgbm_submit_params_cpu.pop("max_bin", None)

#     final_lgbm_submit_model = _make_lgbm_model(lgbm_submit_params_cpu)

#     final_lgbm_submit_model.fit(
#         X_train_submit[final_features],
#         y_train_submit,
#     )

#     lgbm_submit_params = lgbm_submit_params_cpu
#     submit_fallback_used = True


# # ============================================================
# # 7. Предсказываем март 2025
# # ============================================================

# pred_model_submit = final_lgbm_submit_model.predict(
#     X_test_submit[final_features]
# )

# pred_rto_submit = inverse_forecast_target_for_lgbm(
#     pred_model_submit,
#     X_test_submit[final_features],
#     mode=TARGET_MODE,
#     target_col="pto",
#     clip_min=0.0,
# )

# pred_rto_submit = np.asarray(pred_rto_submit, dtype=float)

# if not np.isfinite(pred_rto_submit).all():
#     raise RuntimeError("В прогнозе есть NaN или inf.")

# pred_rto_submit = np.maximum(pred_rto_submit, 0.0)


# # ============================================================
# # 8. Собираем и сохраняем test.csv
# # ============================================================

# submit = pd.DataFrame({
#     "new_id": test_ids,
#     "rto": pred_rto_submit,
# })

# submit = submit.sort_values("new_id").reset_index(drop=True)

# if len(submit) != EXPECTED_TEST_SIZE:
#     raise RuntimeError(
#         f"Неверное число строк в submit: {len(submit)} вместо {EXPECTED_TEST_SIZE}"
#     )

# if submit["new_id"].duplicated().any():
#     raise RuntimeError("В submit есть дубли new_id.")

# if submit["rto"].isna().any():
#     raise RuntimeError("В submit есть NaN в rto.")

# if not np.isfinite(submit["rto"].to_numpy()).all():
#     raise RuntimeError("В submit есть inf в rto.")

# submit.to_csv(OUTPUT_PATH, index=False)

# print("=" * 100)
# print("SUBMIT SAVED")
# print("=" * 100)
# print("period:", f"{TEST_YEAR}-{TEST_MONTH:02d}")
# print("path:", OUTPUT_PATH)
# print("shape:", submit.shape)
# print("fallback_used:", submit_fallback_used)
# print("final_n_estimators:", final_n_estimators)
# print()
# print(submit.head())
# print()
# print(submit["rto"].describe())

# gc.collect()

# check_submit = pd.read_csv("test.csv")

# print(check_submit.shape)
# print(check_submit.columns)
# print(check_submit.head())
# print(check_submit["rto"].describe())

# Optuna

In [20]:
from pathlib import Path

In [ ]:
# ============================================================
# LightGBM Optuna weighted-CV search for X5 direct log-target pipeline
# Strict MAE in log1p(pto) space + weighted MAPE on raw RTO level
# ============================================================

from __future__ import annotations

import gc
import json
import time
from pathlib import Path
from typing import Any, Optional, Sequence

import numpy as np
import pandas as pd
import optuna
from lightgbm import LGBMRegressor


# ============================================================
# 0. Config
# ============================================================

LGBM_OPTUNA_OUTPUT_DIR = Path("outputs_lgbm_optuna")
LGBM_OPTUNA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1-based period id:
#   15 = 2024-03
#   25 = 2025-01
#   26 = 2025-02
# ВАЖНО: это именно 1-based period id, а не year_month_index из ноутбука.
LGBM_WEIGHTED_CV_VALID_MONTHS = [15, 25, 26]

LGBM_WEIGHTED_CV_WEIGHTS = {
    15: 0.50,  # March 2024: сезонный аналог марта 2025
    25: 0.20,  # January 2025: свежий уровень ряда
    26: 0.30,  # February 2025: самый свежий месяц перед мартом
}

# LightGBM search space для direct_log1p таргета.
# Обучение строго на MAE:
#   objective='regression_l1'
#   metric='mae'
# Optuna objective ниже: weighted MAPE уже после expm1() на уровне РТО.
LGBM_OPTUNA_SEARCH_SPACE_LOGTARGET = {
    # Много деревьев, фактическое число выбирает early stopping.
    "n_estimators": (3_000, 14_000, 500),
    "learning_rate": ("float_log", 0.006, 0.060),

    # По твоей идее: деревья примерно 4-7.
    "max_depth": [4, 5, 6, 7],

    # Сэмплим фиксированное пространство, потом капаем под depth:
    # num_leaves = min(num_leaves_raw, 2 ** max_depth - 1)
    # Так не будет ошибки Optuna про dynamic value space.
    "num_leaves_raw": [15, 21, 31, 45, 63, 95, 127],

    # Ограничение листьев / регуляризация.
    "min_child_samples": (30, 220, 10),
    "min_child_weight": ("float_log", 1e-4, 1e-1),
    "min_split_gain": (0.0, 2.0),

    # Stochasticity.
    "subsample": (0.65, 0.95),
    "colsample_bytree": (0.65, 1.00),

    # Регуляризация. Область около твоего fixed-probe: lambda около 5-8,
    # но с возможностью уйти шире.
    "reg_alpha": ("float_log", 1e-4, 5.0),
    "reg_lambda": ("float_log", 1.0, 80.0),

    # Для GPU LightGBM обычно лучше небольшой max_bin, но 127/255 оставлены для проверки.
    "max_bin": [63, 127, 255],
}


# ============================================================
# 1. CV folds
# ============================================================

def period_id_1based_to_year_month(
    period_id: int,
    *,
    min_year: int = 2023,
) -> tuple[int, int]:
    """
    1-based period id:
        1  -> 2023-01
        12 -> 2023-12
        13 -> 2024-01
        15 -> 2024-03
        25 -> 2025-01
        26 -> 2025-02
    """
    period_id = int(period_id)

    if period_id <= 0:
        raise ValueError(f"period_id должен быть >= 1, получено: {period_id}")

    zero_based = period_id - 1
    year = int(min_year) + zero_based // 12
    month = zero_based % 12 + 1

    return int(year), int(month)


def _normalize_fold_weights_local(folds: Sequence[dict]) -> list[dict]:
    folds_out = [dict(f) for f in folds]

    for i, f in enumerate(folds_out):
        if "valid_year" not in f or "valid_month" not in f:
            raise ValueError(f"В fold #{i} должны быть ключи valid_year и valid_month.")

        f["valid_year"] = int(f["valid_year"])
        f["valid_month"] = int(f["valid_month"])
        f["weight"] = float(f.get("weight", 1.0))

        if "fold_name" not in f and "name" in f:
            f["fold_name"] = str(f["name"])

        if "name" not in f and "fold_name" in f:
            f["name"] = str(f["fold_name"])

        if "fold_name" not in f:
            f["fold_name"] = f"{f['valid_year']}_{f['valid_month']:02d}"
            f["name"] = f["fold_name"]

    total_weight = float(np.sum([f["weight"] for f in folds_out]))

    if not np.isfinite(total_weight) or total_weight <= 0:
        raise ValueError("Сумма весов CV folds должна быть положительной и конечной.")

    for f in folds_out:
        f["weight"] = float(f["weight"] / total_weight)

    return folds_out


def make_lgbm_weighted_cv_folds_for_march2025(
    *,
    cv_valid_months: Sequence[int] = LGBM_WEIGHTED_CV_VALID_MONTHS,
    cv_weights: dict[int, float] = LGBM_WEIGHTED_CV_WEIGHTS,
    min_year: int = 2023,
    normalize: bool = True,
) -> list[dict]:
    name_map = {
        (2024, 3): "march_2024",
        (2025, 1): "january_2025",
        (2025, 2): "february_2025",
    }

    folds = []

    for period_id in cv_valid_months:
        period_id = int(period_id)

        if period_id not in cv_weights:
            raise ValueError(f"Для period_id={period_id} не задан вес в cv_weights.")

        year, month = period_id_1based_to_year_month(period_id, min_year=min_year)
        fold_name = name_map.get((year, month), f"period_{period_id}_{year}_{month:02d}")

        folds.append(
            {
                "fold_name": fold_name,
                "name": fold_name,
                "period_id_1based": int(period_id),
                "valid_year": int(year),
                "valid_month": int(month),
                "weight": float(cv_weights[period_id]),
            }
        )

    if normalize:
        folds = _normalize_fold_weights_local(folds)

    return folds


def validate_lgbm_weighted_cv_folds_against_df(
    df: pd.DataFrame,
    folds: Sequence[dict],
    *,
    year_col: str = "year",
    month_col: str = "month",
    target_col: str = "pto",
    verbose: bool = True,
) -> pd.DataFrame:
    rows = []

    for f in folds:
        valid_year = int(f["valid_year"])
        valid_month = int(f["valid_month"])

        mask = (
            (df[year_col].astype(int) == valid_year)
            & (df[month_col].astype(int) == valid_month)
        )

        n_rows = int(mask.sum())
        n_target_notna = int(df.loc[mask, target_col].notna().sum()) if target_col in df.columns else 0

        rows.append(
            {
                "fold_name": f.get("fold_name"),
                "valid_year": valid_year,
                "valid_month": valid_month,
                "weight": float(f["weight"]),
                "n_rows": n_rows,
                "n_target_notna": n_target_notna,
            }
        )

    out = pd.DataFrame(rows)

    bad = out[(out["n_rows"] == 0) | (out["n_target_notna"] == 0)]

    if len(bad) > 0:
        raise ValueError(
            "Некоторые validation folds отсутствуют в df или не имеют target:\n"
            + bad.to_string(index=False)
        )

    if verbose:
        print("[LGBM weighted CV] Folds check:")
        print(out.to_string(index=False))
        print("[LGBM weighted CV] sum weights:", out["weight"].sum())

    return out


# ============================================================
# 2. Metrics and JSON helpers
# ============================================================

def _json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple)):
        return [_json_safe(v) for v in obj]

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, (np.floating,)):
        val = float(obj)
        return None if not np.isfinite(val) else val

    if isinstance(obj, float):
        return None if not np.isfinite(obj) else obj

    if obj is pd.NA:
        return None

    return obj


def _safe_mape_percent(y_true, y_pred, eps: float = 1e-9) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs(y_pred - y_true) / denom) * 100.0)


def _safe_wape_percent(y_true, y_pred, eps: float = 1e-9) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return float(
        np.sum(np.abs(y_pred - y_true))
        / (np.sum(np.abs(y_true)) + eps)
        * 100.0
    )


def _safe_mae(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_pred - y_true)))


def _safe_rmse(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))


def _safe_bias_percent(y_true, y_pred, eps: float = 1e-9) -> float:
    """
    > 0: перепрогноз
    < 0: недопрогноз
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return float(
        np.sum(y_pred - y_true)
        / (np.sum(np.abs(y_true)) + eps)
        * 100.0
    )


def _calc_level_metrics(y_true, y_pred) -> dict[str, float]:
    return {
        "mape": _safe_mape_percent(y_true, y_pred),
        "wape": _safe_wape_percent(y_true, y_pred),
        "mae": _safe_mae(y_true, y_pred),
        "rmse": _safe_rmse(y_true, y_pred),
        "bias_pct": _safe_bias_percent(y_true, y_pred),
    }


def _weighted_mean_from_df(df_metrics: pd.DataFrame, col: str) -> float:
    if df_metrics.empty:
        raise ValueError("df_metrics пустой, weighted mean посчитать нельзя.")

    if "weight" not in df_metrics.columns:
        raise KeyError("В df_metrics нет колонки 'weight'.")

    if col not in df_metrics.columns:
        raise KeyError(f"В df_metrics нет колонки '{col}'.")

    weights = pd.to_numeric(df_metrics["weight"], errors="coerce").astype(float).to_numpy()
    values = pd.to_numeric(df_metrics[col], errors="coerce").astype(float).to_numpy()

    mask = np.isfinite(weights) & np.isfinite(values)

    if mask.sum() == 0:
        raise ValueError(f"Нет конечных значений для weighted mean по колонке '{col}'.")

    weights = weights[mask]
    values = values[mask]

    total_weight = float(np.sum(weights))

    if total_weight <= 0:
        raise ValueError("Сумма весов должна быть положительной.")

    return float(np.sum(weights * values) / total_weight)


# ============================================================
# 3. Optuna suggest and LightGBM fit
# ============================================================

def make_lgbm_weighted_optuna_sampler(seed: int = 42) -> optuna.samplers.TPESampler:
    return optuna.samplers.TPESampler(
        seed=int(seed),
        n_startup_trials=20,
        multivariate=True,
        group=True,
    )


def _suggest_from_space(
    trial: optuna.Trial,
    name: str,
    default_spec,
    search_space: Optional[dict[str, Any]] = None,
    *,
    default_log: bool = False,
):
    spec = default_spec

    if search_space is not None and name in search_space:
        spec = search_space[name]

    if isinstance(spec, list):
        if len(spec) == 1:
            return spec[0]
        return trial.suggest_categorical(name, spec)

    if isinstance(spec, tuple):
        if len(spec) == 0:
            raise ValueError(f"Пустой search spec для параметра '{name}'.")

        if isinstance(spec[0], str):
            mode = spec[0]

            if mode == "fixed":
                return spec[1]

            if mode == "categorical":
                values = list(spec[1])
                if len(values) == 1:
                    return values[0]
                return trial.suggest_categorical(name, values)

            if mode == "int":
                low = int(spec[1])
                high = int(spec[2])
                step = int(spec[3]) if len(spec) >= 4 else 1
                return trial.suggest_int(name, low, high, step=step)

            if mode == "float":
                low = float(spec[1])
                high = float(spec[2])
                step = float(spec[3]) if len(spec) >= 4 else None

                if step is None:
                    return trial.suggest_float(name, low, high)

                return trial.suggest_float(name, low, high, step=step)

            if mode == "float_log":
                low = float(spec[1])
                high = float(spec[2])
                return trial.suggest_float(name, low, high, log=True)

            raise ValueError(f"Неизвестный mode='{mode}' для параметра '{name}'.")

        if len(spec) == 1:
            return spec[0]

        low = spec[0]
        high = spec[1]
        step = spec[2] if len(spec) >= 3 else None

        if isinstance(low, int) and isinstance(high, int) and not default_log:
            return trial.suggest_int(
                name,
                int(low),
                int(high),
                step=int(step) if step is not None else 1,
            )

        low = float(low)
        high = float(high)

        if default_log:
            return trial.suggest_float(name, low, high, log=True)

        if step is None:
            return trial.suggest_float(name, low, high)

        return trial.suggest_float(name, low, high, step=float(step))

    return spec


def suggest_lgbm_logtarget_weighted_optuna_params(
    trial: optuna.Trial,
    *,
    search_space_overrides: Optional[dict[str, Any]] = None,
    fixed_model_params: Optional[dict[str, Any]] = None,
    params_overrides: Optional[dict[str, Any]] = None,
    random_state: int = 42,
    use_gpu: bool = True,
) -> dict[str, Any]:
    """
    LightGBM для direct_log1p таргета.

    Model training:
        objective='regression_l1'
        metric='mae'

    Optuna objective:
        weighted MAPE на восстановленном уровне РТО.
    """
    search_space_overrides = {} if search_space_overrides is None else dict(search_space_overrides)

    max_depth = int(_suggest_from_space(
        trial,
        "max_depth",
        [4, 5, 6, 7],
        search_space_overrides,
    ))

    num_leaves_raw = int(_suggest_from_space(
        trial,
        "num_leaves_raw",
        [15, 21, 31, 45, 63, 95, 127],
        search_space_overrides,
    ))

    max_leaves_by_depth = 2 ** max_depth - 1 if max_depth > 0 else num_leaves_raw
    num_leaves = int(min(num_leaves_raw, max_leaves_by_depth))

    params = {
        "boosting_type": "gbdt",

        # Строго MAE в model-space: для direct_log1p это MAE по log1p(pto).
        "objective": "regression_l1",
        "metric": "mae",

        "n_estimators": int(_suggest_from_space(
            trial,
            "n_estimators",
            (3_000, 14_000, 500),
            search_space_overrides,
        )),
        "learning_rate": float(_suggest_from_space(
            trial,
            "learning_rate",
            ("float_log", 0.006, 0.060),
            search_space_overrides,
        )),

        "max_depth": max_depth,
        "num_leaves": num_leaves,
        "min_child_samples": int(_suggest_from_space(
            trial,
            "min_child_samples",
            (30, 220, 10),
            search_space_overrides,
        )),
        "min_child_weight": float(_suggest_from_space(
            trial,
            "min_child_weight",
            ("float_log", 1e-4, 1e-1),
            search_space_overrides,
        )),
        "min_split_gain": float(_suggest_from_space(
            trial,
            "min_split_gain",
            (0.0, 2.0),
            search_space_overrides,
        )),

        "subsample": float(_suggest_from_space(
            trial,
            "subsample",
            (0.65, 0.95),
            search_space_overrides,
        )),
        "subsample_freq": 1,
        "colsample_bytree": float(_suggest_from_space(
            trial,
            "colsample_bytree",
            (0.65, 1.00),
            search_space_overrides,
        )),

        "reg_alpha": float(_suggest_from_space(
            trial,
            "reg_alpha",
            ("float_log", 1e-4, 5.0),
            search_space_overrides,
        )),
        "reg_lambda": float(_suggest_from_space(
            trial,
            "reg_lambda",
            ("float_log", 1.0, 80.0),
            search_space_overrides,
        )),

        "max_bin": int(_suggest_from_space(
            trial,
            "max_bin",
            [63, 127, 255],
            search_space_overrides,
        )),

        "random_state": int(random_state),
        "n_jobs": -1,
        "verbosity": -1,
    }

    if use_gpu:
        params.update({
            "device_type": "gpu",
            "gpu_use_dp": False,
        })
        # force_col_wise может конфликтовать с GPU.
        params.pop("force_col_wise", None)
    else:
        params.update({
            "force_col_wise": True,
        })
        params.pop("device_type", None)
        params.pop("gpu_use_dp", None)

    # Разрешаем fixed_model_params, но не даём случайно сбить MAE/direct-log логику.
    if fixed_model_params is not None:
        fixed_model_params = dict(fixed_model_params)
        for forbidden_key in [
            "objective",
            "metric",
            "loss_function",
            "eval_metric",
            "early_stopping_rounds",
            "num_leaves_raw",
        ]:
            fixed_model_params.pop(forbidden_key, None)
        params.update(fixed_model_params)

    # Разрешаем params_overrides, но тоже защищаем objective/metric.
    if params_overrides is not None:
        params_overrides = dict(params_overrides)
        for forbidden_key in [
            "objective",
            "metric",
            "loss_function",
            "eval_metric",
            "early_stopping_rounds",
            "num_leaves_raw",
        ]:
            params_overrides.pop(forbidden_key, None)
        params.update(params_overrides)

    # Финальная страховка.
    params["objective"] = "regression_l1"
    params["metric"] = "mae"
    params["max_depth"] = int(params["max_depth"])

    if params["max_depth"] > 0:
        params["num_leaves"] = int(min(int(params["num_leaves"]), 2 ** int(params["max_depth"]) - 1))
    else:
        params["num_leaves"] = int(params["num_leaves"])

    trial.set_user_attr("loss_function", "MAE")
    trial.set_user_attr("target_mode", "direct_log1p")
    trial.set_user_attr("lgbm_objective", "regression_l1")
    trial.set_user_attr("lgbm_metric", "mae")
    trial.set_user_attr("num_leaves_raw", num_leaves_raw)
    trial.set_user_attr("num_leaves_capped", params["num_leaves"])

    return params


def _fit_lgbm_with_eval(
    *,
    params: dict[str, Any],
    X_train: pd.DataFrame,
    y_train,
    X_valid: pd.DataFrame,
    y_valid,
    early_stopping_rounds: int = 50,
    log_evaluation_period: int | None = None,
    fallback_to_cpu: bool = True,
):
    """
    Обучает LGBMRegressor с early stopping.

    Early stopping идёт по validation MAE в model-space:
    - для target_mode='direct_log1p' это MAE по log1p(pto);
    - Optuna objective ниже считается после inverse transform через expm1().
    """
    import lightgbm as lgb

    params = dict(params)

    callbacks = [
        lgb.early_stopping(
            stopping_rounds=int(early_stopping_rounds),
            first_metric_only=True,
            verbose=False,
        )
    ]

    if log_evaluation_period is not None and int(log_evaluation_period) > 0:
        callbacks.append(lgb.log_evaluation(period=int(log_evaluation_period)))

    def _fit_once(p: dict[str, Any]):
        model = LGBMRegressor(**p)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric=p.get("metric", "mae"),
            callbacks=callbacks,
        )
        return model

    try:
        model = _fit_once(params)
        return model, False

    except Exception:
        if not fallback_to_cpu or params.get("device_type") != "gpu":
            raise

        cpu_params = dict(params)
        cpu_params.pop("device_type", None)
        cpu_params.pop("gpu_use_dp", None)
        cpu_params["force_col_wise"] = True

        model = _fit_once(cpu_params)
        return model, True


# ============================================================
# 4. Main weighted Optuna runner
# ============================================================

def _default_lgbm_weighted_encoding_kwargs() -> dict[str, Any]:
    numeric_cat_source_cols = ()

    if "make_x5_lgbm_numeric_cat_source_cols" in globals():
        numeric_cat_source_cols = make_x5_lgbm_numeric_cat_source_cols()

    return {
        "smoothing": 30.0,
        "min_count": 2,

        "drop_original_cat_features": True,
        "drop_numeric_cat_source_cols": True,
        "include_all_detected_cat_features": True,

        "add_default_cat_features": True,
        "add_numeric_cat_copies": True,
        "numeric_cat_source_cols": numeric_cat_source_cols,

        "keep_id_col": False,
        "keep_time_col": True,

        "strict_train_time_order": True,
        "first_block_value": "nan",
        "unknown_value": "global",
    }


def _default_lgbm_weighted_feature_builder_kwargs() -> dict[str, Any]:
    if "make_x5_lgbm_feature_selection_builder_kwargs" in globals():
        return make_x5_lgbm_feature_selection_builder_kwargs()
    return {}


def run_lgbm_logtarget_weighted_optuna_search_march2025(
    *,
    df: pd.DataFrame,
    selected_features: Optional[Sequence[str]] = None,
    folds: Optional[Sequence[dict]] = None,

    feature_builder_func=None,
    matrix_preparer_func=None,
    feature_builder_kwargs: Optional[dict[str, Any]] = None,
    encoding_kwargs: Optional[dict[str, Any]] = None,

    target_mode: str = "direct_log1p",
    target_col: str = "pto",

    n_trials: int = 300,
    timeout_seconds: Optional[int] = None,

    study_name: str = "lgbm_direct_log1p_weighted_cv_march2025_mae_v1",
    storage_path: Optional[str | Path] = None,
    load_if_exists: bool = True,

    random_state: int = 42,
    use_gpu: bool = True,
    fallback_to_cpu: bool = True,
    early_stopping_rounds: int = 50,
    log_evaluation_period: Optional[int] = None,

    search_space_overrides: Optional[dict[str, Any]] = None,
    fixed_model_params: Optional[dict[str, Any]] = None,
    params_overrides: Optional[dict[str, Any]] = None,

    prune: bool = False,

    verbose: bool = True,

    save_artifacts: bool = True,
    output_dir: Path = LGBM_OPTUNA_OUTPUT_DIR,

    return_context: bool = True,
):
    """
    Optuna-перебор LightGBM для текущего X5 direct-log pipeline.

    CV objective:
        0.50 * MAPE(2024-03)
      + 0.20 * MAPE(2025-01)
      + 0.30 * MAPE(2025-02)

    Model loss:
        objective='regression_l1'
        metric='mae'

    Target:
        target_mode='direct_log1p', то есть модель учится на log1p(pto),
        а MAPE считается после обратного преобразования expm1().

    Feature engineering:
        не меняется, используется существующий build_lgbm_encoded_fold_cache.
    """
    if target_mode != "direct_log1p":
        raise ValueError(
            "Эта функция специально зафиксирована под log-target: target_mode='direct_log1p'. "
            f"Получено: {target_mode!r}"
        )

    required_global_funcs = [
        "build_lgbm_encoded_fold_cache",
        "resolve_features_for_lgbm_matrix",
        "inverse_forecast_target_for_lgbm",
    ]

    missing_global = [name for name in required_global_funcs if name not in globals()]

    if missing_global:
        raise RuntimeError(f"Сначала выполни ячейки с функциями: {missing_global}")

    if selected_features is None:
        if "selected_features_light" not in globals():
            raise ValueError("Передай selected_features или определи selected_features_light.")
        selected_features = list(globals()["selected_features_light"])

    selected_features = list(dict.fromkeys(selected_features))

    if feature_builder_func is None:
        if "build_retail_features_with_split" not in globals():
            raise ValueError("Передай feature_builder_func или выполни build_retail_features_with_split.")
        feature_builder_func = globals()["build_retail_features_with_split"]

    if matrix_preparer_func is None:
        if "prepare_lgbm_matrices_with_time_aware_catboost_encoding" not in globals():
            raise ValueError(
                "Передай matrix_preparer_func или выполни "
                "prepare_lgbm_matrices_with_time_aware_catboost_encoding."
            )
        matrix_preparer_func = globals()["prepare_lgbm_matrices_with_time_aware_catboost_encoding"]

    if folds is None:
        folds = make_lgbm_weighted_cv_folds_for_march2025(normalize=True)
    else:
        folds = _normalize_fold_weights_local(folds)

    validate_lgbm_weighted_cv_folds_against_df(
        df=df,
        folds=folds,
        year_col="year",
        month_col="month",
        target_col=target_col,
        verbose=verbose,
    )

    if feature_builder_kwargs is None:
        feature_builder_kwargs = _default_lgbm_weighted_feature_builder_kwargs()
    else:
        feature_builder_kwargs = dict(feature_builder_kwargs)

    if encoding_kwargs is None:
        encoding_kwargs = _default_lgbm_weighted_encoding_kwargs()
    else:
        default_encoding_kwargs = _default_lgbm_weighted_encoding_kwargs()
        default_encoding_kwargs.update(dict(encoding_kwargs))
        encoding_kwargs = default_encoding_kwargs

    if search_space_overrides is None:
        search_space_overrides = LGBM_OPTUNA_SEARCH_SPACE_LOGTARGET

    if storage_path is None:
        storage_path = output_dir / f"{study_name}.sqlite3"

    if verbose:
        print("\n[Optuna:LGBM direct_log1p MAE] Build encoded fold_cache once before optimize...")
        print(f"[Optuna:LGBM direct_log1p MAE] selected raw features: {len(selected_features)}")
        print(f"[Optuna:LGBM direct_log1p MAE] target_mode: {target_mode}")

    # ВАЖНО: текущая build_lgbm_encoded_fold_cache из LGBM-ноутбука
    # не принимает target_anchor_col. Для direct_log1p anchor не нужен.
    fold_cache, feature_map, encoding_meta = build_lgbm_encoded_fold_cache(
        df,
        folds=folds,
        feature_builder_func=feature_builder_func,
        matrix_preparer_func=matrix_preparer_func,
        feature_builder_kwargs=feature_builder_kwargs,
        raw_features_to_use=selected_features,
        encoding_kwargs=encoding_kwargs,
        target_mode=target_mode,
        target_col=target_col,
        verbose=verbose,
    )

    available_features = list(feature_map.get("all_encoded_features") or [])

    if not available_features:
        all_cols = []
        for fd in fold_cache:
            all_cols.extend(list(fd.X_train.columns))
            all_cols.extend(list(fd.X_valid.columns))
        available_features = list(dict.fromkeys(all_cols))

    features_for_model = resolve_features_for_lgbm_matrix(
        selected_features,
        feature_map=feature_map,
        available_features=available_features,
        forbidden_features=[],
        strict=False,
    )

    if not features_for_model:
        raise ValueError("После resolve_features_for_lgbm_matrix не осталось признаков для LightGBM.")

    if verbose:
        print(f"[Optuna:LGBM direct_log1p MAE] resolved model features: {len(features_for_model)}")
        print(f"[Optuna:LGBM direct_log1p MAE] folds: {len(fold_cache)}")
        for fd in fold_cache:
            print(
                f"  - {fd.fold_name}: "
                f"valid={fd.valid_year}-{fd.valid_month:02d}, "
                f"weight={fd.weight:.4f}, "
                f"X_train={fd.X_train.shape}, "
                f"X_valid={fd.X_valid.shape}"
            )

    sampler = make_lgbm_weighted_optuna_sampler(seed=random_state)

    if prune:
        pruner = optuna.pruners.MedianPruner(
            n_startup_trials=20,
            n_warmup_steps=1,
        )
    else:
        pruner = optuna.pruners.NopPruner()

    storage_path = Path(storage_path)
    storage_path.parent.mkdir(parents=True, exist_ok=True)
    storage = f"sqlite:///{storage_path}"

    study = optuna.create_study(
        direction="minimize",
        study_name=study_name,
        sampler=sampler,
        pruner=pruner,
        storage=storage,
        load_if_exists=load_if_exists,
    )

    def objective(trial: optuna.Trial) -> float:
        start_time = time.time()

        params = suggest_lgbm_logtarget_weighted_optuna_params(
            trial,
            search_space_overrides=search_space_overrides,
            fixed_model_params=fixed_model_params,
            params_overrides=params_overrides,
            random_state=random_state,
            use_gpu=use_gpu,
        )

        fold_records = []
        best_iterations = []
        fallback_used_any = False

        for fold_idx, fd in enumerate(fold_cache):
            available = set(fd.X_train.columns).intersection(fd.X_valid.columns)
            fold_features = [f for f in features_for_model if f in available]

            if not fold_features:
                raise optuna.TrialPruned(f"На fold={fd.fold_name} нет доступных признаков.")

            if fd.y_valid is None:
                raise optuna.TrialPruned(f"На fold={fd.fold_name} нет y_valid для early stopping.")

            X_train = fd.X_train.loc[:, fold_features]
            X_valid = fd.X_valid.loc[:, fold_features]

            y_train = pd.Series(fd.y_train).astype(float).reset_index(drop=True)
            y_valid = pd.Series(fd.y_valid).astype(float).reset_index(drop=True)

            model = None

            try:
                model, fallback_used = _fit_lgbm_with_eval(
                    params=params,
                    X_train=X_train,
                    y_train=y_train,
                    X_valid=X_valid,
                    y_valid=y_valid,
                    early_stopping_rounds=early_stopping_rounds,
                    log_evaluation_period=log_evaluation_period,
                    fallback_to_cpu=fallback_to_cpu,
                )
                fallback_used_any = bool(fallback_used_any or fallback_used)

                best_iter = getattr(model, "best_iteration_", None)
                best_iter = None if best_iter is None else int(best_iter)
                best_iterations.append(best_iter)

                if best_iter is not None and best_iter > 0:
                    raw_pred = model.predict(X_valid, num_iteration=best_iter)
                else:
                    raw_pred = model.predict(X_valid)

                pred_level = inverse_forecast_target_for_lgbm(
                    raw_pred,
                    X_valid,
                    mode=target_mode,
                    target_col=target_col,
                    clip_min=0.0,
                )

                y_true_level = np.asarray(fd.y_valid_raw, dtype=float)
                metrics = _calc_level_metrics(y_true_level, pred_level)

                rec = {
                    "trial_number": int(trial.number),
                    "fold_idx": int(fold_idx),
                    "fold_name": fd.fold_name,
                    "valid_year": int(fd.valid_year),
                    "valid_month": int(fd.valid_month),
                    "weight": float(fd.weight),
                    "n_train": int(len(X_train)),
                    "n_valid": int(len(X_valid)),
                    "n_features": int(len(fold_features)),
                    "best_iteration": best_iter,
                    "fallback_to_cpu_used": bool(fallback_used),
                    **metrics,
                }

                fold_records.append(rec)

                partial_df = pd.DataFrame(fold_records)
                partial_weighted_mape = _weighted_mean_from_df(partial_df, "mape")

                trial.report(partial_weighted_mape, step=fold_idx)

                if trial.should_prune():
                    raise optuna.TrialPruned(
                        f"Trial pruned on fold={fd.fold_name}, "
                        f"partial weighted MAPE={partial_weighted_mape:.6f}"
                    )

            except optuna.TrialPruned:
                raise

            except Exception as e:
                trial.set_user_attr("error", repr(e))
                raise optuna.TrialPruned(f"LightGBM trial failed: {repr(e)}")

            finally:
                if model is not None:
                    del model
                gc.collect()

        fold_metrics_df = pd.DataFrame(fold_records)

        weighted_mape = _weighted_mean_from_df(fold_metrics_df, "mape")
        weighted_wape = _weighted_mean_from_df(fold_metrics_df, "wape")
        weighted_bias = _weighted_mean_from_df(fold_metrics_df, "bias_pct")

        elapsed = float(time.time() - start_time)

        trial.set_user_attr("weighted_mape", weighted_mape)
        trial.set_user_attr("weighted_wape", weighted_wape)
        trial.set_user_attr("weighted_bias_pct", weighted_bias)
        trial.set_user_attr("elapsed_seconds", elapsed)
        trial.set_user_attr("n_features", len(features_for_model))
        trial.set_user_attr("fallback_to_cpu_used", bool(fallback_used_any))

        trial.set_user_attr(
            "features_json",
            json.dumps(_json_safe(features_for_model), ensure_ascii=False),
        )

        trial.set_user_attr(
            "model_params_json",
            json.dumps(_json_safe(params), ensure_ascii=False),
        )

        trial.set_user_attr(
            "fold_metrics_json",
            json.dumps(_json_safe(fold_records), ensure_ascii=False),
        )

        trial.set_user_attr(
            "best_iterations_json",
            json.dumps(_json_safe(best_iterations), ensure_ascii=False),
        )

        return weighted_mape

    study.optimize(
        objective,
        n_trials=int(n_trials),
        timeout=timeout_seconds,
        gc_after_trial=True,
        show_progress_bar=True,
    )

    trial_summary_df = study.trials_dataframe()

    fold_metric_records = []

    for t in study.trials:
        raw = t.user_attrs.get("fold_metrics_json")

        if raw is None:
            continue

        try:
            fold_metric_records.extend(json.loads(raw))
        except Exception:
            pass

    fold_metrics_df = pd.DataFrame(fold_metric_records)

    complete_trials = [
        t for t in study.trials
        if t.state == optuna.trial.TrialState.COMPLETE
    ]

    if verbose:
        print("\n[Optuna:LGBM direct_log1p MAE] COMPLETE trials:", len(complete_trials))

        if complete_trials:
            best_trial = min(complete_trials, key=lambda t: t.value)

            print("[Optuna:LGBM direct_log1p MAE] Best trial:", best_trial.number)
            print("[Optuna:LGBM direct_log1p MAE] Best weighted MAPE:", best_trial.value)
            print("[Optuna:LGBM direct_log1p MAE] Best sampled params:")
            print(json.dumps(_json_safe(best_trial.params), ensure_ascii=False, indent=2))

            if "model_params_json" in best_trial.user_attrs:
                print("[Optuna:LGBM direct_log1p MAE] Best full LightGBM params:")
                print(
                    json.dumps(
                        json.loads(best_trial.user_attrs["model_params_json"]),
                        ensure_ascii=False,
                        indent=2,
                    )
                )

    if save_artifacts:
        output_dir.mkdir(parents=True, exist_ok=True)

        trial_summary_df.to_csv(
            output_dir / f"{study_name}_trial_summary.csv",
            index=False,
        )

        fold_metrics_df.to_csv(
            output_dir / f"{study_name}_fold_metrics.csv",
            index=False,
        )

        if isinstance(encoding_meta, pd.DataFrame):
            encoding_meta.to_csv(
                output_dir / f"{study_name}_encoding_meta.csv",
                index=False,
            )

        if complete_trials:
            best_trial = min(complete_trials, key=lambda t: t.value)

            best_payload = {
                "best_trial_number": int(best_trial.number),
                "best_weighted_mape": float(best_trial.value),
                "best_sampled_params": _json_safe(best_trial.params),
                "best_full_lgbm_params": json.loads(best_trial.user_attrs["model_params_json"]),
                "features_for_model": features_for_model,
                "folds": _json_safe(folds),
                "target_mode": target_mode,
                "early_stopping_rounds": int(early_stopping_rounds),
                "search_space": _json_safe(search_space_overrides),
            }

            with open(output_dir / f"{study_name}_best.json", "w", encoding="utf-8") as f:
                json.dump(best_payload, f, ensure_ascii=False, indent=2)

        if verbose:
            print(f"[Optuna:LGBM direct_log1p MAE] Artifacts saved to: {output_dir.resolve()}")

    context = {
        "folds": folds,
        "fold_cache": fold_cache,
        "feature_map": feature_map,
        "encoding_meta": encoding_meta,
        "features_for_model": features_for_model,
        "target_mode": target_mode,
    }

    if return_context:
        return study, trial_summary_df, fold_metrics_df, context

    return study, trial_summary_df, fold_metrics_df


def get_best_lgbm_params_from_weighted_study(study: optuna.Study) -> dict[str, Any]:
    complete_trials = [
        t for t in study.trials
        if t.state == optuna.trial.TrialState.COMPLETE
    ]

    if not complete_trials:
        raise ValueError("В study нет COMPLETE trials.")

    best_trial = min(complete_trials, key=lambda t: t.value)

    if "model_params_json" not in best_trial.user_attrs:
        raise KeyError("У best_trial нет user_attrs['model_params_json'].")

    params = json.loads(best_trial.user_attrs["model_params_json"])

    if params.get("objective") != "regression_l1":
        raise ValueError(f"Best params имеют неверный objective: {params.get('objective')}")

    if params.get("metric") != "mae":
        raise ValueError(f"Best params имеют неверный metric: {params.get('metric')}")

    return params


# ============================================================
# 5. Launch
# ============================================================

if __name__ == "__main__":
    required_objects = [
        "df",
        "selected_features_light",
        "build_retail_features_with_split",
        "prepare_lgbm_matrices_with_time_aware_catboost_encoding",
        "build_lgbm_encoded_fold_cache",
        "resolve_features_for_lgbm_matrix",
        "inverse_forecast_target_for_lgbm",
    ]

    missing = [name for name in required_objects if name not in globals()]

    if missing:
        raise RuntimeError(
            "Сначала выполни ячейки, где определены эти объекты:\n"
            + "\n".join(missing)
        )

    print("All required objects are available.")

    # 1. Собираем weighted CV folds.
    lgbm_weighted_cv_folds = make_lgbm_weighted_cv_folds_for_march2025(
        cv_valid_months=LGBM_WEIGHTED_CV_VALID_MONTHS,
        cv_weights=LGBM_WEIGHTED_CV_WEIGHTS,
        min_year=2023,
        normalize=True,
    )

    folds_check_df = validate_lgbm_weighted_cv_folds_against_df(
        df=df,
        folds=lgbm_weighted_cv_folds,
        year_col="year",
        month_col="month",
        target_col="pto",
        verbose=True,
    )

    try:
        display(folds_check_df)
    except NameError:
        print(folds_check_df)

    # 2. Запускаем Optuna.
    lgbm_weighted_study, lgbm_weighted_trial_summary_df, lgbm_weighted_fold_metrics_df, lgbm_weighted_context = run_lgbm_logtarget_weighted_optuna_search_march2025(
        df=df,

        selected_features=selected_features_light,
        folds=lgbm_weighted_cv_folds,

        feature_builder_func=build_retail_features_with_split,
        matrix_preparer_func=prepare_lgbm_matrices_with_time_aware_catboost_encoding,

        # ВАЖНО: это log-target, а не log-diff.
        target_mode="direct_log1p",
        target_col="pto",

        # Можно уменьшить до 50-100 для быстрой проверки.
        n_trials=300,

        # Быстрый early stopping: 50 boosting rounds без улучшения MAE в log-space.
        early_stopping_rounds=50,
        log_evaluation_period=None,

        study_name="lgbm_direct_log1p_weighted_cv_march2025_mae_v1",
        storage_path=LGBM_OPTUNA_OUTPUT_DIR / "lgbm_direct_log1p_weighted_cv_march2025_mae_v1.sqlite3",
        load_if_exists=True,

        search_space_overrides=LGBM_OPTUNA_SEARCH_SPACE_LOGTARGET,

        fixed_model_params=None,
        params_overrides=None,

        use_gpu=True,
        fallback_to_cpu=True,
        random_state=42,

        # Для 3 фолдов лучше пока без pruning.
        prune=False,

        verbose=True,

        save_artifacts=True,
        return_context=True,
    )

    # 3. Забираем лучшие параметры.
    best_lgbm_weighted_params = get_best_lgbm_params_from_weighted_study(lgbm_weighted_study)

    print("\nBest weighted-CV LightGBM params:")
    print(json.dumps(best_lgbm_weighted_params, ensure_ascii=False, indent=2))

    assert best_lgbm_weighted_params["objective"] == "regression_l1"
    assert best_lgbm_weighted_params["metric"] == "mae"

    print("\nObjective check OK: LightGBM strict MAE on direct_log1p target")

    # 4. Смотрим лучший trial по фолдам.
    complete_trials = [
        t for t in lgbm_weighted_study.trials
        if t.state == optuna.trial.TrialState.COMPLETE
    ]

    if not complete_trials:
        raise RuntimeError("Нет COMPLETE trials. Проверь ошибки в Optuna study.")

    best_trial = min(complete_trials, key=lambda t: t.value)
    best_trial_number = best_trial.number

    print(f"\nBest trial number: {best_trial_number}")
    print(f"Best weighted MAPE: {best_trial.value:.6f}")

    best_fold_view = (
        lgbm_weighted_fold_metrics_df
        .query("trial_number == @best_trial_number")
        .sort_values(["valid_year", "valid_month"])
        .reset_index(drop=True)
    )

    try:
        display(best_fold_view)
    except NameError:
        print(best_fold_view)

    print("\nWeighted fold metrics:")
    print(
        best_fold_view[
            [
                "fold_name",
                "valid_year",
                "valid_month",
                "weight",
                "mape",
                "wape",
                "bias_pct",
                "best_iteration",
                "fallback_to_cpu_used",
            ]
        ].to_string(index=False)
    )

    # 5. Короткая сводка по study.
    print("\nStudy summary:")
    print("n complete trials:", len(complete_trials))
    print("best weighted MAPE:", best_trial.value)
    print("artifacts dir:", LGBM_OPTUNA_OUTPUT_DIR.resolve())


All required objects are available.
[LGBM weighted CV] Folds check:
    fold_name  valid_year  valid_month  weight  n_rows  n_target_notna
   march_2024        2024            3   0.500   18657           18657
 january_2025        2025            1   0.200   18657           18657
february_2025        2025            2   0.300   18657           18657
[LGBM weighted CV] sum weights: 1.0


,fold_name,valid_year,valid_month,weight,n_rows,n_target_notna
0,march_2024,2024,3,0.500,18657,18657
1,january_2025,2025,1,0.200,18657,18657
2,february_2025,2025,2,0.300,18657,18657


[LGBM weighted CV] Folds check:
    fold_name  valid_year  valid_month  weight  n_rows  n_target_notna
   march_2024        2024            3   0.500   18657           18657
 january_2025        2025            1   0.200   18657           18657
february_2025        2025            2   0.300   18657           18657
[LGBM weighted CV] sum weights: 1.0

[Optuna:LGBM direct_log1p MAE] Build encoded fold_cache once before optimize...
[Optuna:LGBM direct_log1p MAE] selected raw features: 34
[Optuna:LGBM direct_log1p MAE] target_mode: direct_log1p
[fold-build:lgbm] march_2024: valid=2024-03


/tmp/ipykernel_58/6676950.py:1365: DeprecationWarning:

is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead

/tmp/ipykernel_58/6676950.py:2228: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/tmp/ipykernel_58/6676950.py:2235: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/tmp/ipykernel_58/6676950.py:2238: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consi

    X_train=(261198, 59), X_valid=(18657, 59), encoded_cats=25
[fold-build:lgbm] january_2025: valid=2025-01


...

  0%|          | 0/300 [00:00<?, ?it/s]

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[I 2026-05-30 14:44:26,306] Trial 0 finished with value: 4.335621342403391 and parameters: {'max_depth': 5, 'num_leaves_raw': 45, 'n_estimators': 14000, 'learning_rate': 0.040793774529546765, 'min_child_samples': 70, 'min_child_weight': 0.0003511356313970409, 'min_split_gain': 0.36680901970686763, 'subsample': 0.7412726728878614, 'colsample_bytree': 0.8336647510712832, 'reg_alpha': 0.010707712109770782, 'reg_lambda': 3.582904732228079, 'max_bin': 63}. Best is trial 0 with value: 4.335621342403391.
[I 2026-05-30 14:45:43,316] Trial 1 finished with value: 4.344840734652394 and parameters: {'max_depth': 6, 'num_leaves_raw': 127, 'n_estimators': 14000, 'learning_rate': 0.03859655995709934, 'min_child_samples': 90, 'min_child_weight': 0.00019634341572933326, 'min_split_gain': 1.3684660530243138, 'subsample': 0.7820457481218803, 'colsample_bytree': 0.6927133821956726, 'reg_alpha': 0.021223717067582068, 'reg_lambda': 1.1626378306531546, 'max_bin': 63}. Best is trial 0 with value: 4.3356213424

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[I 2026-05-30 14:48:25,326] Trial 2 finished with value: 4.3592834528491196 and parameters: {'max_depth': 6, 'num_leaves_raw': 15, 'n_estimators': 5000, 'learning_rate': 0.00665853273666186, 'min_child_samples': 90, 'min_child_weight': 0.001465655388622534, 'min_split_gain': 0.5426980635477918, 'subsample': 0.8986212527455788, 'colsample_bytree': 0.7748636643427562, 'reg_alpha': 0.002089790902378636, 'reg_lambda': 10.784486765834227, 'max_bin': 127}. Best is trial 0 with value: 4.335621342403391.
[I 2026-05-30 14:49:56,696] Trial 3 finished with value: 4.2866759881479535 and parameters: {'max_depth': 4, 'num_leaves_raw': 15, 'n_estimators': 12500, 'learning_rate': 0.025202833900213193, 'min_child_samples': 90, 'min_child_weight': 0.0001551225912648476, 'min_split_gain': 0.6219646434313244, 'subsample': 0.7475549966080242, 'colsample_bytree': 0.9053621624183225, 'reg_alpha': 0.09905204218383899, 'reg_lambda': 48.80290640628797, 'max_bin': 255}. Best is trial 3 with value: 4.286675988147